# Taxpayer Segmentation - v03 - backup

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

In [ ]:
# Executar apenas uma vez para instalar as bibliotecas
!pip install pandas numpy matplotlib seaborn scikit-learn scipy yellowbrick kneed

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer

# Redução de dimensionalidade
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

# Algoritmos de Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# Validação
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

# Utilitários
from kneed import KneeLocator
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Bibliotecas carregadas com sucesso!")

In [ ]:
# ============================================================================
# CARREGAR DADOS
# ============================================================================

# Importação do dataset
# NOTA: caminho local generalizado para fins de publicacao.
# O ficheiro dados_devedores.csv (dados reais de devedores) NAO e distribuido
# neste repositorio, por estar sujeito a sigilo fiscal (art. 198 do CTN).
filepath = 'dados_devedores.csv'

# Carregar CSV
df = pd.read_csv(filepath, encoding='utf-8')

# Visão geral
print(f"Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\nPrimeiras linhas:")
df.head()

## Exploração inicial 

### Estatísticas descritivas

In [ ]:
# ============================================================================
# EXPLORAÇÃO INICIAL
# ============================================================================

# Informações gerais
print("=" * 60)
print("INFORMAÇÕES DO DATASET")
print("=" * 60)
df.info()

# Estatísticas descritivas das variáveis numéricas
print("\n" + "=" * 60)
print("ESTATÍSTICAS DESCRITIVAS (NUMÉRICAS)")
print("=" * 60)
df.describe().T
print(df.describe().T)

# Verificar tipos de dados
print("\n" + "=" * 60)
print("TIPOS DE DADOS")
print("=" * 60)
print(df.dtypes.value_counts())

### Separação variáveis por tipo

In [ ]:
# ============================================================================
# CLASSIFICAR VARIÁVEIS
# ============================================================================

# Chave de identificação (não entra no modelo)
var_id = ['id_cnpj']

# Variáveis categóricas (para encoding ou exclusão do clustering)
var_categoricas = [
    'tp_enterprise_size_cat',
    'tp_economic_sector_desc_cat',
    'tp_registration_status_cat', 
    'tp_tax_regime_cat',
    'tp_legal_form_cat',
    'tp_fiscal_region_cat',
    'tp_geographic_location_cat',
    'tp_special_taxpayer_status_cat',
    'tp_special_tax_regime_cat'
    ]

# Variáveis binárias (0/1)
var_binarias = [  
    'da_new_debt_incidence_bin',
    'da_prescription_risk_bin',
    'pc_recent_payment_indicator_bin'
]

# Variáveis numéricas contínuas (candidatas ao clustering)
var_numericas = [
    # Atributos da dívida
    'da_total_debt_value',
    'da_total_principal_value',
    'da_total_penalties_interest_value',
    'da_debt_pgfn_value',
    'da_enforceable_debt_value',
    'da_admin_suspended_debt_value',
    'da_enforceable_debt_ratio',
    'da_admin_suspended_ratio',
    'da_penalty_to_principal_ratio',
    'da_average_debt_age_months',
    'da_age_weighted_debt_value_months',
    'da_debt_0_6_months_value',
    'da_debt_over_48_months_value',
    'da_number_of_debts',
    'da_number_distinct_tax_types_count',
  
    
    # Capacidade de pagamento
    'ap_declared_gross_revedue_value',
    'ap_payment_capacity_value',
    'pc_total_payment_2025_value',
    'ap_debt_to_revenue_ratio',
    'ap_total_assets_value',
    
    # Histórico
    'pc_defaulted_instalment_plans_count',
    'pc_payment_regularity_ratio',
    'pc_instalment_plan_count',
    'pc_total_payment_2025_value',
    'pc_pending_filing_omissions_ratio',
  
    
    # Perfil
    'tp_enterprise_age_years',
    
    # Contextual
    'ci_sectoral_delinquency_deviation_score'
]

# Verificar quais variáveis existem no dataset
var_numericas_existentes = [v for v in var_numericas if v in df.columns]
var_categoricas_existentes = [v for v in var_categoricas if v in df.columns]
var_binarias_existentes = [v for v in var_binarias if v in df.columns]

print(f"Variáveis numéricas encontradas: {len(var_numericas_existentes)}")
print(f"Variáveis categóricas encontradas: {len(var_categoricas_existentes)}")
print(f"Variáveis binárias encontradas: {len(var_binarias_existentes)}")

### Distribuição variáveis categóricas

In [ ]:
# 5. CATEGORICAL VARIABLES DISTRIBUTION
print("### 5. CATEGORICAL VARIABLES DISTRIBUTION")
print()
categorical_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"Number of categorical variables: {len(categorical_cols)}")
print()

for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"   • {col}:")
    print(f"     Unique values: {unique_count}")
    if unique_count <= 10:
        value_counts = df[col].value_counts()
        for val, count in value_counts.items():
            pct = count / len(df) * 100
            print(f"       - {val}: {count} ({pct:.1f}%)")
    else:
        print(f"     Top 5 categories:")
        value_counts = df[col].value_counts().head(5)
        for val, count in value_counts.items():
            pct = count / len(df) * 100
            print(f"       - {val}: {count} ({pct:.1f}%)")
    print()

## Tratamento inconsistência - variáveis temporais

In [ ]:
# ============================================================================
# DIAGNÓSTICO: VALORES NEGATIVOS EM VARIÁVEIS DE IDADE
# ============================================================================

print("=" * 70)
print("DIAGNÓSTICO: VALORES NEGATIVOS EM VARIÁVEIS DE IDADE")
print("=" * 70)

# Variáveis a verificar
var_idade = ['da_average_debt_age_months', 'da_age_weighted_debt_value_months']

for var in var_idade:
    if var in df.columns:
        total = len(df)
        negativos = (df[var] < 0).sum()
        pct = (negativos / total) * 100
        
        print(f"\n{var}:")
        print(f"  Total de registros: {total:,}")
        print(f"  Valores negativos: {negativos:,} ({pct:.2f}%)")
        print(f"  Mínimo: {df[var].min():.2f}")
        print(f"  Máximo: {df[var].max():.2f}")
        
        if negativos > 0:
            print(f"\n  Distribuição dos valores negativos:")
            print(df[df[var] < 0][var].describe())
    else:
        print(f"\n⚠ {var}: NÃO ENCONTRADA NO DATASET")

In [ ]:
# ============================================================================
# TRATAMENTO: VALORES NEGATIVOS → 0 (inconsistência)
# ============================================================================
# 
# Justificativa:
# Valores negativos indicam débitos com "vencimento futuro" segundo o cálculo.
#
# Tratamento: Substituir por 0 (interpretar como "débito muito recente")
# Isso é conservador e não distorce a análise.
# ============================================================================

print("\n" + "=" * 70)
print("TRATAMENTO: VALORES NEGATIVOS")
print("=" * 70)

for var in var_idade:
    if var in df.columns:
        n_negativos = (df[var] < 0).sum()
        
        if n_negativos > 0:
            # Substituir negativos por 0
            df[var] = df[var].clip(lower=0)
            print(f"✓ {var}: {n_negativos:,} valores negativos → 0")
            print(f"  Novo mínimo: {df[var].min():.2f}")
        else:
            print(f"✓ {var}: Nenhum valor negativo")

## Tratamento de Missing Values

### Diagnóstico variáveis numéricas

In [ ]:
# ============================================================================
# DIAGNÓSTICO DE MISSING VALUES
# ============================================================================

def diagnostico_missing(df, variaveis):
    """Analisa missing values em um conjunto de variáveis."""
    
    missing_stats = []
    
    for var in variaveis:
        if var in df.columns:
            total = len(df)
            missing = df[var].isna().sum()
            pct = (missing / total) * 100
            
            missing_stats.append({
                'variavel': var,
                'total': total,
                'missing': missing,
                'pct_missing': pct,
                'cobertura': 100 - pct
            })
    
    resultado = pd.DataFrame(missing_stats)
    resultado = resultado.sort_values('pct_missing', ascending=False)
    
    return resultado

# Aplicar diagnóstico
missing_report = diagnostico_missing(df, var_numericas_existentes)

print("=" * 70)
print("RELATÓRIO DE MISSING VALUES - VARIÁVEIS NUMÉRICAS")
print("=" * 70)
print(missing_report.to_string(index=False))

# Visualização
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d73027' if x > 30 else '#fc8d59' if x > 10 else '#91cf60' 
          for x in missing_report['pct_missing']]
bars = ax.barh(missing_report['variavel'], missing_report['pct_missing'], color=colors)
ax.set_xlabel('% Missing')
ax.set_title('Percentual de Missing Values por Variável')
ax.axvline(x=30, color='red', linestyle='--', alpha=0.7, label='Threshold crítico (30%)')
ax.legend()
plt.tight_layout()
plt.show()

### Diagnóstico variáveis binárias

In [ ]:
# ============================================================================
# DIAGNÓSTICO DE MISSING VALUES VARIAVEIS BINARIAS
# ============================================================================

def diagnostico_missing(df, variaveis):
    """Analisa missing values em um conjunto de variáveis."""
    
    missing_stats = []
    
    for var in variaveis:
        if var in df.columns:
            total = len(df)
            missing = df[var].isna().sum()
            pct = (missing / total) * 100
            
            missing_stats.append({
                'variavel': var,
                'total': total,
                'missing': missing,
                'pct_missing': pct,
                'cobertura': 100 - pct
            })
    
    resultado = pd.DataFrame(missing_stats)
    resultado = resultado.sort_values('pct_missing', ascending=False)
    
    return resultado

# Aplicar diagnóstico
missing_report = diagnostico_missing(df, var_binarias_existentes)

print("=" * 70)
print("RELATÓRIO DE MISSING VALUES - VARIÁVEIS BINÁRIAS")
print("=" * 70)
print(missing_report.to_string(index=False))

# Visualização
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d73027' if x > 30 else '#fc8d59' if x > 10 else '#91cf60' 
          for x in missing_report['pct_missing']]
bars = ax.barh(missing_report['variavel'], missing_report['pct_missing'], color=colors)
ax.set_xlabel('% Missing')
ax.set_title('Percentual de Missing Values por Variável')
ax.axvline(x=30, color='red', linestyle='--', alpha=0.7, label='Threshold crítico (30%)')
ax.legend()
plt.tight_layout()
plt.show()

### Diagnóstico variáveis categóricas

In [ ]:
# ============================================================================
# DIAGNÓSTICO DE MISSING VALUES VARIAVEIS CATEGÓRICAS
# ============================================================================

def diagnostico_missing(df, variaveis):
    """Analisa missing values em um conjunto de variáveis."""
    
    missing_stats = []
    
    for var in variaveis:
        if var in df.columns:
            total = len(df)
            missing = df[var].isna().sum()
            pct = (missing / total) * 100
            
            missing_stats.append({
                'variavel': var,
                'total': total,
                'missing': missing,
                'pct_missing': pct,
                'cobertura': 100 - pct
            })
    
    resultado = pd.DataFrame(missing_stats)
    resultado = resultado.sort_values('pct_missing', ascending=False)
    
    return resultado

# Aplicar diagnóstico
missing_report = diagnostico_missing(df, var_categoricas_existentes)

print("=" * 70)
print("RELATÓRIO DE MISSING VALUES - VARIÁVEIS CATEGÓRICAS")
print("=" * 70)
print(missing_report.to_string(index=False))

# Visualização
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d73027' if x > 30 else '#fc8d59' if x > 10 else '#91cf60' 
          for x in missing_report['pct_missing']]
bars = ax.barh(missing_report['variavel'], missing_report['pct_missing'], color=colors)
ax.set_xlabel('% Missing')
ax.set_title('Percentual de Missing Values por Variável')
ax.axvline(x=30, color='red', linestyle='--', alpha=0.7, label='Threshold crítico (30%)')
ax.legend()
plt.tight_layout()
plt.show()

### Tratamento missing values

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# TRATAMENTO DE MISSING VALUES
# Pipeline baseado em lógica de negócio + imputação estatística
# =============================================================================

# Criar cópia de trabalho — df permanece intacto como referência
df_pos_missing = df.copy()

cols_antes = df_pos_missing.shape[1]
print(f"Dataset inicial: {df_pos_missing.shape[0]:,} registros | {cols_antes} variáveis")


# -----------------------------------------------------------------------------
# BLOCO 1 | REMOÇÃO
# ap_total_assets_value: cobertura de apenas 14,91% — viés de seleção severo,
# pois apenas empresas obrigadas à DCTF possuem este dado.
# Decisão: remoção da variável.
# -----------------------------------------------------------------------------

df_pos_missing.drop(columns=['ap_total_assets_value'], inplace=True)

print(f"\n[BLOCO 1] Remoção de ap_total_assets_value")
print(f"  Colunas restantes: {df_pos_missing.shape[1]}")


# -----------------------------------------------------------------------------
# BLOCO 2 | IMPUTAÇÃO LÓGICA — Variáveis de parcelamento
# Missing indica ausência de qualquer parcelamento registrado.
# A ausência é informativa e possui interpretação direta de negócio.
#
#   pc_instalment_plan_count          → 0  (nenhum parcelamento)
#   pc_payment_regularity_ratio       → 1  (sem parcelamento = sem irregularidade)
#   pc_defaulted_instalment_plans_count → 0  (sem parcelamento = sem inadimplência)
# -----------------------------------------------------------------------------

imputacao_logica = {
    'pc_instalment_plan_count':           0,
    'pc_payment_regularity_ratio':        1,
    'pc_defaulted_instalment_plans_count': 0,
}

for variavel, valor in imputacao_logica.items():
    n_missing = df_pos_missing[variavel].isna().sum()
    df_pos_missing[variavel] = df_pos_missing[variavel].fillna(valor)
    print(f"\n[BLOCO 2] {variavel}")
    print(f"  Valor imputado : {valor}")
    print(f"  Registros tratados: {n_missing:,}")


# -----------------------------------------------------------------------------
# BLOCO 3 | IMPUTAÇÃO PELA MEDIANA + FLAG DE AUSÊNCIA
# Variáveis com cobertura parcial (~61–80%) onde o missing pode ser
# informativo (ex.: ausência de declaração de receita, omissões fiscais).
# Estratégia: preservar o dado imputado E sinalizar a ausência original.
#
#   ap_debt_to_revenue_ratio          → flag: is_missing_ap_debt_to_revenue_ratio
#   ap_declared_gross_revedue_value   → flag: is_missing_ap_declared_gross_revenue_value
#   pc_pending_filing_omissions_ratio → flag: is_missing_pc_pending_filing_omissions_ratio
# -----------------------------------------------------------------------------

variaveis_com_flag = {
    'ap_debt_to_revenue_ratio':        'is_missing_ap_debt_to_revenue_ratio',
    'ap_declared_gross_revedue_value': 'is_missing_ap_declared_gross_revenue_value',
    'pc_pending_filing_omissions_ratio': 'is_missing_pc_pending_filing_omissions_ratio',
}

for variavel, nome_flag in variaveis_com_flag.items():
    
    n_missing = df_pos_missing[variavel].isna().sum()
    pct_missing = (n_missing / len(df_pos_missing)) * 100
    mediana = df_pos_missing[variavel].median()
    
    # Criar flag ANTES de imputar (preserva a informação original)
    df_pos_missing[nome_flag] = df_pos_missing[variavel].isna().astype(int)
    
    # Imputar mediana
    df_pos_missing[variavel] = df_pos_missing[variavel].fillna(mediana)
    
    print(f"\n[BLOCO 3] {variavel}")
    print(f"  Mediana imputada : {mediana:.4f}")
    print(f"  Registros tratados : {n_missing:,} ({pct_missing:.1f}%)")
    print(f"  Flag criada        : {nome_flag} (soma = {df_pos_missing[nome_flag].sum():,})")


# -----------------------------------------------------------------------------
# BLOCO 4 | IMPUTAÇÃO PELA MEDIANA — Sem flag
# Variáveis com cobertura elevada (>97%). A ausência não possui
# interpretação operacional distinta; imputação direta pela mediana é suficiente.
#
#   da_penalty_to_principal_ratio         (97,94% cobertura)
#   ci_sectoral_delinquency_deviation_score (99,92% cobertura)
# -----------------------------------------------------------------------------

variaveis_mediana_simples = [
    'da_penalty_to_principal_ratio',
    'ci_sectoral_delinquency_deviation_score',
]

for variavel in variaveis_mediana_simples:
    n_missing = df_pos_missing[variavel].isna().sum()
    mediana = df_pos_missing[variavel].median()
    df_pos_missing[variavel] = df_pos_missing[variavel].fillna(mediana)
    print(f"\n[BLOCO 4] {variavel}")
    print(f"  Mediana imputada   : {mediana:.4f}")
    print(f"  Registros tratados : {n_missing:,}")


# =============================================================================
# VERIFICAÇÃO FINAL
# =============================================================================

print("\n" + "=" * 65)
print("VERIFICAÇÃO FINAL — df_pos_missing")
print("=" * 65)

missing_final = df_pos_missing.isna().sum()
vars_com_missing = missing_final[missing_final > 0]

if len(vars_com_missing) == 0:
    print("✔  Nenhum missing value remanescente nas variáveis numéricas.")
else:
    print("⚠  Variáveis ainda com missing:")
    print(vars_com_missing)

print(f"\nDimensões finais : {df_pos_missing.shape[0]:,} registros | {df_pos_missing.shape[1]} variáveis")
print(f"Variáveis adicionadas (flags) : {df_pos_missing.shape[1] - cols_antes + 1}")  # +1 pela remoção
print(f"\nFlags de ausência criadas:")
for flag in variaveis_com_flag.values():
    print(f"  {flag}: {df_pos_missing[flag].sum():,} registros marcados")

In [ ]:
df_pos_missing.info()

## Tratamento Outliers

### Diagnóstico

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# =============================================================================
# DIAGNÓSTICO DE OUTLIERS — IQR 1.5 e 3.0
# Dataset: df_pos_missing
# =============================================================================

def diagnostico_outliers_iqr(df, variaveis, threshold):
    """
    Identifica outliers pelo método IQR para um dado threshold.
    Retorna DataFrame com estatísticas por variável.
    """
    resultados = []

    for var in variaveis:
        if var not in df.columns:
            continue

        serie = df[var].dropna()
        q1  = serie.quantile(0.25)
        q3  = serie.quantile(0.75)
        iqr = q3 - q1

        limite_inf = q1 - threshold * iqr
        limite_sup = q3 + threshold * iqr

        n_abaixo = (serie < limite_inf).sum()
        n_acima  = (serie > limite_sup).sum()
        n_total  = n_abaixo + n_acima
        pct      = (n_total / len(serie)) * 100

        resultados.append({
            'variavel':    var,
            'min':         round(serie.min(), 4),
            'q1':          round(q1, 4),
            'mediana':     round(serie.median(), 4),
            'media':       round(serie.mean(), 4),
            'q3':          round(q3, 4),
            'max':         round(serie.max(), 4),
            'iqr':         round(iqr, 4),
            'limite_inf':  round(limite_inf, 4),
            'limite_sup':  round(limite_sup, 4),
            'n_abaixo':    n_abaixo,
            'n_acima':     n_acima,
            'n_outliers':  n_total,
            'pct_outliers': round(pct, 2),
        })

    return pd.DataFrame(resultados).sort_values('pct_outliers', ascending=False)


# -----------------------------------------------------------------------------
# Filtrar apenas as variáveis numéricas que existem no dataset
# (exclui as flags binárias criadas no passo anterior)
# -----------------------------------------------------------------------------

flags_criadas = [
    'is_missing_ap_debt_to_revenue_ratio',
    'is_missing_ap_declared_gross_revenue_value',
    'is_missing_pc_pending_filing_omissions_ratio',
]

variaveis_para_diagnostico = [
    v for v in var_numericas_existentes
    if v in df_pos_missing.columns and v not in flags_criadas
]

print(f"Variáveis incluídas no diagnóstico: {len(variaveis_para_diagnostico)}")
print(variaveis_para_diagnostico)


# -----------------------------------------------------------------------------
# EXECUTAR DIAGNÓSTICO — IQR 1.5 e IQR 3.0
# -----------------------------------------------------------------------------

diag_15 = diagnostico_outliers_iqr(df_pos_missing, variaveis_para_diagnostico, threshold=1.5)
diag_30 = diagnostico_outliers_iqr(df_pos_missing, variaveis_para_diagnostico, threshold=3.0)


# -----------------------------------------------------------------------------
# TABELA COMPARATIVA — visão consolidada por variável
# -----------------------------------------------------------------------------

comparativo = diag_15[['variavel', 'mediana', 'media', 'min', 'max',
                         'limite_inf', 'limite_sup',
                         'n_outliers', 'pct_outliers']].copy()

comparativo = comparativo.rename(columns={
    'limite_inf':  'lim_inf_15',
    'limite_sup':  'lim_sup_15',
    'n_outliers':  'n_out_15',
    'pct_outliers':'pct_out_15',
})

# Adicionar colunas do IQR 3.0
comparativo['lim_inf_30'] = diag_30['limite_inf'].values
comparativo['lim_sup_30'] = diag_30['limite_sup'].values
comparativo['n_out_30']   = diag_30['n_outliers'].values
comparativo['pct_out_30'] = diag_30['pct_outliers'].values

# Diferença entre os dois thresholds
comparativo['delta_n']   = comparativo['n_out_15'] - comparativo['n_out_30']
comparativo['delta_pct'] = comparativo['pct_out_15'] - comparativo['pct_out_30']

print("\n" + "=" * 90)
print("DIAGNÓSTICO DE OUTLIERS — COMPARATIVO IQR 1.5 vs IQR 3.0")
print("=" * 90)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print(comparativo.to_string(index=False))


# -----------------------------------------------------------------------------
# CATEGORIZAÇÃO DE SEVERIDADE (baseada no IQR 3.0 — critério conservador)
# -----------------------------------------------------------------------------

def categorizar_severidade(pct):
    if pct == 0:
        return 'Sem outliers'
    elif pct <= 1:
        return 'Baixa   (≤1%)'
    elif pct <= 5:
        return 'Moderada (1–5%)'
    elif pct <= 10:
        return 'Alta    (5–10%)'
    else:
        return 'Crítica  (>10%)'

comparativo['severidade_30'] = comparativo['pct_out_30'].apply(categorizar_severidade)

print("\n" + "=" * 70)
print("CATEGORIZAÇÃO DE SEVERIDADE — IQR 3.0")
print("=" * 70)
print(comparativo[['variavel', 'pct_out_15', 'pct_out_30',
                    'delta_pct', 'severidade_30']].to_string(index=False))


# -----------------------------------------------------------------------------
# VISUALIZAÇÃO — Boxplots para cada variável
# -----------------------------------------------------------------------------

n_vars = len(variaveis_para_diagnostico)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, var in enumerate(variaveis_para_diagnostico):
    ax = axes[i]
    serie = df_pos_missing[var].dropna()

    ax.boxplot(serie, vert=True, patch_artist=True,
               boxprops=dict(facecolor='#d6e4f0', color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor='#e74c3c',
                               markersize=2, alpha=0.3),
               whiskerprops=dict(color='#2c3e50'),
               capprops=dict(color='#2c3e50'))

    pct_15 = comparativo.loc[comparativo['variavel'] == var, 'pct_out_15'].values[0]
    pct_30 = comparativo.loc[comparativo['variavel'] == var, 'pct_out_30'].values[0]

    ax.set_title(f"{var}\nIQR1.5: {pct_15:.1f}%  |  IQR3.0: {pct_30:.1f}%",
                 fontsize=8, pad=6)
    ax.set_xticks([])
    ax.grid(axis='y', linestyle='--', alpha=0.5)

# Ocultar eixos excedentes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots — Diagnóstico de Outliers por Variável", 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('diagnostico_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✔  Diagnóstico concluído. Compartilhe os resultados para definir o tratamento.")

### Tratamento Outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# TRATAMENTO DE OUTLIERS
# Pipeline baseado em análise de distribuição e lógica de negócio
# Dataset de entrada : df_pos_missing
# Dataset de saída   : df_pos_outlier
# =============================================================================

df_pos_outlier = df_pos_missing.copy()

print(f"Dataset inicial: {df_pos_outlier.shape[0]:,} registros | {df_pos_outlier.shape[1]} variáveis")
print("Nenhuma observação será removida — apenas transformações de escala.\n")


# =============================================================================
# GRUPO 1 — VALORES MONETÁRIOS: log1p
# Variáveis com distribuição fortemente assimétrica à direita (long-tail).
# log1p comprime a cauda sem descartar informação e preserva a ordem de grandeza.
# np.log1p = log(1 + x), seguro para zeros.
# Variáveis originais são substituídas; nomes recebem sufixo _log para rastreabilidade.
# =============================================================================

variaveis_monetarias = [
    'da_total_debt_value',
    'da_total_principal_value',
    'da_total_penalties_interest_value',
    'da_enforceable_debt_value',
    'da_admin_suspended_debt_value',
    'da_debt_0_6_months_value',
    'da_debt_over_48_months_value',
    'ap_declared_gross_revedue_value',
    'ap_payment_capacity_value',
    'pc_total_payment_2025_value',
    'da_debt_pgfn_value',
]

print("=" * 65)
print("GRUPO 1 — log1p | Valores monetários")
print("=" * 65)

colunas_log_monetarias = []

for var in variaveis_monetarias:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue

    # Garantir que não há valores negativos (erro de ETL)
    n_negativos = (df_pos_outlier[var] < 0).sum()
    if n_negativos > 0:
        print(f"  [AVISO] {var}: {n_negativos} valores negativos → clipping em 0 antes do log1p")
        df_pos_outlier[var] = df_pos_outlier[var].clip(lower=0)

    nome_log = f"{var}_log"
    df_pos_outlier[nome_log] = np.log1p(df_pos_outlier[var])
    colunas_log_monetarias.append(nome_log)

    skew_antes = df_pos_missing[var].skew()
    skew_depois = df_pos_outlier[nome_log].skew()
    print(f"  {var}")
    print(f"    → {nome_log} | skewness: {skew_antes:.2f} → {skew_depois:.2f}")

# Remover colunas originais (versão log é a que entra no clustering)
df_pos_outlier.drop(columns=variaveis_monetarias, inplace=True, errors='ignore')
print(f"\n  ✔  {len(colunas_log_monetarias)} variáveis monetárias transformadas e originais removidas.")


# =============================================================================
# GRUPO 2 — CONTAGENS ZERO-INFLADAS: log1p + winsorizing P99.5
# IQR não é diagnóstico válido aqui (IQR = 0 → qualquer positivo é "outlier").
# log1p preserva a intensidade do comportamento de parcelamento.
# Winsorizing suave no P99.5 protege contra eventuais erros de ETL na cauda extrema.
# =============================================================================

variaveis_contagens = [
    'pc_instalment_plan_count',
    'pc_defaulted_instalment_plans_count',
    'da_number_distinct_tax_types_count',
    'da_number_of_debts_count',
]

print("\n" + "=" * 65)
print("GRUPO 2 — log1p + winsorizing P99.5 | Contagens zero-infladas")
print("=" * 65)

colunas_log_contagens = []

for var in variaveis_contagens:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue

    # Winsorizing P99.5 na variável original antes do log
    p995 = df_pos_outlier[var].quantile(0.995)
    n_caps = (df_pos_outlier[var] > p995).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(upper=p995)

    nome_log = f"{var}_log"
    df_pos_outlier[nome_log] = np.log1p(df_pos_outlier[var])
    colunas_log_contagens.append(nome_log)

    print(f"  {var}")
    print(f"    P99.5 = {p995:.1f} | registros capados: {n_caps:,}")
    print(f"    → {nome_log} | skewness após log: {df_pos_outlier[nome_log].skew():.2f}")

df_pos_outlier.drop(columns=variaveis_contagens, inplace=True, errors='ignore')
print(f"\n  ✔  {len(colunas_log_contagens)} variáveis de contagem transformadas e originais removidas.")


# =============================================================================
# GRUPO 3 — RAZÕES LIMITADAS [0,1]: clip nos limites naturais, sem winsorizing
# O IQR detectou "outliers" espúrios por conta de distribuições bimodais.
# Apenas garantimos que os limites físicos [0,1] estão respeitados (proteção ETL).
# Estas variáveis entram no clustering sem transformação adicional.
# =============================================================================

variaveis_ratio_01 = [
    'pc_payment_regularity_ratio',
    'pc_pending_filing_omissions_ratio',
    'da_enforceable_debt_ratio',
    'da_admin_suspended_ratio',
]

print("\n" + "=" * 65)
print("GRUPO 3 — clip [0,1] | Razões com limites naturais")
print("=" * 65)

for var in variaveis_ratio_01:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue

    n_fora = ((df_pos_outlier[var] < 0) | (df_pos_outlier[var] > 1)).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(lower=0, upper=1)
    print(f"  {var} | valores fora de [0,1] corrigidos: {n_fora:,}")

print(f"\n  ✔  {len(variaveis_ratio_01)} variáveis validadas — sem winsorizing aplicado.")


# =============================================================================
# GRUPO 4 — RAZÕES SEM LIMITE SUPERIOR: log1p + winsorizing P99.5
# ap_debt_to_revenue_ratio e da_penalty_to_principal_ratio podem explodir
# quando o denominador se aproxima de zero — "distance hijackers" no K-means.
# Combinação log1p + P99.5 neutraliza o efeito sem descartar os casos.
# =============================================================================

variaveis_ratio_explosivas = [
    'ap_debt_to_revenue_ratio',
    'da_penalty_to_principal_ratio',
]

print("\n" + "=" * 65)
print("GRUPO 4 — log1p + winsorizing P99.5 | Razões sem limite superior")
print("=" * 65)

colunas_log_ratios = []

for var in variaveis_ratio_explosivas:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue

    # Clip em zero antes do log (denominador pequeno pode gerar negativos)
    df_pos_outlier[var] = df_pos_outlier[var].clip(lower=0)

    # Winsorizing P99.5 antes do log
    p995 = df_pos_outlier[var].quantile(0.995)
    n_caps = (df_pos_outlier[var] > p995).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(upper=p995)

    nome_log = f"{var}_log"
    df_pos_outlier[nome_log] = np.log1p(df_pos_outlier[var])
    colunas_log_ratios.append(nome_log)

    print(f"  {var}")
    print(f"    P99.5 = {p995:.4f} | registros capados: {n_caps:,}")
    print(f"    → {nome_log} | skewness após log: {df_pos_outlier[nome_log].skew():.2f}")

df_pos_outlier.drop(columns=variaveis_ratio_explosivas, inplace=True, errors='ignore')
print(f"\n  ✔  {len(colunas_log_ratios)} razões explosivas transformadas e originais removidas.")


# =============================================================================
# GRUPO 5 — VARIÁVEIS DE TEMPO E IDADE: sem transformação
# Caudas moderadas (IQR 3.0 < 3%), valores com interpretação direta de negócio.
# Escalonamento (RobustScaler) será aplicado na etapa de normalização.
# =============================================================================

variaveis_tempo = [
    'tp_enterprise_age_years',
    'da_average_debt_age_months',
    'da_age_weighted_debt_value_months',
]

print("\n" + "=" * 65)
print("GRUPO 5 — Sem transformação | Variáveis de tempo e idade")
print("=" * 65)

for var in variaveis_tempo:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue
    skew = df_pos_outlier[var].skew()
    alerta = " ⚠  skewness elevado — monitorar" if skew > 2 else ""
    print(f"  {var} | skewness: {skew:.2f}{alerta}")

print(f"\n  ✔  {len(variaveis_tempo)} variáveis mantidas sem transformação.")


# =============================================================================
# GRUPO 6 — SCORE CONTEXTUAL: sem transformação
# ci_sectoral_delinquency_deviation_score é um indicador setorial derivado —
# extremos refletem setores em crise real, não erros individuais.
# RobustScaler na etapa de normalização é suficiente.
# =============================================================================

variaveis_score = [
    'ci_sectoral_delinquency_deviation_score',
]

print("\n" + "=" * 65)
print("GRUPO 6 — Sem transformação | Score contextual setorial")
print("=" * 65)

for var in variaveis_score:
    if var not in df_pos_outlier.columns:
        print(f"  [AVISO] {var} não encontrada — ignorada.")
        continue
    print(f"  {var} | min: {df_pos_outlier[var].min():.2f} | max: {df_pos_outlier[var].max():.2f} | skewness: {df_pos_outlier[var].skew():.2f}")

print(f"\n  ✔  Score setorial mantido sem transformação.")


# =============================================================================
# ATUALIZAR LISTA DE VARIÁVEIS NUMÉRICAS
# Após as transformações, var_numericas_existentes precisa ser reconstruída
# para refletir os novos nomes (_log) e excluir as originais removidas.
# =============================================================================

# Variáveis que NÃO foram transformadas (entram com nome original)
vars_sem_transformacao = (
    variaveis_ratio_01 +
    variaveis_tempo +
    variaveis_score
)

# Variáveis transformadas (entram com sufixo _log)
vars_transformadas_log = (
    colunas_log_monetarias +
    colunas_log_contagens +
    colunas_log_ratios
)

# Flags de missing (criadas na etapa anterior — entram como estão)
flags_missing = [
    'is_missing_ap_debt_to_revenue_ratio',
    'is_missing_ap_declared_gross_revenue_value',
    'is_missing_pc_pending_filing_omissions_ratio',
]

# Lista consolidada de variáveis para as próximas etapas
var_numericas_pos_outlier = [
    v for v in vars_sem_transformacao + vars_transformadas_log + flags_missing
    if v in df_pos_outlier.columns
]

print("\n" + "=" * 65)
print("LISTA CONSOLIDADA — var_numericas_pos_outlier")
print("=" * 65)
for v in var_numericas_pos_outlier:
    print(f"  {v}")
print(f"\n  Total: {len(var_numericas_pos_outlier)} variáveis numéricas")


# =============================================================================
# VERIFICAÇÃO FINAL — distribuições após tratamento
# =============================================================================

print("\n" + "=" * 65)
print("VERIFICAÇÃO FINAL — Skewness antes e depois")
print("=" * 65)

# Construir mapeamento original → tratado para comparação
mapa_comparacao = {}
for var in variaveis_monetarias:
    mapa_comparacao[var] = f"{var}_log"
for var in variaveis_contagens:
    mapa_comparacao[var] = f"{var}_log"
for var in variaveis_ratio_explosivas:
    mapa_comparacao[var] = f"{var}_log"

print(f"\n{'Variável original':<45} {'Skew antes':>12} {'Skew depois':>12}")
print("-" * 70)
for var_orig, var_tratada in mapa_comparacao.items():
    if var_orig in df_pos_missing.columns and var_tratada in df_pos_outlier.columns:
        skew_antes  = df_pos_missing[var_orig].skew()
        skew_depois = df_pos_outlier[var_tratada].skew()
        status = "✔" if abs(skew_depois) < abs(skew_antes) else "⚠"
        print(f"  {status}  {var_orig:<43} {skew_antes:>10.2f}   {skew_depois:>10.2f}")


# =============================================================================
# VISUALIZAÇÃO — Histogramas comparativos (antes vs depois) para monetárias
# =============================================================================

vars_plot = [v for v in variaveis_monetarias[:6] if v in df_pos_missing.columns]

fig, axes = plt.subplots(len(vars_plot), 2, figsize=(14, len(vars_plot) * 3))

for i, var in enumerate(vars_plot):
    var_log = f"{var}_log"

    # Antes
    axes[i, 0].hist(df_pos_missing[var].dropna(), bins=60,
                    color='#c0392b', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i, 0].set_title(f"{var}\n(original | skew={df_pos_missing[var].skew():.1f})",
                         fontsize=8)
    axes[i, 0].set_ylabel("Frequência")

    # Depois
    axes[i, 1].hist(df_pos_outlier[var_log].dropna(), bins=60,
                    color='#27ae60', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i, 1].set_title(f"{var_log}\n(log1p | skew={df_pos_outlier[var_log].skew():.1f})",
                         fontsize=8)

plt.suptitle("Distribuições — Antes (original) vs Depois (log1p)",
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outliers_distribuicoes_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()


print("\n" + "=" * 65)
print("✔  Tratamento de outliers concluído.")
print(f"   Dataset df_pos_outlier: {df_pos_outlier.shape[0]:,} registros | {df_pos_outlier.shape[1]} colunas")
print(f"   Observações removidas: 0")
print("=" * 65)

In [ ]:
df_pos_outlier.info()

### Tratamento complementar

In [ ]:
# =============================================================================
# TRATAMENTO COMPLEMENTAR DE OUTLIERS — Desvios Residuais
# Dataset de entrada : df_pos_outlier
# Dataset de saída   : df_pos_outlier (atualizado in-place)
# As flags de missing são preservadas integralmente.
# =============================================================================

print("=" * 65)
print("TRATAMENTO COMPLEMENTAR — Skewness Residual")
print("=" * 65)

# Snapshot do skewness antes das correções (apenas variáveis a tratar)
variaveis_complementares = [
    'ci_sectoral_delinquency_deviation_score',
    'da_admin_suspended_debt_value_log',
    'ap_debt_to_revenue_ratio_log',
    'da_average_debt_age_months',
    'da_age_weighted_debt_value_months',
    'da_debt_over_48_months_value_log',
    'da_penalty_to_principal_ratio_log',
    'tp_enterprise_age_years',
]

print("\nSkewness ANTES da correção complementar:")
print(f"  {'Variável':<50} {'Skew':>8}")
print("  " + "-" * 58)
for var in variaveis_complementares:
    if var in df_pos_outlier.columns:
        print(f"  {var:<50} {df_pos_outlier[var].skew():>8.2f}")


# -----------------------------------------------------------------------------
# CORREÇÃO 1 — ci_sectoral_delinquency_deviation_score (skew 24.81)
# Variável com valores negativos → não se aplica log1p direto.
# Estratégia: winsorizing P99.5 → deslocar para positivo → log1p → substituir.
# O deslocamento é temporário (apenas para a transformação) e não altera
# a interpretação relativa entre devedores.
# -----------------------------------------------------------------------------

var = 'ci_sectoral_delinquency_deviation_score'

if var in df_pos_outlier.columns:
    # Winsorizing P99.5
    p995 = df_pos_outlier[var].quantile(0.995)
    n_caps = (df_pos_outlier[var] > p995).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(upper=p995)

    # Deslocar para positivo e aplicar log1p
    min_val = df_pos_outlier[var].min()
    deslocamento = abs(min_val) + 1  # garante mínimo = 1 antes do log
    nome_log = f"{var}_log"
    df_pos_outlier[nome_log] = np.log1p(df_pos_outlier[var] + deslocamento)

    # Remover original; atualizar lista
    df_pos_outlier.drop(columns=[var], inplace=True)

    print(f"\n[C1] {var}")
    print(f"     P99.5 = {p995:.4f} | capados: {n_caps:,}")
    print(f"     Deslocamento aplicado: +{deslocamento:.4f}")
    print(f"     → {nome_log} | skewness: {df_pos_outlier[nome_log].skew():.2f}")


# -----------------------------------------------------------------------------
# CORREÇÃO 2 — Variáveis de tempo/idade reclassificadas (skew > 3)
# tp_enterprise_age_years, da_average_debt_age_months,
# da_age_weighted_debt_value_months
# Todas têm mínimo = 0 → log1p direto é seguro.
# Substituem a versão original no dataset.
# -----------------------------------------------------------------------------

variaveis_tempo_log = [
    'tp_enterprise_age_years',
    'da_average_debt_age_months',
    'da_age_weighted_debt_value_months',
]


print(f"\n[C2] Transformação log1p — Variáveis de tempo reclassificadas")

colunas_tempo_log = []
for var in variaveis_tempo_log:
    if var not in df_pos_outlier.columns:
        print(f"     [AVISO] {var} não encontrada.")
        continue

    # Garantir sem negativos
    df_pos_outlier[var] = df_pos_outlier[var].clip(lower=0)

    nome_log = f"{var}_log"
    df_pos_outlier[nome_log] = np.log1p(df_pos_outlier[var])
    colunas_tempo_log.append((var, nome_log))

    skew_depois = df_pos_outlier[nome_log].skew()
    print(f"     {var} → {nome_log} | skewness: {skew_depois:.2f}")

# Remover originais
originais_tempo = [v for v, _ in colunas_tempo_log]
df_pos_outlier.drop(columns=originais_tempo, inplace=True, errors='ignore')


# -----------------------------------------------------------------------------
# CORREÇÃO 3 — Winsorizing P99 sobre variáveis já em escala log
# Aplica-se a variáveis cujo log1p foi insuficiente por causa de cauda
# residual após a transformação.
# Operamos diretamente sobre as colunas _log existentes.
# -----------------------------------------------------------------------------

variaveis_wins_sobre_log = [
    'da_admin_suspended_debt_value_log',   # skew 7.28
    'ap_debt_to_revenue_ratio_log',        # skew 6.10
    'da_debt_over_48_months_value_log',    # skew 3.15
    'da_penalty_to_principal_ratio_log',   # skew 3.43
]

print(f"\n[C3] Winsorizing P99 sobre variáveis em escala log")

for var in variaveis_wins_sobre_log:
    if var not in df_pos_outlier.columns:
        print(f"     [AVISO] {var} não encontrada.")
        continue

    skew_antes = df_pos_outlier[var].skew()
    p99 = df_pos_outlier[var].quantile(0.99)
    n_caps = (df_pos_outlier[var] > p99).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(upper=p99)
    skew_depois = df_pos_outlier[var].skew()

    print(f"     {var}")
    print(f"       P99 (escala log) = {p99:.4f} | capados: {n_caps:,}")
    print(f"       skewness: {skew_antes:.2f} → {skew_depois:.2f}")


# =============================================================================
# RECONSTRUIR var_numericas_pos_outlier
# Reflete todas as renomeações e adições desta etapa complementar.
# Flags de missing são incluídas explicitamente e preservadas.
# =============================================================================

flags_missing = [
    'is_missing_ap_debt_to_revenue_ratio',
    'is_missing_ap_declared_gross_revenue_value',
    'is_missing_pc_pending_filing_omissions_ratio',
]

# Todas as colunas numéricas do dataset (exceto flags e cnpj/identificadores)
var_numericas_pos_outlier = [
    col for col in df_pos_outlier.columns
    if col not in flags_missing
    and df_pos_outlier[col].dtype in [np.float64, np.float32, np.int64, np.int32]
]

# Lista final consolidada: numéricas + flags
var_pos_outlier_completa = var_numericas_pos_outlier + [
    f for f in flags_missing if f in df_pos_outlier.columns
]

exclude_cols = set(flags_missing + ['cnpj8', 'id_cnpj', 'cluster', 'label'])
var_numericas_pos_outlier = [
    col for col in df_pos_outlier.columns
    if col not in exclude_cols
    and np.issubdtype(df_pos_outlier[col].dtype, np.number)
]


for f in flags_missing:
    if f in df_pos_outlier.columns:
        df_pos_outlier[f] = df_pos_outlier[f].fillna(0).astype(int)

        
# =============================================================================
# VERIFICAÇÃO FINAL — Skewness após todas as correções
# =============================================================================

print("\n" + "=" * 65)
print("VERIFICAÇÃO FINAL — Skewness após tratamento complementar")
print("=" * 65)

print(f"\n  {'Variável':<52} {'Skew':>8}  {'Status'}")
print("  " + "-" * 70)

alertas = []
for var in var_numericas_pos_outlier:
    if var in df_pos_outlier.columns:
        skew = df_pos_outlier[var].skew()
        if abs(skew) <= 2:
            status = "✔  aceitável"
        elif abs(skew) <= 3:
            status = "~  moderado"
        else:
            status = "⚠  elevado"
            alertas.append((var, skew))
        print(f"  {var:<52} {skew:>8.2f}  {status}")

print(f"\n  Dataset df_pos_outlier: {df_pos_outlier.shape[0]:,} registros | {df_pos_outlier.shape[1]} colunas")
print(f"  Variáveis numéricas   : {len(var_numericas_pos_outlier)}")
print(f"  Flags de missing      : {len([f for f in flags_missing if f in df_pos_outlier.columns])}")
print(f"  Observações removidas : 0")

if alertas:
    print(f"\n  Variáveis ainda com skewness elevado (|skew| > 3):")
    for var, skew in alertas:
        print(f"    {var}: {skew:.2f}")
    print("  → Avaliar impacto no clustering após normalização (RobustScaler mitiga parcialmente).")
else:
    print("\n  ✔  Todas as variáveis com skewness dentro da faixa aceitável.")

print("\n" + "=" * 65)
print("LISTA FINAL — var_pos_outlier_completa")
print("=" * 65)
for v in var_pos_outlier_completa:
    tipo = "[flag]" if v in flags_missing else ""
    print(f"  {v:<55} {tipo}")
print(f"\n  Total: {len(var_pos_outlier_completa)} variáveis "
      f"({len(var_numericas_pos_outlier)} numéricas + "
      f"{len([f for f in flags_missing if f in df_pos_outlier.columns])} flags)")

### Tratamento complementar 2

In [ ]:
# =============================================================================
# CORREÇÃO PONTUAL — ap_debt_to_revenue_ratio_log
# Winsorizing mais agressivo (P95) na escala log.
# P99 foi insuficiente por conta de cauda residual densa após a transformação.
# =============================================================================

var = 'ap_debt_to_revenue_ratio_log'

if var in df_pos_outlier.columns:
    skew_antes = df_pos_outlier[var].skew()
    p95 = df_pos_outlier[var].quantile(0.95)
    n_caps = (df_pos_outlier[var] > p95).sum()
    df_pos_outlier[var] = df_pos_outlier[var].clip(upper=p95)
    skew_depois = df_pos_outlier[var].skew()

    print(f"[Correção] {var}")
    print(f"  P95 (escala log) = {p95:.4f} | registros capados: {n_caps:,}")
    print(f"  skewness: {skew_antes:.2f} → {skew_depois:.2f}")


# =============================================================================
# RECONSTRUÇÃO DA LISTA FINAL DE VARIÁVEIS
# Exclui: id_cnpj e quaisquer outros identificadores
# Inclui: variáveis numéricas contínuas + flags de missing
# Variáveis binárias (bin) são mantidas separadas para uso na
# caracterização pós-clustering, não entram no espaço de features.
# =============================================================================

flags_missing = [
    'is_missing_ap_debt_to_revenue_ratio',
    'is_missing_ap_declared_gross_revenue_value',
    'is_missing_pc_pending_filing_omissions_ratio',
]

# Variáveis binárias — usadas na caracterização, não no clustering
variaveis_bin = [
    col for col in df_pos_outlier.columns
    if col.endswith('_bin')
]

# Identificadores — excluídos de todas as listas analíticas
identificadores = ['id_cnpj']

# Variáveis numéricas contínuas para o clustering
var_numericas_pos_outlier = [
    col for col in df_pos_outlier.columns
    if col not in flags_missing
    and col not in variaveis_bin
    and col not in identificadores
    and df_pos_outlier[col].dtype in [np.float64, np.float32, np.int64, np.int32]
]

# Lista completa: numéricas contínuas + flags de missing
var_pos_outlier_completa = var_numericas_pos_outlier + [
    f for f in flags_missing if f in df_pos_outlier.columns
]

# =============================================================================
# RELATÓRIO FINAL CONSOLIDADO
# =============================================================================

print("\n" + "=" * 70)
print("RELATÓRIO FINAL — df_pos_outlier")
print("=" * 70)

print(f"\n{'Variável':<55} {'Skew':>8}  {'Status'}")
print("-" * 72)

for var in var_numericas_pos_outlier:
    if var not in df_pos_outlier.columns:
        continue
    skew = df_pos_outlier[var].skew()
    if abs(skew) <= 2:
        status = "✔  aceitável"
    elif abs(skew) <= 3:
        status = "~  moderado"
    else:
        status = "⚠  aceito — estrutural"
    print(f"  {var:<53} {skew:>8.2f}  {status}")

print(f"\n{'─'*72}")
print(f"\n  Flags de missing (preservadas):")
for f in flags_missing:
    if f in df_pos_outlier.columns:
        print(f"    {f}")

print(f"\n  Variáveis binárias (caracterização pós-clustering):")
for v in variaveis_bin:
    print(f"    {v}")

print(f"\n{'─'*72}")
print(f"  Dataset final  : {df_pos_outlier.shape[0]:,} registros | {df_pos_outlier.shape[1]} colunas")
print(f"  Var. contínuas : {len(var_numericas_pos_outlier)}")
print(f"  Flags missing  : {len([f for f in flags_missing if f in df_pos_outlier.columns])}")
print(f"  Var. binárias  : {len(variaveis_bin)} (separadas — não entram no clustering)")
print(f"  Identificadores: {identificadores} (excluídos)")
print(f"  Obs. removidas : 0")

### Recostrução do dataset para normalização

In [ ]:
# =============================================================================
# LISTAS DEFINITIVAS PARA NORMALIZAÇÃO E CLUSTERING
# =============================================================================

identificadores = ['id_cnpj']

# Variáveis que PASSAM pelo scaler
var_para_escalar = [
    col for col in df_pos_outlier.columns
    if col not in identificadores
    and not col.endswith('_bin')
    and not col.startswith('is_missing_')
    and df_pos_outlier[col].dtype in [np.float64, np.float32, np.int64, np.int32]
]

# Variáveis binárias e flags — entram no clustering, mas NÃO no scaler
var_binarias_e_flags = [
    col for col in df_pos_outlier.columns
    if col.endswith('_bin') or col.startswith('is_missing_')
]

# Lista completa para o clustering (scaled + não scaled)
var_clustering_completa = var_para_escalar + var_binarias_e_flags

print("Variáveis para escalar (contínuas):")
for v in var_para_escalar:
    print(f"  {v}")

print(f"\nVariáveis binárias e flags (sem escalar):")
for v in var_binarias_e_flags:
    print(f"  {v}")

print(f"\nTotal para clustering : {len(var_clustering_completa)}")
print(f"  → Contínuas (scaler): {len(var_para_escalar)}")
print(f"  → Binárias/flags    : {len(var_binarias_e_flags)}")

## Normalização

### NORMALIZAÇÃO — RobustScaler

In [ ]:
from sklearn.preprocessing import RobustScaler
import pandas as pd
import numpy as np

# =============================================================================
# NORMALIZAÇÃO — RobustScaler
# Aplicado exclusivamente às variáveis contínuas.
# Variáveis binárias (_bin) e flags de missing (is_missing_*) preservadas
# na escala original e concatenadas ao final.
# Dataset de entrada : df_pos_outlier
# Dataset de saída   : df_normalizado
# =============================================================================

identificadores = ['id_cnpj']

# -----------------------------------------------------------------------------
# Separar os três grupos
# -----------------------------------------------------------------------------

var_continuas = [
    col for col in df_pos_outlier.columns
    if col not in identificadores
    and not col.endswith('_bin')
    and not col.startswith('is_missing_')
    and df_pos_outlier[col].dtype in [np.float64, np.float32, np.int64, np.int32]
]

var_binarias = [
    col for col in df_pos_outlier.columns
    if col.endswith('_bin')
]

var_flags = [
    col for col in df_pos_outlier.columns
    if col.startswith('is_missing_')
]

print("=" * 65)
print("NORMALIZAÇÃO — RobustScaler")
print("=" * 65)
print(f"\n  Variáveis contínuas (scaler)  : {len(var_continuas)}")
print(f"  Variáveis binárias (sem scaler): {len(var_binarias)}")
print(f"  Flags de missing (sem scaler)  : {len(var_flags)}")

# Verificação: confirmar que são exatamente 25 contínuas
assert len(var_continuas) == 25, (
    f"Esperado 25 variáveis contínuas, encontrado {len(var_continuas)}. "
    f"Verificar lista: {var_continuas}"
)
print(f"\n  ✔  Confirmado: {len(var_continuas)} variáveis contínuas.")


# -----------------------------------------------------------------------------
# Aplicar RobustScaler
# RobustScaler: subtrai a mediana e divide pelo IQR (Q3 - Q1).
# Robusto a outliers residuais — adequado para os desvios estruturais
# identificados nas etapas anteriores (zero-inflation, bimodalidade).
# -----------------------------------------------------------------------------

scaler = RobustScaler()

# Fit e transform nas contínuas
X_scaled = scaler.fit_transform(df_pos_outlier[var_continuas])
df_scaled = pd.DataFrame(
    X_scaled,
    columns=var_continuas,
    index=df_pos_outlier.index
)

print(f"\n  RobustScaler ajustado em {len(var_continuas)} variáveis.")


# -----------------------------------------------------------------------------
# Montar dataset final: scaled + binárias + flags + identificador
# -----------------------------------------------------------------------------

df_normalizado = pd.concat(
    [
        df_pos_outlier[identificadores],          # id_cnpj — referência
        df_scaled,                                # contínuas escaladas
        df_pos_outlier[var_binarias],             # binárias — escala original
        df_pos_outlier[var_flags],                # flags missing — escala original
    ],
    axis=1
)

# Lista de variáveis para o clustering (exclui identificador)
var_clustering_final = var_continuas + var_binarias + var_flags


# -----------------------------------------------------------------------------
# Verificação pós-scaling
# -----------------------------------------------------------------------------

print("\n" + "=" * 65)
print("VERIFICAÇÃO — Estatísticas após RobustScaler")
print("=" * 65)

stats_scaled = df_normalizado[var_continuas].agg(
    ['median', 'mean', 'std', 'min', 'max']
).T.round(4)

stats_scaled.columns = ['mediana', 'média', 'std', 'min', 'max']

print(f"\n  {'Variável':<52} {'Mediana':>8} {'Std':>8} {'Min':>10} {'Max':>10}")
print("  " + "-" * 88)
for var in var_continuas:
    med = df_normalizado[var].median()
    std = df_normalizado[var].std()
    mn  = df_normalizado[var].min()
    mx  = df_normalizado[var].max()
    print(f"  {var:<52} {med:>8.4f} {std:>8.4f} {mn:>10.4f} {mx:>10.4f}")

# Confirmar mediana ≈ 0 em todas (propriedade do RobustScaler)
medianas_fora = [
    v for v in var_continuas
    if abs(df_normalizado[v].median()) > 0.05
]
if medianas_fora:
    print(f"\n  ⚠  Variáveis com mediana fora de ≈0 após scaling:")
    for v in medianas_fora:
        print(f"     {v}: {df_normalizado[v].median():.4f}")
else:
    print(f"\n  ✔  Todas as medianas ≈ 0 após RobustScaler.")


# Confirmar binárias e flags inalteradas
print(f"\n  Verificação — binárias e flags inalteradas:")
for var in var_binarias + var_flags:
    valores_unicos = sorted(df_normalizado[var].unique())
    print(f"    {var:<52} valores: {valores_unicos}")


# -----------------------------------------------------------------------------
# Resumo final
# -----------------------------------------------------------------------------

print("\n" + "=" * 65)
print("RESUMO FINAL — df_normalizado")
print("=" * 65)
print(f"\n  Registros         : {df_normalizado.shape[0]:,}")
print(f"  Colunas totais    : {df_normalizado.shape[1]}")
print(f"  Identificador     : {identificadores}")
print(f"  Var. clustering   : {len(var_clustering_final)}")
print(f"    → Contínuas (RobustScaler) : {len(var_continuas)}")
print(f"    → Binárias (_bin)          : {len(var_binarias)}")
print(f"    → Flags missing            : {len(var_flags)}")
print(f"\n  Scaler salvo em: 'scaler' (RobustScaler)")
print(f"  → Usar scaler.transform() para classificar novos devedores.")
print(f"\n  ✔  df_normalizado pronto para clustering.")

### NORMALIZAÇÃO — correção ci_sectoral_delinquency_deviation_score

In [ ]:
# =============================================================================
# CORREÇÃO PONTUAL — ci_sectoral_delinquency_deviation_score_log
# Problema: cauda esquerda extrema (min = -93.81) após RobustScaler.
# Estratégia: winsorizing bilateral (P0.5 / P99.5) sobre a variável
# já normalizada — mais preciso do que refazer todo o pipeline.
# =============================================================================

var = 'ci_sectoral_delinquency_deviation_score_log'

if var in df_normalizado.columns:

    skew_antes = df_normalizado[var].skew()
    min_antes  = df_normalizado[var].min()
    max_antes  = df_normalizado[var].max()

    p005 = df_normalizado[var].quantile(0.005)
    p995 = df_normalizado[var].quantile(0.995)

    n_inf = (df_normalizado[var] < p005).sum()
    n_sup = (df_normalizado[var] > p995).sum()

    df_normalizado[var] = df_normalizado[var].clip(lower=p005, upper=p995)

    skew_depois = df_normalizado[var].skew()
    min_depois  = df_normalizado[var].min()
    max_depois  = df_normalizado[var].max()

    print(f"[Correção] {var}")
    print(f"  Winsorizing bilateral P0.5 / P99.5 (sobre escala normalizada)")
    print(f"  Limite inferior : {min_antes:.4f} → {min_depois:.4f} | capados: {n_inf:,}")
    print(f"  Limite superior : {max_antes:.4f} → {max_depois:.4f} | capados: {n_sup:,}")
    print(f"  Skewness        : {skew_antes:.2f} → {skew_depois:.2f}")


# =============================================================================
# VERIFICAÇÃO FINAL CONSOLIDADA
# =============================================================================

print("\n" + "=" * 72)
print("VERIFICAÇÃO FINAL — df_normalizado (após correção)")
print("=" * 72)

var_continuas_check = [
    col for col in df_normalizado.columns
    if not col.endswith('_bin')
    and not col.startswith('is_missing_')
    and col != 'id_cnpj'
]

print(f"\n  {'Variável':<52} {'Std':>7} {'Min':>10} {'Max':>10}  Status")
print("  " + "-" * 88)

alertas_range = []
for var in var_continuas_check:
    std = df_normalizado[var].std()
    mn  = df_normalizado[var].min()
    mx  = df_normalizado[var].max()
    amplitude = max(abs(mn), abs(mx))

    if amplitude > 20:
        status = "⚠  range elevado"
        alertas_range.append(var)
    else:
        status = "✔"

    print(f"  {var:<52} {std:>7.4f} {mn:>10.4f} {mx:>10.4f}  {status}")

if not alertas_range:
    print(f"\n  ✔  Todas as variáveis com range aceitável após correção.")
else:
    print(f"\n  ⚠  Variáveis ainda com range elevado: {alertas_range}")
    print(f"     Monitorar impacto nos centróides durante o clustering.")

print(f"\n  Dataset df_normalizado: {df_normalizado.shape[0]:,} registros | "
      f"{df_normalizado.shape[1]} colunas")
print(f"  ✔  Pronto para seleção de variáveis e clustering.")

## Determinação do número de clusters - K (CORRIGIDO - FOI FEITA ANÁLISE DE CORRELAÇÃO E SELEÇÃO DE FEATURES)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from kneed import KneeLocator
from collections import Counter

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K_MIN        = 2
K_MAX        = 10
k_range      = range(K_MIN, K_MAX + 1)
N_SEEDS      = 10        # seeds para análise de estabilidade
SAMPLE_SIL   = 30_000   # amostra para silhouette (eficiência computacional)

# =============================================================================
# MATRIZ DE FEATURES PARA CLUSTERING
# Contínuas escaladas + binárias + flags (na escala original)
# id_cnpj excluído
# =============================================================================

X = df_normalizado[var_clustering_final].values

print("=" * 70)
print("DETERMINAÇÃO DO NÚMERO ÓTIMO DE CLUSTERS — K Selection")
print("=" * 70)
print(f"\n  Dataset           : {X.shape[0]:,} observações | {X.shape[1]} variáveis")
print(f"  Range testado     : k = {K_MIN} a {K_MAX}")
print(f"  Seeds estabilidade: {N_SEEDS}")
print(f"  Amostra silhouette: {SAMPLE_SIL:,}")


# =============================================================================
# BLOCO 1 — ELBOW (Inércia / WCSS)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 1 — Método do Cotovelo (Elbow / WCSS)")
print("─" * 70)

inertias = []

for k in k_range:
    t0 = time.time()
    km = KMeans(n_clusters=k, init='k-means++', n_init=15,
                max_iter=300, random_state=RANDOM_STATE)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f"  k={k:2d} | Inércia: {km.inertia_:>18,.2f} | {time.time()-t0:.1f}s")

# Detecção automática do cotovelo
kneedle = KneeLocator(list(k_range), inertias,
                      curve='convex', direction='decreasing')
k_elbow = kneedle.elbow
print(f"\n  ✔  Cotovelo detectado: k = {k_elbow}")


# =============================================================================
# BLOCO 2 — MÉTRICAS DE VALIDAÇÃO INTERNA
# Silhouette, Davies-Bouldin, Calinski-Harabasz
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Métricas de Validação Interna")
print("─" * 70)

sil_scores = []
db_scores  = []
ch_scores  = []

# Amostra estratificada para silhouette (evita custo O(n²))
rng = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=min(SAMPLE_SIL, X.shape[0]), replace=False)
X_sample   = X[idx_sample]

for k in k_range:
    t0 = time.time()

    km     = KMeans(n_clusters=k, init='k-means++', n_init=15,
                    max_iter=300, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)

    # Silhouette sobre amostra
    labels_sample = labels[idx_sample]
    sil = silhouette_score(X_sample, labels_sample, random_state=RANDOM_STATE)

    db  = davies_bouldin_score(X, labels)
    ch  = calinski_harabasz_score(X, labels)

    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)

    print(f"  k={k:2d} | Silhouette: {sil:.4f} | "
          f"Davies-Bouldin: {db:.4f} | "
          f"Calinski-Harabasz: {ch:>12,.1f} | {time.time()-t0:.1f}s")

k_silhouette = list(k_range)[np.argmax(sil_scores)]
k_db         = list(k_range)[np.argmin(db_scores)]
k_ch         = list(k_range)[np.argmax(ch_scores)]

print(f"\n  Melhor Silhouette       : k = {k_silhouette} ({max(sil_scores):.4f})")
print(f"  Melhor Davies-Bouldin   : k = {k_db}  ({min(db_scores):.4f})")
print(f"  Melhor Calinski-Harabasz: k = {k_ch}  ({max(ch_scores):,.1f})")


# =============================================================================
# BLOCO 3 — ANÁLISE DE ESTABILIDADE
# Repete o K-Means com N_SEEDS diferentes para cada k.
# Mede a variação da inércia — quanto menor o desvio padrão,
# mais estável e reprodutível é a solução.
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Análise de Estabilidade (variância entre seeds)")
print("─" * 70)

seeds = [42, 7, 13, 21, 99, 123, 256, 512, 1024, 2048][:N_SEEDS]

estabilidade = {}

for k in k_range:
    inertias_k = []
    for seed in seeds:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    max_iter=300, random_state=seed)
        km.fit(X)
        inertias_k.append(km.inertia_)

    media  = np.mean(inertias_k)
    desvio = np.std(inertias_k)
    cv     = (desvio / media) * 100  # coeficiente de variação em %

    estabilidade[k] = {
        'media':  media,
        'desvio': desvio,
        'cv':     cv,
        'inertias': inertias_k
    }

    estab_label = "✔  estável" if cv < 0.5 else ("~  moderado" if cv < 2 else "⚠  instável")
    print(f"  k={k:2d} | Inércia média: {media:>18,.2f} | "
          f"Std: {desvio:>12,.2f} | CV: {cv:.4f}%  {estab_label}")


# =============================================================================
# BLOCO 4 — VOTAÇÃO E SÍNTESE
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Síntese e Consenso")
print("─" * 70)

votos = Counter([k_elbow, k_silhouette, k_db, k_ch])
k_consenso = votos.most_common(1)[0][0]
n_votos    = votos.most_common(1)[0][1]

print(f"\n  Método do Cotovelo (Elbow)  : k = {k_elbow}")
print(f"  Silhouette Score            : k = {k_silhouette}")
print(f"  Davies-Bouldin Index        : k = {k_db}")
print(f"  Calinski-Harabasz Index     : k = {k_ch}")

if n_votos >= 3:
    consenso_label = f"✔  CONSENSO FORTE ({n_votos}/4 métodos)"
elif n_votos == 2:
    consenso_label = f"~  CONSENSO PARCIAL ({n_votos}/4 métodos)"
else:
    consenso_label = "⚠  SEM CONSENSO — avaliar estabilidade e interpretabilidade"

print(f"\n  {consenso_label}")
print(f"  k recomendado: {k_consenso}")


# =============================================================================
# BLOCO 5 — TABELA RESUMO CONSOLIDADA
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 5 — Tabela Resumo")
print("─" * 70)

resumo = pd.DataFrame({
    'k':                list(k_range),
    'inertia':          inertias,
    'silhouette':       sil_scores,
    'davies_bouldin':   db_scores,
    'calinski_harabasz': ch_scores,
    'estab_cv_pct':     [estabilidade[k]['cv'] for k in k_range],
})

# Ranking normalizado (0=pior, 1=melhor)
resumo['rank_sil']    = (resumo['silhouette']        - resumo['silhouette'].min())        / (resumo['silhouette'].max()        - resumo['silhouette'].min())
resumo['rank_db']     = 1 - (resumo['davies_bouldin'] - resumo['davies_bouldin'].min())   / (resumo['davies_bouldin'].max()   - resumo['davies_bouldin'].min())
resumo['rank_ch']     = (resumo['calinski_harabasz'] - resumo['calinski_harabasz'].min()) / (resumo['calinski_harabasz'].max() - resumo['calinski_harabasz'].min())
resumo['rank_estab']  = 1 - (resumo['estab_cv_pct']  - resumo['estab_cv_pct'].min())      / (resumo['estab_cv_pct'].max()     - resumo['estab_cv_pct'].min())
resumo['score_total'] = resumo[['rank_sil','rank_db','rank_ch','rank_estab']].mean(axis=1)

print(f"\n  {'k':>3} {'Inércia':>20} {'Silhouette':>12} {'DB':>10} "
      f"{'CH':>14} {'CV%':>8} {'Score':>8}")
print("  " + "─" * 76)

for _, row in resumo.iterrows():
    marcador = " ◄" if int(row['k']) == k_consenso else ""
    print(f"  {int(row['k']):>3} {row['inertia']:>20,.2f} {row['silhouette']:>12.4f} "
          f"{row['davies_bouldin']:>10.4f} {row['calinski_harabasz']:>14,.1f} "
          f"{row['estab_cv_pct']:>8.4f} {row['score_total']:>8.4f}{marcador}")

k_score = resumo.loc[resumo['score_total'].idxmax(), 'k']
print(f"\n  k com maior score composto: {int(k_score)}")


# =============================================================================
# BLOCO 6 — VISUALIZAÇÕES
# =============================================================================

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

ks = list(k_range)
cor_principal = '#2c3e50'
cor_destaque  = '#e74c3c'

# --- 1. Elbow ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ks, inertias, 'o-', color=cor_principal, linewidth=2, markersize=7)
if k_elbow:
    ax1.axvline(k_elbow, color=cor_destaque, linestyle='--', lw=1.8,
                label=f'k={k_elbow}')
    ax1.legend(fontsize=9)
ax1.set_title('Elbow (Inércia)', fontweight='bold')
ax1.set_xlabel('k'); ax1.set_ylabel('WCSS')
ax1.set_xticks(ks); ax1.grid(alpha=0.3)

# --- 2. Silhouette ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ks, sil_scores, 'o-', color='#27ae60', linewidth=2, markersize=7)
ax2.axvline(k_silhouette, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'k={k_silhouette}')
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.set_xlabel('k'); ax2.set_ylabel('Score')
ax2.set_xticks(ks); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

# --- 3. Davies-Bouldin ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ks, db_scores, 'o-', color='#8e44ad', linewidth=2, markersize=7)
ax3.axvline(k_db, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'k={k_db}')
ax3.set_title('Davies-Bouldin (↓ melhor)', fontweight='bold')
ax3.set_xlabel('k'); ax3.set_ylabel('Índice')
ax3.set_xticks(ks); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

# --- 4. Calinski-Harabasz ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(ks, ch_scores, 'o-', color='#e67e22', linewidth=2, markersize=7)
ax4.axvline(k_ch, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'k={k_ch}')
ax4.set_title('Calinski-Harabasz (↑ melhor)', fontweight='bold')
ax4.set_xlabel('k'); ax4.set_ylabel('Índice')
ax4.set_xticks(ks); ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# --- 5. Estabilidade (CV%) ---
cvs = [estabilidade[k]['cv'] for k in k_range]
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(ks, cvs, 'o-', color='#16a085', linewidth=2, markersize=7)
ax5.axhline(0.5, color=cor_destaque, linestyle=':', lw=1.5,
            label='Threshold 0.5%')
ax5.set_title('Estabilidade — CV% Inércia (↓ melhor)', fontweight='bold')
ax5.set_xlabel('k'); ax5.set_ylabel('CV (%)')
ax5.set_xticks(ks); ax5.legend(fontsize=9); ax5.grid(alpha=0.3)

# --- 6. Score Composto ---
ax6 = fig.add_subplot(gs[1, 2])
scores = resumo['score_total'].tolist()
cores_bar = [cor_destaque if k == k_consenso else '#bdc3c7' for k in ks]
ax6.bar(ks, scores, color=cores_bar, edgecolor='white', linewidth=0.8)
ax6.set_title('Score Composto (médio dos rankings)', fontweight='bold')
ax6.set_xlabel('k'); ax6.set_ylabel('Score [0–1]')
ax6.set_xticks(ks); ax6.grid(axis='y', alpha=0.3)
ax6.annotate(f'k={k_consenso}',
             xy=(k_consenso, resumo.loc[resumo['k']==k_consenso,'score_total'].values[0]),
             xytext=(k_consenso + 0.3,
                     resumo.loc[resumo['k']==k_consenso,'score_total'].values[0] + 0.02),
             fontsize=9, color=cor_destaque,
             arrowprops=dict(arrowstyle='->', color=cor_destaque))

# --- 7. Silhouette Plot detalhado para k_consenso ---
ax7 = fig.add_subplot(gs[2, :])
km_final = KMeans(n_clusters=k_consenso, init='k-means++', n_init=15,
                  max_iter=300, random_state=RANDOM_STATE)
labels_final = km_final.fit_predict(X)
sil_vals     = silhouette_samples(X[idx_sample], labels_final[idx_sample])

y_lower = 10
cores_clusters = plt.cm.tab10(np.linspace(0, 1, k_consenso))

for i in range(k_consenso):
    sil_i = np.sort(sil_vals[labels_final[idx_sample] == i])
    size_i = len(sil_i)
    y_upper = y_lower + size_i
    ax7.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                      facecolor=cores_clusters[i], alpha=0.8,
                      label=f'Cluster {i}')
    ax7.text(-0.05, y_lower + size_i / 2, f'C{i}', fontsize=8,
             color=cores_clusters[i], fontweight='bold')
    y_lower = y_upper + 10

media_sil = np.mean(sil_vals)
ax7.axvline(media_sil, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'Média = {media_sil:.4f}')
ax7.set_title(f'Silhouette Plot — k={k_consenso} (amostra {SAMPLE_SIL:,})',
              fontweight='bold')
ax7.set_xlabel('Coeficiente de Silhouette')
ax7.set_ylabel('Cluster')
ax7.set_yticks([])
ax7.legend(loc='lower right', fontsize=8)
ax7.grid(axis='x', alpha=0.3)

plt.suptitle("Determinação do Número Ótimo de Clusters — K Selection",
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('k_selection.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE FINAL")
print("=" * 70)
print(f"""
  Elbow (Inércia)         : k = {k_elbow}
  Silhouette Score        : k = {k_silhouette}
  Davies-Bouldin Index    : k = {k_db}
  Calinski-Harabasz Index : k = {k_ch}
  Score Composto          : k = {int(k_score)}

  {consenso_label}
  → k recomendado para modelagem: {k_consenso}

  Coeficiente de variação (estabilidade) para k={k_consenso}:
  {estabilidade[k_consenso]['cv']:.4f}% (< 0.5% = estável)

  Próximo passo: executar K-Means com k={k_consenso} e caracterizar os segmentos.
""")

## Análise correçação variáveis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# =============================================================================
# ANÁLISE DE CORRELAÇÃO E REDUNDÂNCIA
# Dataset   : df_normalizado
# Variáveis : var_clustering_final (excluindo id_cnpj)
# Threshold : |r| > 0.80
# =============================================================================

THRESHOLD_CORR = 0.80

# Separar variáveis contínuas das binárias/flags para a matriz de correlação
# Pearson é adequado para contínuas; binárias serão examinadas separadamente

var_continuas_corr = [
    v for v in var_clustering_final
    if not v.endswith('_bin')
    and not v.startswith('is_missing_')
]

var_binarias_corr = [
    v for v in var_clustering_final
    if v.endswith('_bin') or v.startswith('is_missing_')
]

print("=" * 70)
print("ANÁLISE DE CORRELAÇÃO — Selecção de Variáveis")
print("=" * 70)
print(f"\n  Variáveis contínuas analisadas : {len(var_continuas_corr)}")
print(f"  Variáveis binárias/flags       : {len(var_binarias_corr)}")
print(f"  Threshold                      : |r| > {THRESHOLD_CORR}")


# =============================================================================
# BLOCO 1 — MATRIZ DE CORRELAÇÃO DE PEARSON (variáveis contínuas)
# =============================================================================

corr_matrix = df_normalizado[var_continuas_corr].corr(method='pearson')


# =============================================================================
# BLOCO 2 — IDENTIFICAÇÃO DE PARES ACIMA DO THRESHOLD
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Pares com |r| > 0.80")
print("─" * 70)

pares_altos = []

for v1, v2 in combinations(var_continuas_corr, 2):
    r = corr_matrix.loc[v1, v2]
    if abs(r) >= THRESHOLD_CORR:
        pares_altos.append({
            'var_1':      v1,
            'var_2':      v2,
            'correlacao': round(r, 4),
            'abs_r':      round(abs(r), 4),
        })

df_pares = (pd.DataFrame(pares_altos)
              .sort_values('abs_r', ascending=False)
              .reset_index(drop=True))

if len(df_pares) == 0:
    print("\n  ✔  Nenhum par acima do threshold.")
else:
    print(f"\n  {len(df_pares)} pares identificados:\n")
    print(f"  {'Variável 1':<45} {'Variável 2':<45} {'r':>8}")
    print("  " + "─" * 100)
    for _, row in df_pares.iterrows():
        sinal = "⚠ " if row['abs_r'] >= 0.95 else "→ "
        print(f"  {sinal}{row['var_1']:<43} {row['var_2']:<45} {row['correlacao']:>8.4f}")


# =============================================================================
# BLOCO 3 — ANÁLISE DE CORRELAÇÃO PONTO-BISSERIAL
# Binárias vs contínuas — identifica binárias redundantes com contínuas
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Correlação Ponto-Bisserial (binárias vs contínuas)")
print("─" * 70)

from scipy.stats import pointbiserialr

pares_bin = []

for vbin in var_binarias_corr:
    for vcont in var_continuas_corr:
        try:
            r_pb, p_val = pointbiserialr(
                df_normalizado[vbin],
                df_normalizado[vcont]
            )
            if abs(r_pb) >= 0.60:   # threshold mais baixo para binárias
                pares_bin.append({
                    'binaria':    vbin,
                    'continua':   vcont,
                    'r_pb':       round(r_pb, 4),
                    'abs_r_pb':   round(abs(r_pb), 4),
                    'p_valor':    round(p_val, 6),
                })
        except Exception:
            continue

df_pares_bin = (pd.DataFrame(pares_bin)
                  .sort_values('abs_r_pb', ascending=False)
                  .reset_index(drop=True))

if len(df_pares_bin) == 0:
    print("\n  ✔  Nenhuma binária com correlação ponto-bisserial > 0.60.")
else:
    print(f"\n  {len(df_pares_bin)} pares relevantes:\n")
    print(f"  {'Binária':<45} {'Contínua':<45} {'r_pb':>8}")
    print("  " + "─" * 100)
    for _, row in df_pares_bin.iterrows():
        print(f"  → {row['binaria']:<43} {row['continua']:<45} {row['r_pb']:>8.4f}")


# =============================================================================
# BLOCO 4 — GRAFO DE REDUNDÂNCIA
# Identifica clusters de variáveis altamente correlacionadas
# e sugere qual manter em cada grupo
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Grupos de Redundância")
print("─" * 70)

# Construir grupos por conectividade (union-find simples)
grupos = {}
pertence = {}

for _, row in df_pares.iterrows():
    v1, v2 = row['var_1'], row['var_2']

    g1 = pertence.get(v1)
    g2 = pertence.get(v2)

    if g1 is None and g2 is None:
        novo_id = len(grupos)
        grupos[novo_id] = {v1, v2}
        pertence[v1] = novo_id
        pertence[v2] = novo_id
    elif g1 is not None and g2 is None:
        grupos[g1].add(v2)
        pertence[v2] = g1
    elif g1 is None and g2 is not None:
        grupos[g2].add(v1)
        pertence[v1] = g2
    elif g1 != g2:
        # Fundir os dois grupos
        grupos[g1] = grupos[g1].union(grupos[g2])
        for v in grupos[g2]:
            pertence[v] = g1
        del grupos[g2]

if grupos:
    print(f"\n  {len(grupos)} grupo(s) de variáveis redundantes:\n")
    for gid, membros in grupos.items():
        print(f"  Grupo {gid + 1}: {sorted(membros)}")
else:
    print("\n  ✔  Nenhum grupo de redundância identificado.")


# =============================================================================
# BLOCO 5 — VISUALIZAÇÕES
# =============================================================================

# --- 5.1 Heatmap completo ---
fig, ax = plt.subplots(figsize=(20, 18))

mascara = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mascara,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.4,
    linecolor='#ecf0f1',
    annot_kws={'size': 7},
    ax=ax,
    cbar_kws={'shrink': 0.6}
)

ax.set_title('Matriz de Correlação de Pearson — Variáveis Contínuas\n'
             f'(threshold de remoção: |r| > {THRESHOLD_CORR})',
             fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig('correlacao_heatmap_completo.png', dpi=150, bbox_inches='tight')
plt.show()


# --- 5.2 Heatmap filtrado — apenas pares acima do threshold ---
if len(df_pares) > 0:

    vars_envolvidas = sorted(set(
        df_pares['var_1'].tolist() + df_pares['var_2'].tolist()
    ))

    corr_filtrada = corr_matrix.loc[vars_envolvidas, vars_envolvidas]

    fig2, ax2 = plt.subplots(figsize=(14, 12))

    sns.heatmap(
        corr_filtrada,
        annot=True,
        fmt='.2f',
        cmap='RdBu_r',
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        linecolor='#ecf0f1',
        annot_kws={'size': 9},
        ax=ax2,
        cbar_kws={'shrink': 0.6}
    )

    # Destacar células acima do threshold
    for _, row in df_pares.iterrows():
        i = vars_envolvidas.index(row['var_2'])
        j = vars_envolvidas.index(row['var_1'])
        ax2.add_patch(plt.Rectangle((j, i), 1, 1,
                                    fill=False,
                                    edgecolor='#e74c3c',
                                    lw=2.5))

    ax2.set_title(f'Variáveis com |r| > {THRESHOLD_CORR} — Detalhe\n'
                  '(bordas vermelhas = pares acima do threshold)',
                  fontsize=12, fontweight='bold', pad=12)
    ax2.tick_params(axis='x', rotation=45, labelsize=9)
    ax2.tick_params(axis='y', rotation=0,  labelsize=9)

    plt.tight_layout()
    plt.savefig('correlacao_heatmap_filtrado.png', dpi=150, bbox_inches='tight')
    plt.show()


# --- 5.3 Barplot de correlações máximas por variável ---
max_corr_por_var = {}
for var in var_continuas_corr:
    outras = [abs(corr_matrix.loc[var, v2])
              for v2 in var_continuas_corr if v2 != var]
    max_corr_por_var[var] = max(outras)

df_max_corr = (pd.DataFrame.from_dict(max_corr_por_var, orient='index',
                                       columns=['max_abs_r'])
                 .sort_values('max_abs_r', ascending=True))

fig3, ax3 = plt.subplots(figsize=(10, 10))

cores = ['#e74c3c' if v >= THRESHOLD_CORR else '#2ecc71'
         for v in df_max_corr['max_abs_r']]

ax3.barh(df_max_corr.index, df_max_corr['max_abs_r'],
         color=cores, edgecolor='white', linewidth=0.5)
ax3.axvline(THRESHOLD_CORR, color='#e74c3c', linestyle='--',
            linewidth=1.8, label=f'Threshold {THRESHOLD_CORR}')
ax3.set_title('Correlação Máxima de Cada Variável com as Demais',
              fontweight='bold', fontsize=11)
ax3.set_xlabel('|r| máximo')
ax3.legend()
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('correlacao_max_por_variavel.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# BLOCO 6 — SÍNTESE PARA DECISÃO
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE — Variáveis candidatas à remoção por redundância")
print("=" * 70)

print(f"""
  Total de variáveis analisadas     : {len(var_continuas_corr)} contínuas
                                      + {len(var_binarias_corr)} binárias/flags
  Pares contínuas |r| > {THRESHOLD_CORR}         : {len(df_pares)}
  Pares binárias  |r_pb| > 0.60     : {len(df_pares_bin)}
  Grupos de redundância identificados: {len(grupos)}

  Próximo passo:
  → Analisar os grupos identificados
  → Decidir qual variável manter em cada par (critério: interpretabilidade RFB)
  → Remover redundâncias e refazer K Selection com conjunto enxuto
""")

# Exportar tabela de pares para referência
df_pares.to_csv('pares_correlacionados.csv', index=False)
print("  Tabela exportada: pares_correlacionados.csv")

### Seleção das features

In [ ]:
# =============================================================================
# REMOÇÃO DE VARIÁVEIS REDUNDANTES — Decisões pós-análise de correlação
# Dataset de entrada : df_normalizado (31 variáveis em var_clustering_final)
# Dataset de saída   : df_clustering  (23 variáveis em var_clustering_final)
# =============================================================================

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Variáveis a remover com justificativa documentada
# -----------------------------------------------------------------------------

remocoes = {

    # GRUPO 1 — Suspensão e Exequibilidade
    # da_enforceable_debt_ratio e da_admin_suspended_ratio: r = -1.00 (complementares por construção)
    # da_admin_suspended_debt_value_log: |r| = 0.877 com ambas as ratios
    # Mantém: da_enforceable_debt_ratio (indicador primário de accionabilidade da cobrança)
    'da_admin_suspended_ratio':          'r = -1.00 com da_enforceable_debt_ratio (complementares por construção)',
    'da_admin_suspended_debt_value_log': '|r| = 0.877 com ratios de suspensão; informação captada por da_enforceable_debt_ratio',

    # GRUPO 2 — Envelhecimento da Dívida
    # r = 0.947 entre as duas; versão ponderada por valor é mais rica informativamente
    # Mantém: da_age_weighted_debt_value_months_log
    'da_average_debt_age_months_log':    'r = 0.947 com versão ponderada por valor; versão ponderada preserva mais informação',

    # GRUPO 3 — Componentes do Valor da Dívida
    # Relação matemática directa: total = principal + juros/multas
    # da_penalty_to_principal_ratio_log já captura a composição relativa
    # Mantém: da_total_debt_value_log
    'da_total_principal_value_log':           'r = 0.847 com da_total_debt_value_log; componente captada por da_penalty_to_principal_ratio_log',
    'da_total_penalties_interest_value_log':  'r = 0.935 com da_total_debt_value_log; componente captada por da_penalty_to_principal_ratio_log',

    # PARES PONTO-BISSERIAIS — Binárias redundantes com contínuas
    # Contínuas preservam granularidade (valor em reais vs presença/ausência)
    'da_prescription_risk_bin':        'r_pb = 0.96 com da_debt_over_48_months_value_log; contínua é mais informativa',
    'da_new_debt_incidence_bin':       'r_pb = 0.94 com da_debt_0_6_months_value_log; contínua é mais informativa',
    'pc_recent_payment_indicator_bin': 'r_pb = 0.82 com pc_total_payment_2025_value_log; valor marginal insuficiente',
}

print("=" * 70)
print("REMOÇÃO DE VARIÁVEIS REDUNDANTES")
print("=" * 70)

print(f"\n  Variáveis antes : {len(var_clustering_final)}")
print(f"  Variáveis a remover: {len(remocoes)}\n")

# Verificar existência antes de remover
nao_encontradas = [v for v in remocoes if v not in df_normalizado.columns]
if nao_encontradas:
    print(f"  ⚠  Não encontradas (ignoradas): {nao_encontradas}")

variaveis_remover_existentes = [v for v in remocoes if v in df_normalizado.columns]

for var in variaveis_remover_existentes:
    print(f"  ✗  {var}")
    print(f"       {remocoes[var]}\n")


# -----------------------------------------------------------------------------
# Aplicar remoções e reconstruir lista de variáveis
# -----------------------------------------------------------------------------

df_clustering = df_normalizado.drop(
    columns=variaveis_remover_existentes,
    errors='ignore'
)

var_clustering_final = [
    v for v in var_clustering_final
    if v not in variaveis_remover_existentes
]

# Separar sublistas para referência nas próximas etapas
var_continuas_final = [
    v for v in var_clustering_final
    if not v.endswith('_bin')
    and not v.startswith('is_missing_')
]

var_binarias_final = [
    v for v in var_clustering_final
    if v.endswith('_bin') or v.startswith('is_missing_')
]

print(f"  Variáveis depois : {len(var_clustering_final)}")
print(f"    → Contínuas    : {len(var_continuas_final)}")
print(f"    → Binárias/flags: {len(var_binarias_final)}")


# -----------------------------------------------------------------------------
# Verificação de correlação residual — confirmar que nenhum par > 0.80 persiste
# -----------------------------------------------------------------------------

print("\n" + "─" * 70)
print("VERIFICAÇÃO — Correlação residual após remoções")
print("─" * 70)

from itertools import combinations

corr_residual = df_clustering[var_continuas_final].corr(method='pearson')

pares_residuais = []
for v1, v2 in combinations(var_continuas_final, 2):
    r = corr_residual.loc[v1, v2]
    if abs(r) >= 0.80:
        pares_residuais.append((v1, v2, round(r, 4)))

if pares_residuais:
    print(f"\n  ⚠  {len(pares_residuais)} par(es) ainda acima do threshold:")
    for v1, v2, r in pares_residuais:
        print(f"     {v1}  ×  {v2}  →  r = {r}")
else:
    print(f"\n  ✔  Nenhum par residual acima de |r| = 0.80.")


# -----------------------------------------------------------------------------
# Relatório final do conjunto de variáveis para clustering
# -----------------------------------------------------------------------------

print("\n" + "=" * 70)
print("CONJUNTO FINAL — var_clustering_final")
print("=" * 70)

print(f"\n  {'#':<4} {'Variável':<52} {'Tipo'}")
print("  " + "─" * 65)

for i, var in enumerate(var_clustering_final, 1):
    if var.startswith('is_missing_'):
        tipo = 'flag missing'
    elif var.endswith('_bin'):
        tipo = 'binária'
    else:
        tipo = 'contínua'
    print(f"  {i:<4} {var:<52} {tipo}")

print(f"\n  Total             : {len(var_clustering_final)} variáveis")
print(f"  Contínuas (scaler): {len(var_continuas_final)}")
print(f"  Binárias/flags    : {len(var_binarias_final)}")
print(f"  Dataset           : {df_clustering.shape[0]:,} registros × {df_clustering.shape[1]} colunas")
print(f"  (inclui id_cnpj como identificador — excluído da matriz X)")

## DETERMINAÇÃO DO NÚMERO DE CLUSTERS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from kneed import KneeLocator
from collections import Counter

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K_MIN        = 2
K_MAX        = 10
k_range      = range(K_MIN, K_MAX + 1)
N_SEEDS      = 10
SAMPLE_SIL   = 30_000

# =============================================================================
# MATRIZ X — 23 variáveis, sem identificador
# =============================================================================

X = df_clustering[var_clustering_final].values

print("=" * 70)
print("K SELECTION — Conjunto enxuto pós-análise de correlação")
print("=" * 70)
print(f"\n  Dataset           : {X.shape[0]:,} observações | {X.shape[1]} variáveis")
print(f"  Contínuas         : {len(var_continuas_final)}")
print(f"  Binárias/flags    : {len(var_binarias_final)}")
print(f"  Range testado     : k = {K_MIN} a {K_MAX}")
print(f"  Seeds estabilidade: {N_SEEDS}")
print(f"  Amostra silhouette: {SAMPLE_SIL:,}")


# =============================================================================
# BLOCO 1 — ELBOW (Inércia / WCSS)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 1 — Método do Cotovelo (Elbow / WCSS)")
print("─" * 70)

inertias = []

for k in k_range:
    t0 = time.time()
    km = KMeans(n_clusters=k, init='k-means++', n_init=15,
                max_iter=300, random_state=RANDOM_STATE)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f"  k={k:2d} | Inércia: {km.inertia_:>18,.2f} | {time.time()-t0:.1f}s")

kneedle = KneeLocator(list(k_range), inertias,
                      curve='convex', direction='decreasing')
k_elbow = kneedle.elbow
print(f"\n  ✔  Cotovelo detectado: k = {k_elbow}")


# =============================================================================
# BLOCO 2 — MÉTRICAS DE VALIDAÇÃO INTERNA
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Métricas de Validação Interna")
print("─" * 70)

sil_scores = []
db_scores  = []
ch_scores  = []

rng       = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=min(SAMPLE_SIL, X.shape[0]), replace=False)
X_sample  = X[idx_sample]

for k in k_range:
    t0     = time.time()
    km     = KMeans(n_clusters=k, init='k-means++', n_init=15,
                    max_iter=300, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)

    labels_sample = labels[idx_sample]
    sil = silhouette_score(X_sample, labels_sample, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X, labels)
    ch  = calinski_harabasz_score(X, labels)

    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)

    print(f"  k={k:2d} | Silhouette: {sil:.4f} | "
          f"Davies-Bouldin: {db:.4f} | "
          f"Calinski-Harabasz: {ch:>12,.1f} | {time.time()-t0:.1f}s")

k_silhouette = list(k_range)[np.argmax(sil_scores)]
k_db         = list(k_range)[np.argmin(db_scores)]
k_ch         = list(k_range)[np.argmax(ch_scores)]

print(f"\n  Melhor Silhouette       : k = {k_silhouette}  ({max(sil_scores):.4f})")
print(f"  Melhor Davies-Bouldin   : k = {k_db}  ({min(db_scores):.4f})")
print(f"  Melhor Calinski-Harabasz: k = {k_ch}  ({max(ch_scores):,.1f})")


# =============================================================================
# BLOCO 3 — ESTABILIDADE (variância entre seeds)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Análise de Estabilidade (variância entre seeds)")
print("─" * 70)

seeds        = [42, 7, 13, 21, 99, 123, 256, 512, 1024, 2048][:N_SEEDS]
estabilidade = {}

for k in k_range:
    inertias_k = []
    for seed in seeds:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    max_iter=300, random_state=seed)
        km.fit(X)
        inertias_k.append(km.inertia_)

    media  = np.mean(inertias_k)
    desvio = np.std(inertias_k)
    cv     = (desvio / media) * 100

    estabilidade[k] = {'media': media, 'desvio': desvio, 'cv': cv}

    label = "✔  estável" if cv < 0.5 else ("~  moderado" if cv < 2 else "⚠  instável")
    print(f"  k={k:2d} | Média: {media:>18,.2f} | "
          f"Std: {desvio:>12,.2f} | CV: {cv:.4f}%  {label}")


# =============================================================================
# BLOCO 4 — SCORE COMPOSTO E CONSENSO
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Score Composto e Consenso")
print("─" * 70)

resumo = pd.DataFrame({
    'k':                 list(k_range),
    'inertia':           inertias,
    'silhouette':        sil_scores,
    'davies_bouldin':    db_scores,
    'calinski_harabasz': ch_scores,
    'estab_cv_pct':      [estabilidade[k]['cv'] for k in k_range],
})

resumo['rank_sil']   = (resumo['silhouette']        - resumo['silhouette'].min())        / (resumo['silhouette'].max()        - resumo['silhouette'].min())
resumo['rank_db']    = 1 - (resumo['davies_bouldin'] - resumo['davies_bouldin'].min())   / (resumo['davies_bouldin'].max()    - resumo['davies_bouldin'].min())
resumo['rank_ch']    = (resumo['calinski_harabasz'] - resumo['calinski_harabasz'].min()) / (resumo['calinski_harabasz'].max() - resumo['calinski_harabasz'].min())
resumo['rank_estab'] = 1 - (resumo['estab_cv_pct']  - resumo['estab_cv_pct'].min())      / (resumo['estab_cv_pct'].max()     - resumo['estab_cv_pct'].min() + 1e-10)
resumo['score_total']= resumo[['rank_sil','rank_db','rank_ch','rank_estab']].mean(axis=1)

votos      = Counter([k_elbow, k_silhouette, k_db, k_ch])
k_consenso = votos.most_common(1)[0][0]
n_votos    = votos.most_common(1)[0][1]
k_score    = int(resumo.loc[resumo['score_total'].idxmax(), 'k'])

if n_votos >= 3:
    consenso_label = f"✔  CONSENSO FORTE ({n_votos}/4 métodos)"
elif n_votos == 2:
    consenso_label = f"~  CONSENSO PARCIAL ({n_votos}/4 métodos)"
else:
    consenso_label = "⚠  SEM CONSENSO — avaliar estabilidade e interpretabilidade"

print(f"\n  Elbow                   : k = {k_elbow}")
print(f"  Silhouette              : k = {k_silhouette}")
print(f"  Davies-Bouldin          : k = {k_db}")
print(f"  Calinski-Harabasz       : k = {k_ch}")
print(f"  Score composto          : k = {k_score}")
print(f"\n  {consenso_label}")


# =============================================================================
# BLOCO 5 — TABELA RESUMO
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 5 — Tabela Resumo Consolidada")
print("─" * 70)

print(f"\n  {'k':>3} {'Inércia':>20} {'Silhouette':>12} {'DB':>10} "
      f"{'CH':>14} {'CV%':>8} {'Score':>8}")
print("  " + "─" * 78)

for _, row in resumo.iterrows():
    marcador = "  ◄" if int(row['k']) == k_consenso else ""
    print(f"  {int(row['k']):>3} {row['inertia']:>20,.2f} {row['silhouette']:>12.4f} "
          f"{row['davies_bouldin']:>10.4f} {row['calinski_harabasz']:>14,.1f} "
          f"{row['estab_cv_pct']:>8.4f} {row['score_total']:>8.4f}{marcador}")


# =============================================================================
# BLOCO 6 — VISUALIZAÇÕES
# =============================================================================

ks            = list(k_range)
cor_principal = '#2c3e50'
cor_destaque  = '#e74c3c'

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)


# --- 1. Elbow ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ks, inertias, 'o-', color=cor_principal, linewidth=2, markersize=7)
if k_elbow:
    ax1.axvline(k_elbow, color=cor_destaque, linestyle='--',
                lw=1.8, label=f'k={k_elbow}')
    ax1.legend(fontsize=9)
ax1.set_title('Elbow (Inércia / WCSS)', fontweight='bold')
ax1.set_xlabel('k'); ax1.set_ylabel('WCSS')
ax1.set_xticks(ks); ax1.grid(alpha=0.3)


# --- 2. Silhouette ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ks, sil_scores, 'o-', color='#27ae60', linewidth=2, markersize=7)
ax2.axvline(k_silhouette, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_silhouette}')
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.set_xlabel('k'); ax2.set_ylabel('Score')
ax2.set_xticks(ks); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)


# --- 3. Davies-Bouldin ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ks, db_scores, 'o-', color='#8e44ad', linewidth=2, markersize=7)
ax3.axvline(k_db, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_db}')
ax3.set_title('Davies-Bouldin (↓ melhor)', fontweight='bold')
ax3.set_xlabel('k'); ax3.set_ylabel('Índice')
ax3.set_xticks(ks); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)


# --- 4. Calinski-Harabasz ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(ks, ch_scores, 'o-', color='#e67e22', linewidth=2, markersize=7)
ax4.axvline(k_ch, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_ch}')
ax4.set_title('Calinski-Harabasz (↑ melhor)', fontweight='bold')
ax4.set_xlabel('k'); ax4.set_ylabel('Índice')
ax4.set_xticks(ks); ax4.legend(fontsize=9); ax4.grid(alpha=0.3)


# --- 5. Estabilidade ---
cvs = [estabilidade[k]['cv'] for k in k_range]
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(ks, cvs, 'o-', color='#16a085', linewidth=2, markersize=7)
ax5.axhline(0.5, color=cor_destaque, linestyle=':', lw=1.5,
            label='Threshold 0.5%')
ax5.set_title('Estabilidade — CV% Inércia (↓ melhor)', fontweight='bold')
ax5.set_xlabel('k'); ax5.set_ylabel('CV (%)')
ax5.set_xticks(ks); ax5.legend(fontsize=9); ax5.grid(alpha=0.3)


# --- 6. Score Composto ---
ax6 = fig.add_subplot(gs[1, 2])
scores    = resumo['score_total'].tolist()
cores_bar = [cor_destaque if k == k_consenso else '#bdc3c7' for k in ks]
ax6.bar(ks, scores, color=cores_bar, edgecolor='white', linewidth=0.8)
ax6.set_title('Score Composto — média dos rankings', fontweight='bold')
ax6.set_xlabel('k'); ax6.set_ylabel('Score [0–1]')
ax6.set_xticks(ks); ax6.grid(axis='y', alpha=0.3)

val_k = resumo.loc[resumo['k'] == k_consenso, 'score_total'].values[0]
ax6.annotate(f'k={k_consenso}',
             xy=(k_consenso, val_k),
             xytext=(k_consenso + 0.4, val_k + 0.02),
             fontsize=9, color=cor_destaque,
             arrowprops=dict(arrowstyle='->', color=cor_destaque))


# --- 7. Silhouette Plot detalhado para k_consenso ---
ax7 = fig.add_subplot(gs[2, :])

km_final     = KMeans(n_clusters=k_consenso, init='k-means++', n_init=15,
                      max_iter=300, random_state=RANDOM_STATE)
labels_final = km_final.fit_predict(X)
sil_vals     = silhouette_samples(X[idx_sample], labels_final[idx_sample])

y_lower       = 10
cores_clusters = plt.cm.tab10(np.linspace(0, 1, k_consenso))

for i in range(k_consenso):
    sil_i  = np.sort(sil_vals[labels_final[idx_sample] == i])
    size_i = len(sil_i)
    y_upper = y_lower + size_i

    ax7.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                      facecolor=cores_clusters[i], alpha=0.8)
    ax7.text(-0.06, y_lower + size_i / 2, f'C{i}',
             fontsize=9, color=cores_clusters[i], fontweight='bold')
    y_lower = y_upper + 10

media_sil = np.mean(sil_vals)
ax7.axvline(media_sil, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'Média = {media_sil:.4f}')
ax7.set_title(f'Silhouette Plot — k={k_consenso}  (amostra {SAMPLE_SIL:,})',
              fontweight='bold')
ax7.set_xlabel('Coeficiente de Silhouette')
ax7.set_ylabel('Cluster'); ax7.set_yticks([])
ax7.legend(loc='lower right', fontsize=9)
ax7.grid(axis='x', alpha=0.3)

plt.suptitle(
    f"K Selection — {X.shape[1]} variáveis (pós-correlação)\n"
    f"377.419 devedores | RFB 9ª Região Fiscal",
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('k_selection_pos_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE FINAL — K Selection pós-correlação")
print("=" * 70)
print(f"""
  Variáveis utilizadas    : {X.shape[1]} (reduzidas de 31 para 23)
  Pares redundantes removidos: 8

  Elbow (Inércia)         : k = {k_elbow}
  Silhouette Score        : k = {k_silhouette}
  Davies-Bouldin Index    : k = {k_db}
  Calinski-Harabasz Index : k = {k_ch}
  Score Composto          : k = {k_score}

  {consenso_label}
  → k recomendado: {k_consenso}

  Estabilidade para k={k_consenso}: CV = {estabilidade[k_consenso]['cv']:.4f}%

  Candidatos operacionais a avaliar: k=4 e k=5
  → Decisão final após caracterização dos segmentos e validação com RFB.
""")

## Modelling

### TREINAR K-MEANS com k=3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K3           = 3

# =============================================================================
# BLOCO 1 — TREINAR K-MEANS com k=3
# Espaço de features: var_clustering_final (23 variáveis normalizadas)
# =============================================================================

X = df_clustering[var_clustering_final].values

print("=" * 70)
print(f"K-MEANS — k={K3}")
print("=" * 70)

km3 = KMeans(
    n_clusters    = K3,
    init          = 'k-means++',
    n_init        = 20,
    max_iter      = 500,
    random_state  = RANDOM_STATE
)

labels_k3 = km3.fit_predict(X)

# Distribuição bruta
dist_k3 = pd.Series(labels_k3).value_counts().sort_index()
print(f"\n  Inércia  : {km3.inertia_:,.2f}")
print(f"  Silhouette (amostra 30k): ", end="")

rng        = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=30_000, replace=False)
sil_k3     = silhouette_score(X[idx_sample], labels_k3[idx_sample])
print(f"{sil_k3:.4f}")

print(f"\n  Distribuição dos clusters:")
for c, n in dist_k3.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k3)*100:.1f}%)")


# =============================================================================
# BLOCO 2 — JUNTAR LABELS AO DATASET ORIGINAL
# Caracterização usa df (escala original, em reais) — não df_normalizado
# A ligação é feita pelo índice — garantir alinhamento
# =============================================================================

df_original = df.copy()
df_original = df_original.loc[df_clustering.index].copy()
df_original['cluster_k3'] = labels_k3

assert len(df_original) == len(labels_k3)


# =============================================================================
# BLOCO 3 — VARIÁVEIS OPERACIONAIS PARA PERFILAMENTO
# Usando nomes originais (pré-transformação), em reais e meses
# =============================================================================

# Variáveis contínuas operacionais (escala original)
vars_perfil_continuas = {
    'da_total_debt_value':               'Dívida Total (R$)',
    # 'da_total_principal_value'         → removida: redundante com dívida total
    # 'da_total_penalties_interest_value'→ removida: redundante com dívida total
    'da_enforceable_debt_value':         'Dívida Executável (R$)',
    'da_debt_pgfn_value':                'Dívida PGFN (R$)',
    'da_debt_0_6_months_value':          'Dívida 0–6 meses (R$)',
    'da_debt_over_48_months_value':      'Dívida >48 meses (R$)',
    'da_penalty_to_principal_ratio':     'Ratio Multas/Principal',
    'da_enforceable_debt_ratio':         'Proporção Executável',
    'da_age_weighted_debt_value_months': 'Idade Pond. Dívida (meses)',
    'ap_declared_gross_revedue_value':   'Receita Declarada (R$)',
    'ap_payment_capacity_value':         'Capacidade Pagamento (R$)',
    'ap_debt_to_revenue_ratio':          'Dívida/Receita',
    'pc_total_payment_2025_value':       'Pagamentos 2025 (R$)',
    'pc_instalment_plan_count':          'Nº Parcelamentos',
    'pc_defaulted_instalment_plans_count': 'Parcelamentos Rompidos',
    'pc_payment_regularity_ratio':       'Regularidade Parcelamentos',
    'pc_pending_filing_omissions_ratio': 'Omissões de Declaração',
    'da_number_of_debts_count':          'Nº Débitos',
    'da_number_distinct_tax_types_count':'Nº Tipos Tributo',
    'tp_enterprise_age_years':           'Idade Empresa (anos)',
    'ci_sectoral_delinquency_deviation_score': 'Desvio Setorial',
}

# Variáveis descritivas binárias (não entraram no clustering — usadas como validação)
vars_perfil_binarias = {
    'da_prescription_risk_bin':     'Risco Prescrição (%)',
    'da_new_debt_incidence_bin':    'Incidência Nova Dívida (%)',
    'pc_recent_payment_indicator_bin': 'Pagamento Recente (%)',
}

vars_flags = {
    'is_missing_ap_declared_gross_revenue_value':  'Sem Receita Declarada (%)',
    'is_missing_ap_debt_to_revenue_ratio':         'Sem Ratio Dívida/Receita (%)',
    'is_missing_pc_pending_filing_omissions_ratio':'Sem Omissões Declaração (%)',  # ← faltava esta
}

# Filtrar apenas as que existem no dataset original
vars_cont_ok   = {k: v for k, v in vars_perfil_continuas.items() if k in df_original.columns}
vars_bin_ok    = {k: v for k, v in vars_perfil_binarias.items()   if k in df_original.columns}
vars_flags_ok  = {k: v for k, v in vars_flags.items()             if k in df_original.columns}


# =============================================================================
# BLOCO 4 — TABELA DE PERFILAMENTO k=3
# Mediana + IQR para contínuas; % para binárias/flags
# =============================================================================

def tabela_perfil(df_labeled, col_cluster, vars_cont, vars_bin, vars_flags, k):
    """
    Gera tabela de perfilamento operacional por cluster.
    Contínuas: mediana e IQR.
    Binárias/flags: percentual de 1s.
    """
    clusters  = sorted(df_labeled[col_cluster].unique())
    n_total   = len(df_labeled)

    linhas = []

    # --- Tamanho do cluster ---
    for c in clusters:
        mask = df_labeled[col_cluster] == c
        n    = mask.sum()
        pct  = n / n_total * 100
        linhas.append({
            'secao':    '── Dimensão',
            'variavel': 'Tamanho do cluster',
            'label':    'Tamanho do cluster',
            **{f'C{c}': f"{n:,} ({pct:.1f}%)" for c in clusters},
        })
        break  # uma única linha de tamanho

    row_tamanho = {'secao': '── Dimensão', 'label': 'Tamanho do cluster'}
    for c in clusters:
        mask = df_labeled[col_cluster] == c
        n    = mask.sum()
        pct  = n / n_total * 100
        row_tamanho[f'C{c}'] = f"{n:,}  ({pct:.1f}%)"
    row_tamanho['Global'] = f"{n_total:,}  (100%)"

    resultado = [row_tamanho]

    # --- Variáveis contínuas: mediana e IQR ---
    resultado.append({
        'secao': '── Variáveis Operacionais (Mediana  [P25 – P75])',
        'label': '',
        **{f'C{c}': '' for c in clusters},
        'Global': '',
    })

    for var, label in vars_cont.items():
        if var not in df_labeled.columns:
            continue
        row = {'secao': '', 'label': label}
        for c in clusters:
            serie = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            med   = serie.median()
            p25   = serie.quantile(0.25)
            p75   = serie.quantile(0.75)

            # Formatar de acordo com magnitude
            if med >= 1_000:
                row[f'C{c}'] = f"R$ {med:>12,.0f}  [{p25:,.0f} – {p75:,.0f}]"
            elif med >= 1:
                row[f'C{c}'] = f"{med:>8.2f}  [{p25:.2f} – {p75:.2f}]"
            else:
                row[f'C{c}'] = f"{med:>8.4f}  [{p25:.4f} – {p75:.4f}]"

        # Global
        serie_g = df_labeled[var].dropna()
        med_g   = serie_g.median()
        p25_g   = serie_g.quantile(0.25)
        p75_g   = serie_g.quantile(0.75)
        if med_g >= 1_000:
            row['Global'] = f"R$ {med_g:>12,.0f}  [{p25_g:,.0f} – {p75_g:,.0f}]"
        elif med_g >= 1:
            row['Global'] = f"{med_g:>8.2f}  [{p25_g:.2f} – {p75_g:.2f}]"
        else:
            row['Global'] = f"{med_g:>8.4f}  [{p25_g:.4f} – {p75_g:.4f}]"

        resultado.append(row)

    # --- Variáveis binárias e flags: percentual ---
    resultado.append({
        'secao': '── Indicadores Descritivos (%)',
        'label': '',
        **{f'C{c}': '' for c in clusters},
        'Global': '',
    })

    for var, label in {**vars_bin, **vars_flags}.items():
        if var not in df_labeled.columns:
            continue
        row = {'secao': '', 'label': label}
        for c in clusters:
            serie = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            pct   = serie.mean() * 100
            row[f'C{c}'] = f"{pct:.1f}%"
        row['Global'] = f"{df_labeled[var].dropna().mean() * 100:.1f}%"
        resultado.append(row)

    return pd.DataFrame(resultado)


print("\n" + "=" * 70)
print(f"TABELA DE PERFILAMENTO — k={K3}")
print("=" * 70)

df_perfil_k3 = tabela_perfil(
    df_labeled  = df_original,
    col_cluster = 'cluster_k3',
    vars_cont   = vars_cont_ok,
    vars_bin    = vars_bin_ok,
    vars_flags  = vars_flags_ok,
    k           = K3,
)

# Imprimir tabela formatada
colunas_exibir = ['label'] + [f'C{c}' for c in range(K3)] + ['Global']

for _, row in df_perfil_k3.iterrows():
    if row['label'] == '' and row.get('secao', ''):
        print(f"\n  {row['secao']}")
        print("  " + "─" * 115)
        continue
    if row['label'] == '':
        continue
    linha = f"  {row['label']:<45}"
    for col in [f'C{c}' for c in range(K3)] + ['Global']:
        val = row.get(col, '')
        linha += f"  {str(val):<30}"
    print(linha)


# =============================================================================
# BLOCO 5 — VISUALIZAÇÕES k=3
# =============================================================================

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

cores_k3      = ['#2980b9', '#e74c3c', '#27ae60']
clusters_k3   = sorted(df_original['cluster_k3'].unique())
labels_nomes  = [f'Cluster {c}' for c in clusters_k3]


# --- 1. Distribuição de tamanhos ---
ax1 = fig.add_subplot(gs[0, 0])
tamanhos = [dist_k3[c] for c in clusters_k3]
bars = ax1.bar(labels_nomes, tamanhos, color=cores_k3,
               edgecolor='white', linewidth=0.8)
for bar, n in zip(bars, tamanhos):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             f'{n:,}\n({n/len(labels_k3)*100:.1f}%)',
             ha='center', va='bottom', fontsize=9)
ax1.set_title('Tamanho dos Clusters', fontweight='bold')
ax1.set_ylabel('N devedores')
ax1.grid(axis='y', alpha=0.3)


# --- 2. Boxplot — Dívida Total ---
var_box = 'da_total_debt_value'
if var_box in df_original.columns:
    ax2 = fig.add_subplot(gs[0, 1])
    dados_box = [
        np.log1p(df_original.loc[df_original['cluster_k3'] == c, var_box].dropna())
        for c in clusters_k3
    ]
    bp = ax2.boxplot(dados_box, patch_artist=True, notch=False,
                     medianprops=dict(color='white', linewidth=2))
    for patch, cor in zip(bp['boxes'], cores_k3):
        patch.set_facecolor(cor)
        patch.set_alpha(0.8)
    ax2.set_xticklabels(labels_nomes)
    ax2.set_title('Dívida Total — log1p (R$)', fontweight='bold')
    ax2.set_ylabel('log1p(Dívida Total)')
    ax2.grid(axis='y', alpha=0.3)


# --- 3. Boxplot — Proporção Executável ---
var_enf = 'da_enforceable_debt_ratio'
if var_enf in df_original.columns:
    ax3 = fig.add_subplot(gs[0, 2])
    dados_enf = [
        df_original.loc[df_original['cluster_k3'] == c, var_enf].dropna()
        for c in clusters_k3
    ]
    bp3 = ax3.boxplot(dados_enf, patch_artist=True,
                      medianprops=dict(color='white', linewidth=2))
    for patch, cor in zip(bp3['boxes'], cores_k3):
        patch.set_facecolor(cor)
        patch.set_alpha(0.8)
    ax3.set_xticklabels(labels_nomes)
    ax3.set_title('Proporção Dívida Executável', fontweight='bold')
    ax3.set_ylabel('Ratio [0–1]')
    ax3.grid(axis='y', alpha=0.3)


# --- 4. Heatmap de desvio percentual da mediana global ---
vars_heatmap = [v for v in [
    'da_total_debt_value', 'da_enforceable_debt_ratio',
    'da_debt_over_48_months_value', 'da_penalty_to_principal_ratio',
    'pc_total_payment_2025_value', 'pc_instalment_plan_count',
    'pc_payment_regularity_ratio', 'ap_payment_capacity_value',
    'tp_enterprise_age_years', 'ci_sectoral_delinquency_deviation_score'
] if v in df_original.columns]

med_global = df_original[vars_heatmap].median()
desvios    = pd.DataFrame(index=clusters_k3, columns=vars_heatmap, dtype=float)

for c in clusters_k3:
    mask       = df_original['cluster_k3'] == c
    med_c      = df_original.loc[mask, vars_heatmap].median()
    desvios.loc[c] = ((med_c - med_global) / (med_global.replace(0, np.nan))) * 100

desvios.index = labels_nomes

ax4 = fig.add_subplot(gs[1, :])
import seaborn as sns
sns.heatmap(
    desvios.T.astype(float),
    annot=True, fmt='.0f',
    cmap='RdBu_r', center=0,
    linewidths=0.5, linecolor='#ecf0f1',
    annot_kws={'size': 9},
    ax=ax4,
    cbar_kws={'label': '% desvio da mediana global', 'shrink': 0.6}
)
ax4.set_title('Desvio Percentual da Mediana Global por Cluster — k=3',
              fontweight='bold', fontsize=11)
ax4.set_xlabel('')
ax4.tick_params(axis='x', labelsize=9)
ax4.tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle(f"Perfilamento Operacional — K-Means k={K3}  |  377.419 devedores",
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('perfil_k3.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# BLOCO 6 — SILHOUETTE PLOT k=3
# =============================================================================

fig2, ax_sil = plt.subplots(figsize=(10, 6))

sil_vals  = silhouette_samples(X[idx_sample], labels_k3[idx_sample])
y_lower   = 10
cores_sil = ['#2980b9', '#e74c3c', '#27ae60']

for i in clusters_k3:
    sil_i   = np.sort(sil_vals[labels_k3[idx_sample] == i])
    size_i  = len(sil_i)
    y_upper = y_lower + size_i
    ax_sil.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                         facecolor=cores_sil[i], alpha=0.8)
    ax_sil.text(-0.06, y_lower + size_i / 2, f'C{i}',
                fontsize=10, color=cores_sil[i], fontweight='bold')
    y_lower = y_upper + 10

ax_sil.axvline(sil_k3, color='black', linestyle='--', lw=1.8,
               label=f'Média = {sil_k3:.4f}')
ax_sil.set_title(f'Silhouette Plot — k={K3}  (amostra 30k)',
                 fontweight='bold', fontsize=11)
ax_sil.set_xlabel('Coeficiente de Silhouette')
ax_sil.set_ylabel('Cluster'); ax_sil.set_yticks([])
ax_sil.legend(fontsize=10); ax_sil.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('silhouette_k3.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE k=3
# =============================================================================

print("\n" + "=" * 70)
print(f"SÍNTESE — K-Means k={K3}")
print("=" * 70)
print(f"\n  Inércia            : {km3.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k3:.4f}")
print(f"\n  Distribuição:")
for c in clusters_k3:
    n   = dist_k3[c]
    pct = n / len(labels_k3) * 100
    print(f"    Cluster {c}: {n:>7,}  ({pct:.1f}%)")

print(f"""
  Labels guardadas em : df_original['cluster_k3']
  Modelo guardado em  : km3  (KMeans object)

  Próximo passo: repetir com k=4 para comparação de perfis.
""")

### Diagnóstico valores suspensos administrativos

In [ ]:
# =============================================================================
# DIAGNÓSTICO — da_enforceable_debt_ratio e da_admin_suspended_ratio
# =============================================================================

print("=" * 70)
print("DIAGNÓSTICO — Proporção Executável vs Suspensa")
print("=" * 70)

# --- 1. Distribuição do ratio no df_original ---
for var in ['da_enforceable_debt_ratio', 'da_admin_suspended_ratio']:
    if var not in df_original.columns:
        print(f"\n  ⚠  {var} NÃO encontrada em df_original")
        continue

    serie = df_original[var].dropna()
    print(f"\n  {var}")
    print(f"    N          : {len(serie):,}")
    print(f"    N com valor = 1.00 : {(serie == 1.0).sum():,}  ({(serie == 1.0).mean()*100:.1f}%)")
    print(f"    N com valor = 0.00 : {(serie == 0.0).sum():,}  ({(serie == 0.0).mean()*100:.1f}%)")
    print(f"    N com 0 < valor < 1: {((serie > 0) & (serie < 1)).sum():,}  ({((serie > 0) & (serie < 1)).mean()*100:.1f}%)")
    print(f"    Mediana    : {serie.median():.4f}")
    print(f"    P25 / P75  : {serie.quantile(0.25):.4f} / {serie.quantile(0.75):.4f}")
    print(f"    Min / Max  : {serie.min():.4f} / {serie.max():.4f}")


# --- 2. Cruzar valor absoluto suspenso com o ratio ---
print("\n" + "─" * 70)
print("Cruzamento — Valor Suspenso vs Ratio Executável")
print("─" * 70)

var_suspenso = 'da_admin_suspended_debt_value'
var_ratio    = 'da_enforceable_debt_ratio'
var_total    = 'da_total_debt_value'

vars_check = [v for v in [var_suspenso, var_ratio, var_total]
              if v in df_original.columns]

if len(vars_check) == 3:
    # Devedores com dívida suspensa > 0
    tem_suspenso = df_original[var_suspenso] > 0
    print(f"\n  Devedores com dívida suspensa > 0  : {tem_suspenso.sum():,}  ({tem_suspenso.mean()*100:.1f}%)")

    # Destes, qual o ratio executável?
    ratio_com_suspenso = df_original.loc[tem_suspenso, var_ratio]
    print(f"  Destes, ratio executável = 1.00    : {(ratio_com_suspenso == 1.0).sum():,}  ({(ratio_com_suspenso == 1.0).mean()*100:.1f}%)")
    print(f"  Destes, ratio executável < 1.00    : {(ratio_com_suspenso < 1.0).sum():,}  ({(ratio_com_suspenso < 1.0).mean()*100:.1f}%)")

    # Recalcular o ratio manualmente para verificar
    ratio_calculado = (
        df_original[var_total] - df_original[var_suspenso]
    ) / df_original[var_total].replace(0, np.nan)

    discrepancia = (ratio_calculado - df_original[var_ratio]).abs()
    print(f"\n  Discrepância entre ratio original e ratio recalculado:")
    print(f"    Máximo   : {discrepancia.max():.6f}")
    print(f"    Mediana  : {discrepancia.median():.6f}")
    print(f"    N > 0.01 : {(discrepancia > 0.01).sum():,}")

else:
    print(f"\n  ⚠  Variáveis em falta para cruzamento: {set([var_suspenso, var_ratio, var_total]) - set(df_original.columns)}")


# --- 3. Verificar se o ratio foi recalculado durante o pré-processamento ---
print("\n" + "─" * 70)
print("Verificação — df_original vs df_pos_missing vs df_pos_outlier")
print("─" * 70)

for nome_df, df_check in [
    ('df_original',    df_original),
    ('df_pos_missing', df_pos_missing if 'df_pos_missing' in dir() else None),
    ('df_pos_outlier', df_pos_outlier if 'df_pos_outlier' in dir() else None),
]:
    if df_check is None:
        print(f"\n  {nome_df}: não disponível em memória")
        continue
    if var_ratio not in df_check.columns:
        print(f"\n  {nome_df}: coluna '{var_ratio}' não encontrada")
        continue

    serie = df_check[var_ratio].dropna()
    pct_um = (serie == 1.0).mean() * 100
    print(f"\n  {nome_df}:")
    print(f"    % com ratio = 1.00 : {pct_um:.1f}%")
    print(f"    Mediana            : {serie.median():.4f}")
    print(f"    Min / Max          : {serie.min():.4f} / {serie.max():.4f}")


# --- 4. Distribuição por decis ---
print("\n" + "─" * 70)
print("Distribuição por decis — da_enforceable_debt_ratio (df_original)")
print("─" * 70)

if var_ratio in df_original.columns:
    decis = df_original[var_ratio].quantile(
        [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    )
    for q, v in decis.items():
        print(f"    P{int(q*100):>3}  :  {v:.4f}")

In [ ]:
# =============================================================================
# DIAGNÓSTICO — Verificação no dataset original (df)
# Soma dos valores absolutos e contagem de ratios = 1
# =============================================================================

print("=" * 70)
print("DIAGNÓSTICO — df (dataset original, pré-pipeline)")
print("=" * 70)

vars_check = [
    'da_enforceable_debt_value',
    'da_admin_suspended_debt_value',
    'da_total_debt_value',
    'da_enforceable_debt_ratio',
    'da_admin_suspended_ratio',
]

# --- 1. Verificar existência das colunas ---
print("\n  Colunas disponíveis em df:")
for v in vars_check:
    existe = v in df.columns
    print(f"    {'✔' if existe else '✗'}  {v}")


# --- 2. Soma dos valores absolutos ---
print("\n" + "─" * 70)
print("Valores absolutos — Soma total na carteira (df)")
print("─" * 70)

for var in ['da_enforceable_debt_value',
            'da_admin_suspended_debt_value',
            'da_total_debt_value']:
    if var in df.columns:
        soma  = df[var].sum()
        nulos = df[var].isna().sum()
        zeros = (df[var] == 0).sum()
        print(f"\n  {var}")
        print(f"    Soma total  : R$ {soma:>20,.2f}")
        print(f"    N nulos     : {nulos:>10,}")
        print(f"    N zeros     : {zeros:>10,}")
        print(f"    N > 0       : {(df[var] > 0).sum():>10,}")


# --- 3. Proporção entre executável e suspenso ---
print("\n" + "─" * 70)
print("Proporção — Executável vs Suspenso (sobre soma total)")
print("─" * 70)

if all(v in df.columns for v in ['da_enforceable_debt_value',
                                  'da_admin_suspended_debt_value',
                                  'da_total_debt_value']):
    soma_exec  = df['da_enforceable_debt_value'].sum()
    soma_susp  = df['da_admin_suspended_debt_value'].sum()
    soma_total = df['da_total_debt_value'].sum()

    print(f"\n  Dívida Executável  : R$ {soma_exec:>20,.2f}  ({soma_exec/soma_total*100:.1f}% do total)")
    print(f"  Dívida Suspensa    : R$ {soma_susp:>20,.2f}  ({soma_susp/soma_total*100:.1f}% do total)")
    print(f"  Dívida Total       : R$ {soma_total:>20,.2f}  (100%)")
    print(f"\n  Soma exec + susp   : R$ {soma_exec + soma_susp:>20,.2f}")
    print(f"  Diferença vs total : R$ {soma_total - (soma_exec + soma_susp):>20,.2f}")


# --- 4. Contagem de linhas com ratio = 1 ---
print("\n" + "─" * 70)
print("Contagem — Linhas com ratio = 1.00 (df original)")
print("─" * 70)

for var_ratio in ['da_enforceable_debt_ratio', 'da_admin_suspended_ratio']:
    if var_ratio not in df.columns:
        print(f"\n  ⚠  {var_ratio} não encontrada em df")
        continue

    serie   = df[var_ratio].dropna()
    n_um    = (serie == 1.0).sum()
    n_zero  = (serie == 0.0).sum()
    n_meio  = ((serie > 0) & (serie < 1)).sum()
    n_nulo  = df[var_ratio].isna().sum()

    print(f"\n  {var_ratio}")
    print(f"    ratio = 1.00       : {n_um:>10,}  ({n_um/len(df)*100:.1f}%)")
    print(f"    ratio = 0.00       : {n_zero:>10,}  ({n_zero/len(df)*100:.1f}%)")
    print(f"    0 < ratio < 1      : {n_meio:>10,}  ({n_meio/len(df)*100:.1f}%)")
    print(f"    nulos              : {n_nulo:>10,}  ({n_nulo/len(df)*100:.1f}%)")


# --- 5. Amostra de devedores com dívida suspensa > 0 ---
print("\n" + "─" * 70)
print("Amostra — 10 devedores com da_admin_suspended_debt_value > 0")
print("─" * 70)

cols_amostra = [v for v in [
    'id_cnpj',
    'da_total_debt_value',
    'da_enforceable_debt_value',
    'da_admin_suspended_debt_value',
    'da_enforceable_debt_ratio',
    'da_admin_suspended_ratio',
] if v in df.columns]

if 'da_admin_suspended_debt_value' in df.columns:
    amostra = (df.loc[df['da_admin_suspended_debt_value'] > 0, cols_amostra]
                 .sort_values('da_admin_suspended_debt_value', ascending=False)
                 .head(10))
    print(f"\n{amostra.to_string(index=False)}")

In [ ]:
# =============================================================================
# VERIFICAÇÃO — da_enforceable_debt_value_log no espaço de features
# =============================================================================

print("=" * 70)
print("VERIFICAÇÃO — da_enforceable_debt_value_log")
print("=" * 70)

var_enf_log   = 'da_enforceable_debt_value_log'
var_total_log = 'da_total_debt_value_log'

# --- 1. Confirmar presença na lista de features ---
print(f"\n  Presente em var_clustering_final : "
      f"{'✔  SIM' if var_enf_log in var_clustering_final else '✗  NÃO'}")
print(f"  Presente em df_clustering        : "
      f"{'✔  SIM' if var_enf_log in df_clustering.columns else '✗  NÃO'}")

# --- 2. Correlação com da_total_debt_value_log ---
print("\n" + "─" * 70)
print("Correlação — da_enforceable_debt_value_log vs da_total_debt_value_log")
print("─" * 70)

if all(v in df_clustering.columns for v in [var_enf_log, var_total_log]):
    r = df_clustering[var_enf_log].corr(df_clustering[var_total_log])
    print(f"\n  r de Pearson : {r:.4f}")

    if abs(r) >= 0.80:
        print(f"\n  ⚠  Correlação acima do threshold (|r| > 0.80).")
        print(f"     Esta variável deveria ter sido removida na etapa de correlação.")
        print(f"     Acção: avaliar remoção e manter apenas da_total_debt_value_log.")
    else:
        print(f"\n  ✔  Correlação abaixo do threshold. Variável discriminante.")

# --- 3. Distribuição no df_original ---
print("\n" + "─" * 70)
print("Distribuição — da_enforceable_debt_value (escala original, df)")
print("─" * 70)

var_enf_orig = 'da_enforceable_debt_value'
if var_enf_orig in df.columns:
    serie = df[var_enf_orig]
    print(f"\n  N > 0       : {(serie > 0).sum():>10,}  ({(serie > 0).mean()*100:.1f}%)")
    print(f"  N = 0       : {(serie == 0).sum():>10,}  ({(serie == 0).mean()*100:.1f}%)")
    print(f"  Mediana     : R$ {serie.median():>15,.2f}")
    print(f"  Média       : R$ {serie.mean():>15,.2f}")
    print(f"  P75         : R$ {serie.quantile(0.75):>15,.2f}")
    print(f"  P90         : R$ {serie.quantile(0.90):>15,.2f}")
    print(f"  P99         : R$ {serie.quantile(0.99):>15,.2f}")
    print(f"  Máximo      : R$ {serie.max():>15,.2f}")
    print(f"  Soma total  : R$ {serie.sum():>15,.2f}")

# --- 4. Poder discriminante marginal ---
# Devedores em que enforceable ≠ total (os 3% com suspensão)
print("\n" + "─" * 70)
print("Poder discriminante — diferença entre executável e total")
print("─" * 70)

if all(v in df.columns for v in [var_enf_orig, 'da_total_debt_value']):
    diff = (df['da_total_debt_value'] - df[var_enf_orig]).abs()
    mask_diff = diff > 0
    print(f"\n  Devedores onde executável ≠ total : {mask_diff.sum():>10,}  ({mask_diff.mean()*100:.1f}%)")
    print(f"  Diferença mediana (nesses casos) : R$ {diff[mask_diff].median():>15,.2f}")
    print(f"  Diferença máxima                 : R$ {diff.max():>15,.2f}")
    print(f"\n  → Para os restantes {(~mask_diff).sum():,} devedores ({(~mask_diff).mean()*100:.1f}%),")
    print(f"    da_enforceable_debt_value_log == da_total_debt_value_log")
    print(f"    (contribuição marginal zero para a distância euclidiana)")

### Remoção variável da_enforceable_debt_ratio

In [ ]:
# Resumo consolidado — decisões sobre variáveis

print("=" * 70)
print("ESPAÇO DE FEATURES — Estado actual")
print("=" * 70)

decisoes = {
    'da_enforceable_debt_ratio':          '✗ Removida — variância quasi-nula (97% = 1.00)',
    'da_enforceable_debt_value_log':      '✔ Mantida  — r = 0.73 com total; discrimina os 3% suspensos',
    'da_admin_suspended_ratio':           '✗ Removida — r = -1.00 com enforceable_ratio',
    'da_admin_suspended_debt_value_log':  '✗ Removida — |r| = 0.877 com ratios de suspensão',
}

for var, decisao in decisoes.items():
    print(f"\n  {var}")
    print(f"    {decisao}")

print(f"\n  Total de variáveis no espaço de features : {len(var_clustering_final)}")
print(f"  Contínuas                                : {len(var_continuas_final)}")
print(f"  Binárias / flags                         : {len(var_binarias_final)}")

In [ ]:
df_clustering.info()

## DETERMINAÇÃO DO NÚMERO DE CLUSTERS - COM 22 VARIÁVEIS APÓS REMOÇÃO DA VARIÁVEL da_enforceable_debt_ratio

### Remoção da feature

In [ ]:
# =============================================================================
# CORRECÇÃO — Remover da_enforceable_debt_ratio do espaço de features
# =============================================================================

var_remover = 'da_enforceable_debt_ratio'

# 1. Remover da lista de features
if var_remover in var_clustering_final:
    var_clustering_final.remove(var_remover)
    print(f"  ✔  Removida de var_clustering_final")
else:
    print(f"  ⚠  Não encontrada em var_clustering_final")

# 2. Remover da sublista de contínuas
if var_remover in var_continuas_final:
    var_continuas_final.remove(var_remover)
    print(f"  ✔  Removida de var_continuas_final")

# 3. Remover da matriz de clustering
if var_remover in df_clustering.columns:
    df_clustering = df_clustering.drop(columns=[var_remover])
    print(f"  ✔  Removida de df_clustering")

# 4. Confirmação
print(f"\n  var_clustering_final : {len(var_clustering_final)} variáveis")
print(f"  df_clustering        : {df_clustering.shape}")
print(f"\n  Variáveis incluídas:")
for i, v in enumerate(var_clustering_final, 1):
    print(f"    {i:>2}. {v}")

### Determinação K

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)
from kneed import KneeLocator
from collections import Counter

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K_MIN        = 2
K_MAX        = 10
k_range      = range(K_MIN, K_MAX + 1)
N_SEEDS      = 10
SAMPLE_SIL   = 30_000

# =============================================================================
# CONFIRMAÇÃO DO ESPAÇO DE FEATURES — 22 variáveis
# =============================================================================

X = df_clustering[var_clustering_final].values

print("=" * 70)
print("K SELECTION — Espaço de features final (22 variáveis)")
print("=" * 70)
print(f"\n  Dataset           : {X.shape[0]:,} observações | {X.shape[1]} variáveis")
print(f"  Contínuas         : {len(var_continuas_final)}")
print(f"  Binárias/flags    : {len(var_binarias_final)}")
print(f"  Range testado     : k = {K_MIN} a {K_MAX}")
print(f"  Seeds estabilidade: {N_SEEDS}")
print(f"  Amostra silhouette: {SAMPLE_SIL:,}")

print(f"\n  Variáveis incluídas:")
for i, v in enumerate(var_clustering_final, 1):
    print(f"    {i:>2}. {v}")

# Confirmação explícita das remoções acumuladas
print(f"""
  Remoções acumuladas ao longo do pipeline:
    ✗  da_admin_suspended_ratio          r = -1.00 com enforceable_ratio
    ✗  da_admin_suspended_debt_value_log |r| = 0.877 com ratios de suspensão
    ✗  da_average_debt_age_months_log    r = 0.947 com versão ponderada
    ✗  da_total_principal_value_log      r = 0.847 com total
    ✗  da_total_penalties_interest_value_log  r = 0.935 com total
    ✗  da_prescription_risk_bin          r_pb = 0.96 com debt_over_48_months
    ✗  da_new_debt_incidence_bin         r_pb = 0.94 com debt_0_6_months
    ✗  pc_recent_payment_indicator_bin   r_pb = 0.82 com total_payment_2025
    ✗  da_enforceable_debt_ratio         variância quasi-nula (97% = 1.00)
    ─────────────────────────────────────────────────────
    31 variáveis originais → 22 variáveis finais
""")


# =============================================================================
# BLOCO 1 — ELBOW (Inércia / WCSS)
# =============================================================================

print("─" * 70)
print("BLOCO 1 — Método do Cotovelo (Elbow / WCSS)")
print("─" * 70)

inertias = []

for k in k_range:
    t0 = time.time()
    km = KMeans(n_clusters=k, init='k-means++', n_init=15,
                max_iter=300, random_state=RANDOM_STATE)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f"  k={k:2d} | Inércia: {km.inertia_:>18,.2f} | {time.time()-t0:.1f}s")

kneedle = KneeLocator(list(k_range), inertias,
                      curve='convex', direction='decreasing')
k_elbow = kneedle.elbow
print(f"\n  ✔  Cotovelo detectado: k = {k_elbow}")


# =============================================================================
# BLOCO 2 — MÉTRICAS DE VALIDAÇÃO INTERNA
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Métricas de Validação Interna")
print("─" * 70)

sil_scores = []
db_scores  = []
ch_scores  = []

rng        = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=min(SAMPLE_SIL, X.shape[0]), replace=False)
X_sample   = X[idx_sample]

for k in k_range:
    t0     = time.time()
    km     = KMeans(n_clusters=k, init='k-means++', n_init=15,
                    max_iter=300, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)

    sil = silhouette_score(X_sample, labels[idx_sample],
                           random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X, labels)
    ch  = calinski_harabasz_score(X, labels)

    sil_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)

    print(f"  k={k:2d} | Silhouette: {sil:.4f} | "
          f"Davies-Bouldin: {db:.4f} | "
          f"Calinski-Harabasz: {ch:>12,.1f} | {time.time()-t0:.1f}s")

k_silhouette = list(k_range)[np.argmax(sil_scores)]
k_db         = list(k_range)[np.argmin(db_scores)]
k_ch         = list(k_range)[np.argmax(ch_scores)]

print(f"\n  Melhor Silhouette       : k = {k_silhouette}  ({max(sil_scores):.4f})")
print(f"  Melhor Davies-Bouldin   : k = {k_db}  ({min(db_scores):.4f})")
print(f"  Melhor Calinski-Harabasz: k = {k_ch}  ({max(ch_scores):,.1f})")


# =============================================================================
# BLOCO 3 — ESTABILIDADE (variância entre seeds)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Análise de Estabilidade (variância entre seeds)")
print("─" * 70)

seeds        = [42, 7, 13, 21, 99, 123, 256, 512, 1024, 2048][:N_SEEDS]
estabilidade = {}

for k in k_range:
    inertias_k = []
    for seed in seeds:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    max_iter=300, random_state=seed)
        km.fit(X)
        inertias_k.append(km.inertia_)

    media  = np.mean(inertias_k)
    desvio = np.std(inertias_k)
    cv     = (desvio / media) * 100
    estabilidade[k] = {'media': media, 'desvio': desvio, 'cv': cv}

    label = ("✔  estável" if cv < 0.5
             else "~  moderado" if cv < 2
             else "⚠  instável")
    print(f"  k={k:2d} | Média: {media:>18,.2f} | "
          f"Std: {desvio:>12,.2f} | CV: {cv:.4f}%  {label}")


# =============================================================================
# BLOCO 4 — SCORE COMPOSTO E CONSENSO
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Score Composto e Consenso")
print("─" * 70)

resumo = pd.DataFrame({
    'k':                 list(k_range),
    'inertia':           inertias,
    'silhouette':        sil_scores,
    'davies_bouldin':    db_scores,
    'calinski_harabasz': ch_scores,
    'estab_cv_pct':      [estabilidade[k]['cv'] for k in k_range],
})

eps = 1e-10
resumo['rank_sil']    = (resumo['silhouette'] - resumo['silhouette'].min()) / (resumo['silhouette'].max() - resumo['silhouette'].min() + eps)
resumo['rank_db']     = 1 - (resumo['davies_bouldin'] - resumo['davies_bouldin'].min()) / (resumo['davies_bouldin'].max() - resumo['davies_bouldin'].min() + eps)
resumo['rank_ch']     = (resumo['calinski_harabasz'] - resumo['calinski_harabasz'].min()) / (resumo['calinski_harabasz'].max() - resumo['calinski_harabasz'].min() + eps)
resumo['rank_estab']  = 1 - (resumo['estab_cv_pct'] - resumo['estab_cv_pct'].min()) / (resumo['estab_cv_pct'].max() - resumo['estab_cv_pct'].min() + eps)
resumo['score_total'] = resumo[['rank_sil','rank_db','rank_ch','rank_estab']].mean(axis=1)

votos      = Counter([k_elbow, k_silhouette, k_db, k_ch])
k_consenso = votos.most_common(1)[0][0]
n_votos    = votos.most_common(1)[0][1]
k_score    = int(resumo.loc[resumo['score_total'].idxmax(), 'k'])

if n_votos >= 3:
    consenso_label = f"✔  CONSENSO FORTE ({n_votos}/4 métodos)"
elif n_votos == 2:
    consenso_label = f"~  CONSENSO PARCIAL ({n_votos}/4 métodos)"
else:
    consenso_label = "⚠  SEM CONSENSO — avaliar estabilidade e interpretabilidade"

print(f"\n  Elbow                   : k = {k_elbow}")
print(f"  Silhouette              : k = {k_silhouette}")
print(f"  Davies-Bouldin          : k = {k_db}")
print(f"  Calinski-Harabasz       : k = {k_ch}")
print(f"  Score composto          : k = {k_score}")
print(f"\n  {consenso_label}")


# =============================================================================
# BLOCO 5 — TABELA RESUMO
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 5 — Tabela Resumo Consolidada")
print("─" * 70)

print(f"\n  {'k':>3} {'Inércia':>20} {'Silhouette':>12} {'DB':>10} "
      f"{'CH':>14} {'CV%':>8} {'Score':>8}")
print("  " + "─" * 78)

for _, row in resumo.iterrows():
    k_val     = int(row['k'])
    marcador  = "  ◄" if k_val == k_score else ""
    print(f"  {k_val:>3} {row['inertia']:>20,.2f} {row['silhouette']:>12.4f} "
          f"{row['davies_bouldin']:>10.4f} {row['calinski_harabasz']:>14,.1f} "
          f"{row['estab_cv_pct']:>8.4f} {row['score_total']:>8.4f}{marcador}")

print(f"\n  ◄  k com maior score composto: {k_score}")


# =============================================================================
# BLOCO 6 — COMPARAÇÃO COM RODADAS ANTERIORES
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 6 — Comparação entre rodadas")
print("─" * 70)

print(f"""
  {'k':>3}  {'Silhouette (31 vars)':>22}  {'Silhouette (23 vars)':>22}  {'Silhouette (22 vars)':>22}
  {'─'*75}""")

sil_31 = {2: 0.4101, 3: 0.3852, 4: 0.2549, 5: 0.2587}
sil_23 = {2: 0.4296, 3: 0.4235, 4: 0.2557, 5: 0.2416}

for k, sil_22 in zip(k_range, sil_scores):
    s31 = f"{sil_31[k]:.4f}" if k in sil_31 else "  —   "
    s23 = f"{sil_23[k]:.4f}" if k in sil_23 else "  —   "
    delta = ""
    if k in sil_23:
        diff = sil_22 - sil_23[k]
        delta = f"  ({'+' if diff >= 0 else ''}{diff:.4f})"
    print(f"  {k:>3}  {s31:>22}  {s23:>22}  {sil_22:>8.4f}{delta}")


# =============================================================================
# BLOCO 7 — VISUALIZAÇÕES
# =============================================================================

ks            = list(k_range)
cor_principal = '#2c3e50'
cor_destaque  = '#e74c3c'

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# --- 1. Elbow ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ks, inertias, 'o-', color=cor_principal, linewidth=2, markersize=7)
if k_elbow:
    ax1.axvline(k_elbow, color=cor_destaque, linestyle='--',
                lw=1.8, label=f'k={k_elbow}')
    ax1.legend(fontsize=9)
ax1.set_title('Elbow (Inércia / WCSS)', fontweight='bold')
ax1.set_xlabel('k'); ax1.set_ylabel('WCSS')
ax1.set_xticks(ks); ax1.grid(alpha=0.3)

# --- 2. Silhouette ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ks, sil_scores, 'o-', color='#27ae60', linewidth=2, markersize=7)
ax2.axvline(k_silhouette, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_silhouette}')
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.set_xlabel('k'); ax2.set_ylabel('Score')
ax2.set_xticks(ks); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

# --- 3. Davies-Bouldin ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ks, db_scores, 'o-', color='#8e44ad', linewidth=2, markersize=7)
ax3.axvline(k_db, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_db}')
ax3.set_title('Davies-Bouldin (↓ melhor)', fontweight='bold')
ax3.set_xlabel('k'); ax3.set_ylabel('Índice')
ax3.set_xticks(ks); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

# --- 4. Calinski-Harabasz ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(ks, ch_scores, 'o-', color='#e67e22', linewidth=2, markersize=7)
ax4.axvline(k_ch, color=cor_destaque, linestyle='--',
            lw=1.8, label=f'k={k_ch}')
ax4.set_title('Calinski-Harabasz (↑ melhor)', fontweight='bold')
ax4.set_xlabel('k'); ax4.set_ylabel('Índice')
ax4.set_xticks(ks); ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# --- 5. Estabilidade ---
cvs = [estabilidade[k]['cv'] for k in k_range]
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(ks, cvs, 'o-', color='#16a085', linewidth=2, markersize=7)
ax5.axhline(0.5, color=cor_destaque, linestyle=':', lw=1.5,
            label='Threshold 0.5%')
ax5.set_title('Estabilidade — CV% Inércia (↓ melhor)', fontweight='bold')
ax5.set_xlabel('k'); ax5.set_ylabel('CV (%)')
ax5.set_xticks(ks); ax5.legend(fontsize=9); ax5.grid(alpha=0.3)

# --- 6. Score Composto ---
ax6 = fig.add_subplot(gs[1, 2])
scores    = resumo['score_total'].tolist()
cores_bar = [cor_destaque if k == k_score else '#bdc3c7' for k in ks]
ax6.bar(ks, scores, color=cores_bar, edgecolor='white', linewidth=0.8)
ax6.set_title('Score Composto — média dos rankings', fontweight='bold')
ax6.set_xlabel('k'); ax6.set_ylabel('Score [0–1]')
ax6.set_xticks(ks); ax6.grid(axis='y', alpha=0.3)
val_k = resumo.loc[resumo['k'] == k_score, 'score_total'].values[0]
ax6.annotate(f'k={k_score}',
             xy=(k_score, val_k),
             xytext=(k_score + 0.4, val_k + 0.02),
             fontsize=9, color=cor_destaque,
             arrowprops=dict(arrowstyle='->', color=cor_destaque))

# --- 7. Silhouette Plot para k_score ---
ax7 = fig.add_subplot(gs[2, :])
km_final     = KMeans(n_clusters=k_score, init='k-means++', n_init=15,
                      max_iter=300, random_state=RANDOM_STATE)
labels_final = km_final.fit_predict(X)
sil_vals     = silhouette_samples(X[idx_sample], labels_final[idx_sample])

y_lower        = 10
cores_clusters = plt.cm.tab10(np.linspace(0, 1, k_score))

for i in range(k_score):
    sil_i   = np.sort(sil_vals[labels_final[idx_sample] == i])
    size_i  = len(sil_i)
    y_upper = y_lower + size_i
    ax7.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                      facecolor=cores_clusters[i], alpha=0.8)
    ax7.text(-0.06, y_lower + size_i / 2, f'C{i}',
             fontsize=9, color=cores_clusters[i], fontweight='bold')
    y_lower = y_upper + 10

media_sil = np.mean(sil_vals)
ax7.axvline(media_sil, color=cor_destaque, linestyle='--', lw=1.8,
            label=f'Média = {media_sil:.4f}')
ax7.set_title(f'Silhouette Plot — k={k_score}  (amostra {SAMPLE_SIL:,})',
              fontweight='bold')
ax7.set_xlabel('Coeficiente de Silhouette')
ax7.set_ylabel('Cluster'); ax7.set_yticks([])
ax7.legend(loc='lower right', fontsize=9)
ax7.grid(axis='x', alpha=0.3)

plt.suptitle(
    f"K Selection — 22 variáveis (espaço de features final)\n"
    f"377.419 devedores | RFB 9ª Região Fiscal",
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('k_selection_22vars.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE FINAL — K Selection (22 variáveis)")
print("=" * 70)
print(f"""
  Elbow (Inércia)         : k = {k_elbow}
  Silhouette Score        : k = {k_silhouette}
  Davies-Bouldin Index    : k = {k_db}
  Calinski-Harabasz Index : k = {k_ch}
  Score Composto          : k = {k_score}

  {consenso_label}

  Estabilidade k=3 : CV = {estabilidade[3]['cv']:.4f}%
  Estabilidade k=4 : CV = {estabilidade[4]['cv']:.4f}%

  Candidatos para modelagem comparativa : k=3  e  k=4
  → Aguardar perfilamento para decisão final.
""")

## Modelling - 2

### Treino e comparação K = 3 e K = 4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
X = df_clustering[var_clustering_final].values

# Amostra para silhouette
rng        = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=30_000, replace=False)

# =============================================================================
# DICIONÁRIOS DE PERFILAMENTO
# Variáveis operacionais em escala original (df)
# =============================================================================

vars_perfil_continuas = {
    'da_total_debt_value':                'Dívida Total (R$)',
    'da_enforceable_debt_value':          'Dívida Executável (R$)',
    'da_debt_pgfn_value':                 'Dívida PGFN (R$)',
    'da_debt_0_6_months_value':           'Dívida 0–6 meses (R$)',
    'da_debt_over_48_months_value':       'Dívida >48 meses (R$)',
    'da_penalty_to_principal_ratio':      'Ratio Multas/Principal',
    'da_age_weighted_debt_value_months':  'Idade Pond. Dívida (meses)',
    'da_number_of_debts_count':           'Nº Débitos',
    'da_number_distinct_tax_types_count': 'Nº Tipos Tributo',
    'ap_declared_gross_revedue_value':    'Receita Declarada (R$)',
    'ap_payment_capacity_value':          'Capacidade Pagamento (R$)',
    'ap_debt_to_revenue_ratio':           'Dívida/Receita',
    'pc_total_payment_2025_value':        'Pagamentos 2025 (R$)',
    'pc_instalment_plan_count':           'Nº Parcelamentos',
    'pc_defaulted_instalment_plans_count':'Parcelamentos Rompidos',
    'pc_payment_regularity_ratio':        'Regularidade Parcelamentos',
    'pc_pending_filing_omissions_ratio':  'Omissões de Declaração',
    'tp_enterprise_age_years':            'Idade Empresa (anos)',
    'ci_sectoral_delinquency_deviation_score': 'Desvio Setorial',
}

vars_perfil_binarias = {
    'da_prescription_risk_bin':      'Risco Prescrição (%)',
    'da_new_debt_incidence_bin':     'Incidência Nova Dívida (%)',
    'pc_recent_payment_indicator_bin': 'Pagamento Recente (%)',
}

vars_flags = {
    'is_missing_ap_declared_gross_revenue_value':  'Sem Receita Declarada (%)',
    'is_missing_ap_debt_to_revenue_ratio':         'Sem Ratio Dívida/Receita (%)',
    'is_missing_pc_pending_filing_omissions_ratio':'Sem Dados Omissões (%)',
}

# Filtrar apenas colunas existentes em df
vars_cont_ok  = {k: v for k, v in vars_perfil_continuas.items() if k in df.columns}
vars_bin_ok   = {k: v for k, v in vars_perfil_binarias.items()  if k in df.columns}
vars_flags_ok = {k: v for k, v in vars_flags.items()            if k in df.columns}


# =============================================================================
# FUNÇÃO DE PERFILAMENTO — reutilizada para k=3 e k=4
# =============================================================================

def formatar_valor(v, p25=None, p75=None):
    """Formata mediana + IQR de acordo com magnitude."""
    iqr = f"  [{p25:,.0f} – {p75:,.0f}]" if p25 is not None and v >= 1_000 else \
          f"  [{p25:.2f} – {p75:.2f}]"   if p25 is not None and v >= 1   else \
          f"  [{p25:.4f} – {p75:.4f}]"   if p25 is not None              else ""
    if v >= 1_000:
        return f"R$ {v:>12,.0f}{iqr}"
    elif v >= 1:
        return f"{v:>10.2f}{iqr}"
    else:
        return f"{v:>10.4f}{iqr}"


def gerar_perfil(df_labeled, col_cluster, n_clusters):
    """
    Retorna DataFrame com mediana + IQR para contínuas
    e percentual para binárias/flags, por cluster.
    """
    clusters  = list(range(n_clusters))
    col_names = [f'C{c}' for c in clusters] + ['Global']
    rows      = []

    # --- Tamanho ---
    row = {'Label': 'Tamanho do cluster', 'Secao': '── Dimensão'}
    n_total = len(df_labeled)
    for c in clusters:
        n   = (df_labeled[col_cluster] == c).sum()
        row[f'C{c}'] = f"{n:,}  ({n/n_total*100:.1f}%)"
    row['Global'] = f"{n_total:,}  (100%)"
    rows.append(row)

    # --- Contínuas ---
    rows.append({'Label': '', 'Secao': '── Variáveis Operacionais (Mediana  [P25 – P75])'})
    for var, label in vars_cont_ok.items():
        row = {'Label': label, 'Secao': ''}
        for c in clusters:
            s   = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            row[f'C{c}'] = formatar_valor(s.median(), s.quantile(0.25), s.quantile(0.75))
        sg  = df_labeled[var].dropna()
        row['Global'] = formatar_valor(sg.median(), sg.quantile(0.25), sg.quantile(0.75))
        rows.append(row)

    # --- Binárias e flags ---
    rows.append({'Label': '', 'Secao': '── Indicadores Descritivos (%)'})
    for var, label in {**vars_bin_ok, **vars_flags_ok}.items():
        row = {'Label': label, 'Secao': ''}
        for c in clusters:
            s   = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            row[f'C{c}'] = f"{s.mean()*100:.1f}%"
        row['Global'] = f"{df_labeled[var].dropna().mean()*100:.1f}%"
        rows.append(row)

    return pd.DataFrame(rows)


def imprimir_perfil(df_perfil, n_clusters, titulo):
    col_data  = [f'C{c}' for c in range(n_clusters)] + ['Global']
    largura_c = 32

    print(f"\n{'='*70}")
    print(titulo)
    print(f"{'='*70}")

    for _, row in df_perfil.iterrows():
        if row['Secao'] and row['Label'] == '':
            print(f"\n  {row['Secao']}")
            print("  " + "─" * (45 + largura_c * len(col_data)))
            continue
        if row['Label'] == '':
            continue
        linha = f"  {row['Label']:<44}"
        for col in col_data:
            linha += f"  {str(row.get(col,'')):<{largura_c}}"
        print(linha)


# =============================================================================
# BLOCO A — K-MEANS k=3
# =============================================================================

print("=" * 70)
print("MODELAGEM — K-Means  k=3")
print("=" * 70)

km3 = KMeans(n_clusters=3, init='k-means++', n_init=20,
             max_iter=500, random_state=RANDOM_STATE)
labels_k3 = km3.fit_predict(X)

sil_k3 = silhouette_score(X[idx_sample], labels_k3[idx_sample])
dist_k3 = pd.Series(labels_k3).value_counts().sort_index()

print(f"\n  Inércia            : {km3.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k3:.4f}")
print(f"\n  Distribuição:")
for c, n in dist_k3.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k3)*100:.1f}%)")

# Juntar labels ao df original
df_orig_k3             = df.copy()
df_orig_k3['cluster']  = labels_k3

# Perfilamento k=3
perfil_k3 = gerar_perfil(df_orig_k3, 'cluster', 3)
imprimir_perfil(perfil_k3, 3, "TABELA DE PERFILAMENTO — k=3")


# =============================================================================
# BLOCO B — K-MEANS k=4
# =============================================================================

print("\n\n" + "=" * 70)
print("MODELAGEM — K-Means  k=4")
print("=" * 70)

km4 = KMeans(n_clusters=4, init='k-means++', n_init=20,
             max_iter=500, random_state=RANDOM_STATE)
labels_k4 = km4.fit_predict(X)

sil_k4 = silhouette_score(X[idx_sample], labels_k4[idx_sample])
dist_k4 = pd.Series(labels_k4).value_counts().sort_index()

print(f"\n  Inércia            : {km4.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k4:.4f}")
print(f"\n  Distribuição:")
for c, n in dist_k4.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k4)*100:.1f}%)")

# Juntar labels ao df original
df_orig_k4            = df.copy()
df_orig_k4['cluster'] = labels_k4

# Perfilamento k=4
perfil_k4 = gerar_perfil(df_orig_k4, 'cluster', 4)
imprimir_perfil(perfil_k4, 4, "TABELA DE PERFILAMENTO — k=4")


# =============================================================================
# BLOCO C — VISUALIZAÇÕES COMPARATIVAS
# =============================================================================

fig = plt.figure(figsize=(20, 18))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.38)

cores_k3 = ['#2980b9', '#e74c3c', '#27ae60']
cores_k4 = ['#2980b9', '#e74c3c', '#27ae60', '#f39c12']

# --- 1. Distribuição k=3 ---
ax1 = fig.add_subplot(gs[0, 0])
ns3  = [dist_k3[c] for c in range(3)]
bars = ax1.bar([f'C{c}' for c in range(3)], ns3,
               color=cores_k3, edgecolor='white')
for bar, n in zip(bars, ns3):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             f'{n:,}\n({n/len(labels_k3)*100:.1f}%)',
             ha='center', va='bottom', fontsize=8)
ax1.set_title('Tamanho — k=3', fontweight='bold')
ax1.set_ylabel('N devedores'); ax1.grid(axis='y', alpha=0.3)

# --- 2. Distribuição k=4 ---
ax2 = fig.add_subplot(gs[0, 1])
ns4  = [dist_k4[c] for c in range(4)]
bars = ax2.bar([f'C{c}' for c in range(4)], ns4,
               color=cores_k4, edgecolor='white')
for bar, n in zip(bars, ns4):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             f'{n:,}\n({n/len(labels_k4)*100:.1f}%)',
             ha='center', va='bottom', fontsize=8)
ax2.set_title('Tamanho — k=4', fontweight='bold')
ax2.set_ylabel('N devedores'); ax2.grid(axis='y', alpha=0.3)


# --- 3. Boxplot Dívida Total k=3 ---
var_box = 'da_total_debt_value'
ax3 = fig.add_subplot(gs[0, 2])
dados = [np.log1p(df_orig_k3.loc[df_orig_k3['cluster']==c, var_box].dropna())
         for c in range(3)]
bp = ax3.boxplot(dados, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, cor in zip(bp['boxes'], cores_k3):
    patch.set_facecolor(cor); patch.set_alpha(0.8)
ax3.set_xticklabels([f'C{c}' for c in range(3)])
ax3.set_title('Dívida Total log — k=3', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# --- 4. Boxplot Dívida Total k=4 ---
ax4 = fig.add_subplot(gs[0, 3])
dados = [np.log1p(df_orig_k4.loc[df_orig_k4['cluster']==c, var_box].dropna())
         for c in range(4)]
bp = ax4.boxplot(dados, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, cor in zip(bp['boxes'], cores_k4):
    patch.set_facecolor(cor); patch.set_alpha(0.8)
ax4.set_xticklabels([f'C{c}' for c in range(4)])
ax4.set_title('Dívida Total log — k=4', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)


# --- 5. Heatmap desvios k=3 ---
vars_hm = [v for v in [
    'da_total_debt_value', 'da_debt_over_48_months_value',
    'da_penalty_to_principal_ratio', 'pc_total_payment_2025_value',
    'pc_instalment_plan_count', 'pc_payment_regularity_ratio',
    'ap_payment_capacity_value', 'ap_debt_to_revenue_ratio',
    'tp_enterprise_age_years', 'ci_sectoral_delinquency_deviation_score'
] if v in df.columns]

def heatmap_desvios(ax, df_labeled, n_clusters, cores, titulo):
    med_g  = df_labeled[vars_hm].median()
    desvios = pd.DataFrame(index=range(n_clusters), columns=vars_hm, dtype=float)
    for c in range(n_clusters):
        med_c = df_labeled.loc[df_labeled['cluster']==c, vars_hm].median()
        desvios.loc[c] = ((med_c - med_g) / med_g.replace(0, np.nan)) * 100
    desvios.index = [f'C{c}' for c in range(n_clusters)]
    sns.heatmap(desvios.T.astype(float), annot=True, fmt='.0f',
                cmap='RdBu_r', center=0, linewidths=0.4,
                linecolor='#ecf0f1', annot_kws={'size': 8}, ax=ax,
                cbar_kws={'shrink': 0.6})
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=7)

ax5 = fig.add_subplot(gs[1, :2])
heatmap_desvios(ax5, df_orig_k3, 3, cores_k3,
                'Desvio da Mediana Global — k=3')

ax6 = fig.add_subplot(gs[1, 2:])
heatmap_desvios(ax6, df_orig_k4, 4, cores_k4,
                'Desvio da Mediana Global — k=4')


# --- 6. Silhouette plots ---
def silhouette_subplot(ax, X, labels, k, cores, titulo):
    sil_vals = silhouette_samples(X[idx_sample], labels[idx_sample])
    y_lower  = 10
    for i in range(k):
        sil_i  = np.sort(sil_vals[labels[idx_sample] == i])
        size_i = len(sil_i)
        ax.fill_betweenx(np.arange(y_lower, y_lower + size_i),
                         0, sil_i, facecolor=cores[i], alpha=0.8)
        ax.text(-0.06, y_lower + size_i/2, f'C{i}',
                fontsize=9, color=cores[i], fontweight='bold')
        y_lower += size_i + 10
    media = np.mean(sil_vals)
    ax.axvline(media, color='black', linestyle='--', lw=1.8,
               label=f'Média = {media:.4f}')
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.set_xlabel('Coeficiente de Silhouette')
    ax.set_yticks([])
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

ax7 = fig.add_subplot(gs[2, :2])
silhouette_subplot(ax7, X, labels_k3, 3, cores_k3,
                   f'Silhouette Plot — k=3  (amostra 30k)')

ax8 = fig.add_subplot(gs[2, 2:])
silhouette_subplot(ax8, X, labels_k4, 4, cores_k4,
                   f'Silhouette Plot — k=4  (amostra 30k)')

plt.suptitle("Modelagem Comparativa — K-Means k=3 vs k=4\n"
             "377.419 devedores | RFB 9ª Região Fiscal | 22 variáveis",
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('modelagem_k3_k4_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# BLOCO D — SÍNTESE COMPARATIVA
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE COMPARATIVA — k=3 vs k=4")
print("=" * 70)

print(f"""
  {'Métrica':<30} {'k=3':>12} {'k=4':>12}
  {'─'*56}
  {'Inércia':<30} {km3.inertia_:>12,.0f} {km4.inertia_:>12,.0f}
  {'Silhouette (30k)':<30} {sil_k3:>12.4f} {sil_k4:>12.4f}
  {'Nº variáveis':<30} {'22':>12} {'22':>12}
  {'Nº devedores':<30} {len(labels_k3):>12,} {len(labels_k4):>12,}

  Distribuição k=3:""")
for c, n in dist_k3.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k3)*100:.1f}%)")

print(f"\n  Distribuição k=4:")
for c, n in dist_k4.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k4)*100:.1f}%)")

print(f"""
  Modelos guardados em   : km3, km4
  Labels k=3 guardadas em: df_orig_k3['cluster']
  Labels k=4 guardadas em: df_orig_k4['cluster']

  Próximo passo:
  → Analisar perfis e identificar o cluster de k=3 que k=4 subdivide
  → Redigir parágrafo comparativo para a dissertação
  → Seleccionar k final com base na utilidade operacional dos segmentos
""")

### Testes de avaliação 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score,
                             adjusted_rand_score,
                             normalized_mutual_info_score)
from sklearn.metrics.pairwise import euclidean_distances
import warnings

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SEEDS      = 20
X            = df_clustering[var_clustering_final].values

seeds = [42, 7, 13, 21, 99, 123, 256, 512, 1024, 2048,
         11, 17, 31, 47, 61, 73, 89, 101, 137, 199][:N_SEEDS]


# =============================================================================
# BLOCO 1 — ESTABILIDADE DE LABELS (ARI + NMI)
# Para k=3 e k=4 — N_SEEDS execuções com sementes distintas
# =============================================================================

print("=" * 70)
print("BLOCO 1 — Estabilidade de Labels (ARI / NMI)")
print("=" * 70)


def estabilidade_labels(X, k, seeds, random_state_ref=42):
    """
    Treina k-means com N seeds distintas.
    Compara cada solução com a solução de referência (seed=random_state_ref)
    via ARI e NMI.
    """
    # Solução de referência
    km_ref = KMeans(n_clusters=k, init='k-means++', n_init=20,
                    max_iter=500, random_state=random_state_ref)
    labels_ref = km_ref.fit_predict(X)

    aris, nmis, inertias = [], [], []

    for seed in seeds:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    max_iter=300, random_state=seed)
        labels = km.fit_predict(X)

        ari = adjusted_rand_score(labels_ref, labels)
        nmi = normalized_mutual_info_score(labels_ref, labels,
                                           average_method='arithmetic')
        aris.append(ari)
        nmis.append(nmi)
        inertias.append(km.inertia_)

    return {
        'labels_ref': labels_ref,
        'aris':       aris,
        'nmis':       nmis,
        'inertias':   inertias,
        'ari_mean':   np.mean(aris),
        'ari_std':    np.std(aris),
        'ari_min':    np.min(aris),
        'nmi_mean':   np.mean(nmis),
        'nmi_std':    np.std(nmis),
        'nmi_min':    np.min(nmis),
    }


resultados = {}
for k in [3, 4]:
    print(f"\n  k={k} — {N_SEEDS} seeds")
    print(f"  {'Seed':>6}  {'ARI':>8}  {'NMI':>8}  {'Inércia':>18}")
    print("  " + "─" * 46)

    res = estabilidade_labels(X, k, seeds)
    resultados[k] = res

    for i, seed in enumerate(seeds):
        print(f"  {seed:>6}  {res['aris'][i]:>8.4f}  "
              f"{res['nmis'][i]:>8.4f}  {res['inertias'][i]:>18,.2f}")

    print(f"\n  Resumo k={k}:")
    print(f"    ARI  — Média: {res['ari_mean']:.4f} | "
          f"Std: {res['ari_std']:.4f} | Min: {res['ari_min']:.4f}")
    print(f"    NMI  — Média: {res['nmi_mean']:.4f} | "
          f"Std: {res['nmi_std']:.4f} | Min: {res['nmi_min']:.4f}")

    # Interpretação
    if res['ari_min'] >= 0.90:
        estab = "✔  ESTABILIDADE FORTE (ARI_min ≥ 0.90)"
    elif res['ari_min'] >= 0.75:
        estab = "~  ESTABILIDADE MODERADA (0.75 ≤ ARI_min < 0.90)"
    else:
        estab = "⚠  ESTABILIDADE FRACA (ARI_min < 0.75)"
    print(f"    {estab}")


# =============================================================================
# BLOCO 2 — DUNN INDEX
# Razão entre menor distância inter-cluster e maior diâmetro intra-cluster
# Computacionalmente custoso — usar amostra estratificada
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Dunn Index (amostra 10k por cluster)")
print("─" * 70)

SAMPLE_DUNN = 10_000

def dunn_index_amostra(X, labels, sample_size=SAMPLE_DUNN, seed=42):
    """
    Estima o Dunn Index via amostragem estratificada por cluster.
    """
    rng      = np.random.default_rng(seed)
    clusters = np.unique(labels)
    k        = len(clusters)

    # Amostra estratificada
    idx_amostras = []
    for c in clusters:
        idx_c = np.where(labels == c)[0]
        n_s   = min(sample_size // k, len(idx_c))
        idx_amostras.append(rng.choice(idx_c, size=n_s, replace=False))

    # Diâmetros intra-cluster (distância máxima dentro de cada cluster)
    diametros = []
    for idx in idx_amostras:
        if len(idx) > 1:
            D = euclidean_distances(X[idx])
            diametros.append(D.max())
        else:
            diametros.append(0)

    # Distâncias inter-cluster (distância mínima entre clusters distintos)
    inter_dists = []
    for i in range(k):
        for j in range(i + 1, k):
            D_ij = euclidean_distances(X[idx_amostras[i]], X[idx_amostras[j]])
            inter_dists.append(D_ij.min())

    dunn = min(inter_dists) / max(diametros) if max(diametros) > 0 else 0
    return dunn, diametros, inter_dists


print(f"\n  {'k':>3}  {'Dunn Index':>12}  {'Min Inter-Dist':>16}  "
      f"{'Max Intra-Diam':>16}  Avaliação")
print("  " + "─" * 70)

dunn_resultados = {}
for k in [3, 4]:
    labels = resultados[k]['labels_ref']
    dunn, diametros, inter_dists = dunn_index_amostra(X, labels)
    dunn_resultados[k] = dunn

    avaliacao = "✔  bom" if dunn >= 0.3 else "~  aceitável" if dunn >= 0.1 else "⚠  fraco"
    print(f"  {k:>3}  {dunn:>12.4f}  {min(inter_dists):>16.4f}  "
          f"{max(diametros):>16.4f}  {avaliacao}")

    print(f"       Diâmetros por cluster : "
          f"{[f'{d:.2f}' for d in diametros]}")


# =============================================================================
# BLOCO 3 — GAP STATISTIC (versão eficiente com amostra)
# Tibshirani et al. (2001) — compara inércia observada vs referência uniforme
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Gap Statistic  (B=10 referências, amostra 20k)")
print("─" * 70)

SAMPLE_GAP = 20_000
B_GAP      = 10

rng_gap    = np.random.default_rng(RANDOM_STATE)
idx_gap    = rng_gap.choice(X.shape[0], size=SAMPLE_GAP, replace=False)
X_gap      = X[idx_gap]

# Limites para distribuição uniforme de referência
X_min = X_gap.min(axis=0)
X_max = X_gap.max(axis=0)

gap_stats  = {}
for k in range(2, 8):

    # Inércia observada
    km_obs = KMeans(n_clusters=k, init='k-means++', n_init=5,
                    max_iter=200, random_state=RANDOM_STATE)
    km_obs.fit(X_gap)
    log_inertia_obs = np.log(km_obs.inertia_)

    # Inércias de referência (dados uniformes)
    log_inertias_ref = []
    for b in range(B_GAP):
        X_ref = rng_gap.uniform(X_min, X_max, size=X_gap.shape)
        km_ref = KMeans(n_clusters=k, init='k-means++', n_init=3,
                        max_iter=200, random_state=b)
        km_ref.fit(X_ref)
        log_inertias_ref.append(np.log(km_ref.inertia_))

    gap       = np.mean(log_inertias_ref) - log_inertia_obs
    sdk       = np.std(log_inertias_ref) * np.sqrt(1 + 1/B_GAP)
    gap_stats[k] = {'gap': gap, 'sdk': sdk}

    print(f"  k={k:>2} | Gap = {gap:>7.4f} | sdk = {sdk:>7.4f}")

# Regra de Tibshirani: menor k tal que Gap(k) ≥ Gap(k+1) - sdk(k+1)
k_gap = None
for k in range(2, 7):
    if gap_stats[k]['gap'] >= gap_stats[k+1]['gap'] - gap_stats[k+1]['sdk']:
        k_gap = k
        break

print(f"\n  ✔  Gap Statistic sugere: k = {k_gap}")


# =============================================================================
# BLOCO 4 — TABELA SÍNTESE — TODAS AS MÉTRICAS
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Síntese de Todas as Métricas (k=3 vs k=4)")
print("─" * 70)

print(f"""
  {'Métrica':<35} {'k=3':>12} {'k=4':>12}  Interpretação
  {'─'*75}
  Inércia (WCSS)              {km3.inertia_:>12,.0f} {km4.inertia_:>12,.0f}  Menor = melhor
  Silhouette Score            {sil_k3:>12.4f} {sil_k4:>12.4f}  Maior = melhor
  Davies-Bouldin Index        {'ver K-Selection':>12} {'ver K-Selection':>12}  Menor = melhor
  Calinski-Harabasz Index     {'ver K-Selection':>12} {'ver K-Selection':>12}  Maior = melhor
  Dunn Index                  {dunn_resultados[3]:>12.4f} {dunn_resultados[4]:>12.4f}  Maior = melhor
  ARI — Média ({N_SEEDS} seeds)      {resultados[3]['ari_mean']:>12.4f} {resultados[4]['ari_mean']:>12.4f}  Maior = melhor
  ARI — Mínimo                {resultados[3]['ari_min']:>12.4f} {resultados[4]['ari_min']:>12.4f}  > 0.90 = forte
  NMI — Média                 {resultados[3]['nmi_mean']:>12.4f} {resultados[4]['nmi_mean']:>12.4f}  Maior = melhor
  Gap Statistic (k óptimo)    {'k='+str(k_gap):>12} {'k='+str(k_gap):>12}  referência
""")


# =============================================================================
# BLOCO 5 — VISUALIZAÇÕES
# =============================================================================

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ks      = list(range(2, 8))
cor_k3  = '#2980b9'
cor_k4  = '#e74c3c'

# --- 1. ARI por seed — k=3 ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(1, N_SEEDS+1), resultados[3]['aris'],
         'o-', color=cor_k3, linewidth=1.8, markersize=6, label='k=3')
ax1.plot(range(1, N_SEEDS+1), resultados[4]['aris'],
         's-', color=cor_k4, linewidth=1.8, markersize=6, label='k=4')
ax1.axhline(0.90, color='grey', linestyle=':', lw=1.5, label='Threshold 0.90')
ax1.set_title('ARI por Seed (vs referência seed=42)', fontweight='bold')
ax1.set_xlabel('Seed (índice)'); ax1.set_ylabel('ARI')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1.05)

# --- 2. NMI por seed ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(range(1, N_SEEDS+1), resultados[3]['nmis'],
         'o-', color=cor_k3, linewidth=1.8, markersize=6, label='k=3')
ax2.plot(range(1, N_SEEDS+1), resultados[4]['nmis'],
         's-', color=cor_k4, linewidth=1.8, markersize=6, label='k=4')
ax2.axhline(0.90, color='grey', linestyle=':', lw=1.5, label='Threshold 0.90')
ax2.set_title('NMI por Seed (vs referência seed=42)', fontweight='bold')
ax2.set_xlabel('Seed (índice)'); ax2.set_ylabel('NMI')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1.05)

# --- 3. Boxplot ARI k=3 vs k=4 ---
ax3 = fig.add_subplot(gs[0, 2])
bp = ax3.boxplot([resultados[3]['aris'], resultados[4]['aris']],
                 patch_artist=True, notch=False,
                 medianprops=dict(color='white', linewidth=2))
bp['boxes'][0].set_facecolor(cor_k3); bp['boxes'][0].set_alpha(0.8)
bp['boxes'][1].set_facecolor(cor_k4); bp['boxes'][1].set_alpha(0.8)
ax3.axhline(0.90, color='grey', linestyle=':', lw=1.5)
ax3.set_xticklabels(['k=3', 'k=4'])
ax3.set_title('Distribuição ARI — k=3 vs k=4', fontweight='bold')
ax3.set_ylabel('ARI'); ax3.grid(axis='y', alpha=0.3)
ax3.set_ylim(0, 1.05)

# --- 4. Gap Statistic ---
ax4 = fig.add_subplot(gs[1, 0])
gap_vals = [gap_stats[k]['gap'] for k in ks]
sdk_vals = [gap_stats[k]['sdk'] for k in ks]
ax4.errorbar(ks, gap_vals, yerr=sdk_vals, fmt='o-',
             color='#27ae60', linewidth=2, markersize=7,
             ecolor='grey', elinewidth=1.5, capsize=4)
if k_gap:
    ax4.axvline(k_gap, color=cor_k4, linestyle='--', lw=1.8,
                label=f'k óptimo = {k_gap}')
ax4.set_title('Gap Statistic ± sdk', fontweight='bold')
ax4.set_xlabel('k'); ax4.set_ylabel('Gap')
ax4.set_xticks(ks); ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# --- 5. Dunn Index ---
ax5 = fig.add_subplot(gs[1, 1])
dunns = [dunn_resultados[3], dunn_resultados[4]]
cores = [cor_k3, cor_k4]
bars  = ax5.bar(['k=3', 'k=4'], dunns, color=cores,
                edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, dunns):
    ax5.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.002,
             f'{v:.4f}', ha='center', va='bottom', fontsize=10)
ax5.set_title('Dunn Index', fontweight='bold')
ax5.set_ylabel('Índice'); ax5.grid(axis='y', alpha=0.3)

# --- 6. Radar — síntese normalizada ---
ax6 = fig.add_subplot(gs[1, 2], polar=True)

metricas_radar = ['Silhouette', 'Dunn', 'ARI_mean', 'NMI_mean']
vals_k3 = [sil_k3, dunn_resultados[3],
           resultados[3]['ari_mean'], resultados[3]['nmi_mean']]
vals_k4 = [sil_k4, dunn_resultados[4],
           resultados[4]['ari_mean'], resultados[4]['nmi_mean']]

# Normalizar 0-1 por métrica
combined = list(zip(vals_k3, vals_k4))
norm_k3, norm_k4 = [], []
for v3, v4 in combined:
    mx = max(v3, v4)
    mn = min(v3, v4)
    rng_v = mx - mn if mx != mn else 1
    norm_k3.append((v3 - mn) / rng_v)
    norm_k4.append((v4 - mn) / rng_v)

angles = np.linspace(0, 2*np.pi, len(metricas_radar),
                     endpoint=False).tolist()
norm_k3 += norm_k3[:1]; norm_k4 += norm_k4[:1]
angles  += angles[:1]

ax6.plot(angles, norm_k3, 'o-', color=cor_k3,
         linewidth=2, label='k=3')
ax6.fill(angles, norm_k3, color=cor_k3, alpha=0.2)
ax6.plot(angles, norm_k4, 's-', color=cor_k4,
         linewidth=2, label='k=4')
ax6.fill(angles, norm_k4, color=cor_k4, alpha=0.2)
ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(metricas_radar, fontsize=9)
ax6.set_title('Radar — Síntese Normalizada', fontweight='bold', pad=15)
ax6.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.suptitle("Validação Avançada — k=3 vs k=4\n"
             "ARI · NMI · Dunn Index · Gap Statistic",
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('validacao_avancada_k3_k4.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE — Critérios para escolha de k=4")
print("=" * 70)
print(f"""
  Condição 1 — Estabilidade de labels (ARI_min ≥ 0.90):
    k=3: ARI_min = {resultados[3]['ari_min']:.4f}  {'✔' if resultados[3]['ari_min'] >= 0.90 else '✗'}
    k=4: ARI_min = {resultados[4]['ari_min']:.4f}  {'✔' if resultados[4]['ari_min'] >= 0.90 else '✗'}

  Condição 2 — Cluster C0 de k=4 operacionalmente distinto:
    Capacidade pagamento C0 : R$ 411.820  (vs R$ 22.475 em C1)
    Pagamentos 2025 C0      : R$ 41.373   (vs R$ 500 em C1)
    Receita declarada C0    : R$ 1.120.976 (vs R$ 177.640 em C1)
    → Diferença de capacidade: 18x — estratégia de cobrança distinta ✔

  Decisão:
  → Se ARI_min k=4 ≥ 0.90: k=4 validado em estabilidade e operacionalidade
  → Se ARI_min k=4 < 0.90: reportar instabilidade e justificar k=4
    apenas pelo critério operacional com ressalva metodológica
""")

### Mapeamento Final - Nomenclatura dos Clusters k=4

In [ ]:
# --- 5. Flags de missing — CORRIGIDO ---
ax_flags = fig.add_subplot(gs[2, 2:])

vars_flags_plot = {
    'is_missing_ap_declared_gross_revenue_value':  'No Revenue Data',
    'is_missing_ap_debt_to_revenue_ratio':         'No D/R Ratio',
    'is_missing_pc_pending_filing_omissions_ratio':'No Filing Data',
}

# Filtrar apenas variáveis presentes — labels e vars em sincronia
vars_flags_ok_plot = {k: v for k, v in vars_flags_plot.items()
                      if k in df_orig_k4.columns}

x2          = np.arange(len(vars_flags_ok_plot))
labels_flags = list(vars_flags_ok_plot.values())
width        = 0.18

for i, c in enumerate(range(4)):
    mask = df_orig_k4['cluster'] == c
    vals = [df_orig_k4.loc[mask, v].mean() * 100
            for v in vars_flags_ok_plot.keys()]   # sem filtro — já filtrado acima
    ax_flags.bar(x2 + i * width - 1.5 * width, vals,
                 width, label=cluster_short[c],
                 color=cores[c], alpha=0.85, edgecolor='white')

ax_flags.set_xticks(x2)
ax_flags.set_xticklabels(labels_flags, fontsize=9)
ax_flags.set_ylabel('% devedores')
ax_flags.set_title('Flags de Dados em Falta por Cluster (%)',
                   fontweight='bold')
ax_flags.legend(fontsize=8)
ax_flags.grid(axis='y', alpha=0.3)
ax_flags.set_ylim(0, 115)

In [ ]:
# =============================================================================
# MAPEAMENTO FINAL — Nomenclatura dos Clusters k=4
# =============================================================================

cluster_names = {
    0: 'Active High-Capacity Debtors',
    1: 'Small Low-Engagement Debtors',
    2: 'Legacy Debt Debtors',
    3: 'Financially Stressed Active Debtors',
}

cluster_short = {
    0: 'AHCD',
    1: 'SLED',
    2: 'LDD',
    3: 'FSAD',
}

cluster_colors = {
    0: '#2980b9',   # azul
    1: '#27ae60',   # verde
    2: '#e74c3c',   # vermelho
    3: '#f39c12',   # laranja
}

# Aplicar ao dataframe final
df_orig_k4['cluster_id']    = df_orig_k4['cluster']
df_orig_k4['cluster_name']  = df_orig_k4['cluster'].map(cluster_names)
df_orig_k4['cluster_short'] = df_orig_k4['cluster'].map(cluster_short)

# Verificação
print("=" * 70)
print("NOMENCLATURA FINAL — k=4")
print("=" * 70)
print(f"\n  {'ID':<6} {'Sigla':<8} {'Nome':<40} {'N':>8} {'%':>7}")
print("  " + "─" * 72)

for c in range(4):
    n   = (df_orig_k4['cluster'] == c).sum()
    pct = n / len(df_orig_k4) * 100
    print(f"  C{c:<5} {cluster_short[c]:<8} {cluster_names[c]:<40} "
          f"{n:>8,} {pct:>6.1f}%")

print(f"\n  Total: {len(df_orig_k4):,} devedores")


# =============================================================================
# VISUALIZAÇÃO FINAL — Perfil dos 4 clusters com nomes
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.40)

nomes_eixo = [f"C{c}\n{cluster_short[c]}" for c in range(4)]
cores      = [cluster_colors[c] for c in range(4)]


# --- 1. Distribuição de tamanhos ---
ax1 = fig.add_subplot(gs[0, :2])
ns  = [(df_orig_k4['cluster'] == c).sum() for c in range(4)]
bars = ax1.barh(nomes_eixo, ns, color=cores, edgecolor='white', height=0.6)
for bar, n in zip(bars, ns):
    ax1.text(bar.get_width() + 1000,
             bar.get_y() + bar.get_height()/2,
             f'{n:,}  ({n/len(df_orig_k4)*100:.1f}%)',
             va='center', fontsize=9)
ax1.set_title('Dimensão dos Clusters', fontweight='bold')
ax1.set_xlabel('N devedores')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()


# --- 2. Radar de perfil operacional ---
ax2 = fig.add_subplot(gs[0, 2:], polar=True)

# Variáveis para radar — medianas normalizadas 0-1
vars_radar = {
    'da_total_debt_value':         'Dívida Total',
    'ap_payment_capacity_value':   'Cap. Pagamento',
    'pc_total_payment_2025_value': 'Pagamentos 2025',
    'da_age_weighted_debt_value_months': 'Idade Dívida',
    'ap_debt_to_revenue_ratio':    'Dívida/Receita',
    'pc_payment_regularity_ratio': 'Regularidade',
}

medianas = {}
for c in range(4):
    mask = df_orig_k4['cluster'] == c
    medianas[c] = [df_orig_k4.loc[mask, v].median()
                   for v in vars_radar.keys()]

# Normalizar 0-1 por variável
arr = np.array([medianas[c] for c in range(4)])
arr_min = arr.min(axis=0)
arr_max = arr.max(axis=0)
arr_norm = (arr - arr_min) / (arr_max - arr_min + 1e-10)

labels_radar = list(vars_radar.values())
angles = np.linspace(0, 2*np.pi, len(labels_radar),
                     endpoint=False).tolist()
angles += angles[:1]

for c in range(4):
    vals = arr_norm[c].tolist() + [arr_norm[c][0]]
    ax2.plot(angles, vals, 'o-', color=cores[c],
             linewidth=2, label=cluster_short[c])
    ax2.fill(angles, vals, color=cores[c], alpha=0.08)

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(labels_radar, fontsize=8)
ax2.set_title('Radar — Perfil Operacional\n(medianas normalizadas)',
              fontweight='bold', pad=18)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)


# --- 3. Boxplots operacionais ---
vars_box = [
    ('da_total_debt_value',        'Dívida Total — log1p (R$)'),
    ('ap_payment_capacity_value',  'Capacidade Pagamento — log1p (R$)'),
    ('da_age_weighted_debt_value_months', 'Idade Pond. Dívida (meses)'),
    ('ap_debt_to_revenue_ratio',   'Dívida / Receita'),
]

for idx, (var, titulo) in enumerate(vars_box):
    ax = fig.add_subplot(gs[1, idx])
    if var not in df_orig_k4.columns:
        continue

    dados = []
    for c in range(4):
        s = df_orig_k4.loc[df_orig_k4['cluster'] == c, var].dropna()
        if var in ['da_total_debt_value', 'ap_payment_capacity_value',
                   'pc_total_payment_2025_value']:
            s = np.log1p(s)
        dados.append(s)

    bp = ax.boxplot(dados, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    for patch, cor in zip(bp['boxes'], cores):
        patch.set_facecolor(cor); patch.set_alpha(0.8)

    ax.set_xticklabels(nomes_eixo, fontsize=7)
    ax.set_title(titulo, fontweight='bold', fontsize=9)
    ax.grid(axis='y', alpha=0.3)


# --- 4. Indicadores binários (barras agrupadas) ---
ax_bin = fig.add_subplot(gs[2, :2])

vars_bin_plot = {
    'da_prescription_risk_bin':       'Prescription Risk',
    'da_new_debt_incidence_bin':       'New Debt Incidence',
    'pc_recent_payment_indicator_bin': 'Recent Payment',
}

# Filtrar apenas variáveis presentes — x e labels em sincronia
vars_bin_ok = {k: v for k, v in vars_bin_plot.items()
               if k in df_orig_k4.columns}

x          = np.arange(len(vars_bin_ok))
width      = 0.18
labels_bin = list(vars_bin_ok.values())

for i, c in enumerate(range(4)):
    mask = df_orig_k4['cluster'] == c
    vals = [df_orig_k4.loc[mask, v].mean() * 100
            for v in vars_bin_ok.keys()]
    ax_bin.bar(x + i * width - 1.5 * width, vals,
               width, label=cluster_short[c],
               color=cores[c], alpha=0.85, edgecolor='white')

ax_bin.set_xticks(x)
ax_bin.set_xticklabels(labels_bin, fontsize=9)
ax_bin.set_ylabel('% devedores')
ax_bin.set_title('Indicadores Descritivos por Cluster (%)',
                 fontweight='bold')
ax_bin.legend(fontsize=8)
ax_bin.grid(axis='y', alpha=0.3)
ax_bin.set_ylim(0, 115)


# --- 5. Flags de missing ---
ax_flags = fig.add_subplot(gs[2, 2:])

vars_flags_plot = {
    'is_missing_ap_declared_gross_revenue_value':  'No Revenue Data',
    'is_missing_ap_debt_to_revenue_ratio':         'No D/R Ratio',
    'is_missing_pc_pending_filing_omissions_ratio':'No Filing Data',
}

# Filtrar apenas variáveis presentes — x e labels em sincronia
vars_flags_ok = {k: v for k, v in vars_flags_plot.items()
                 if k in df_orig_k4.columns}

x2           = np.arange(len(vars_flags_ok))
labels_flags = list(vars_flags_ok.values())

for i, c in enumerate(range(4)):
    mask = df_orig_k4['cluster'] == c
    vals = [df_orig_k4.loc[mask, v].mean() * 100
            for v in vars_flags_ok.keys()]
    ax_flags.bar(x2 + i * width - 1.5 * width, vals,
                 width, label=cluster_short[c],
                 color=cores[c], alpha=0.85, edgecolor='white')

ax_flags.set_xticks(x2)
ax_flags.set_xticklabels(labels_flags, fontsize=9)
ax_flags.set_ylabel('% devedores')
ax_flags.set_title('Flags de Dados em Falta por Cluster (%)',
                   fontweight='bold')
ax_flags.legend(fontsize=8)
ax_flags.grid(axis='y', alpha=0.3)
ax_flags.set_ylim(0, 115)

plt.suptitle(
    "Segmentação Final — K-Means k=4\n"
    "Active High-Capacity  ·  Small Low-Engagement  ·  "
    "Legacy Debt  ·  Financially Stressed Active",
    fontsize=12, fontweight='bold', y=1.01
)
plt.savefig('clusters_k4_nomenclatura_final.png', dpi=150,
            bbox_inches='tight')
plt.show()


# =============================================================================
# SALVAR MODELOS E LABELS — artefactos para deployment
# =============================================================================

import pickle, os

output_dir = 'artefactos_clustering'
os.makedirs(output_dir, exist_ok=True)

# Modelo K-Means k=4
with open(f'{output_dir}/kmeans_k4.pkl', 'wb') as f:
    pickle.dump(km4, f)

# Mapeamento de nomes
with open(f'{output_dir}/cluster_names.pkl', 'wb') as f:
    pickle.dump(cluster_names, f)

# Labels com id_cnpj
df_labels = df_orig_k4[['id_cnpj',
                          'cluster_id',
                          'cluster_name',
                          'cluster_short']].copy()
df_labels.to_csv(f'{output_dir}/labels_k4.csv', index=False)

# Centróides (escala normalizada — para deployment)
centroides = pd.DataFrame(
    km4.cluster_centers_,
    columns=var_clustering_final
)
centroides['cluster_id']   = range(4)
centroides['cluster_name'] = [cluster_names[c] for c in range(4)]
centroides.to_csv(f'{output_dir}/centroides_k4.csv', index=False)

print("\n" + "=" * 70)
print("ARTEFACTOS GUARDADOS")
print("=" * 70)
print(f"""
  {output_dir}/
  ├── kmeans_k4.pkl           ← modelo treinado (KMeans object)
  ├── cluster_names.pkl       ← dicionário id → nome
  ├── labels_k4.csv           ← {len(df_labels):,} registos com id_cnpj + cluster
  └── centroides_k4.csv       ← 4 centróides × 22 variáveis

  Colunas disponíveis em df_orig_k4:
    cluster_id    : 0–3  (numérico)
    cluster_name  : nome completo
    cluster_short : sigla (AHCD / SLED / LDD / FSAD)

  Próximos passos:
  → Parágrafo comparativo k=3 vs k=4 para a dissertação
  → Comparação algorítmica K-means vs GMM vs Hierarchical
  → Pipeline de classificação de novos devedores
""")

### Comparação dos algoritmos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import warnings

from sklearn.cluster         import KMeans, AgglomerativeClustering
from sklearn.mixture         import GaussianMixture
from sklearn.metrics         import (silhouette_score, davies_bouldin_score,
                                     calinski_harabasz_score,
                                     adjusted_rand_score,
                                     normalized_mutual_info_score)
from sklearn.decomposition   import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
K_FINAL       = 4
SAMPLE_SIL    = 30_000
SAMPLE_HIER   = 15_000   # Hierarchical completo é O(n²) — amostra estratificada

X = df_clustering[var_clustering_final].values

rng        = np.random.default_rng(RANDOM_STATE)
idx_sil    = rng.choice(X.shape[0], size=SAMPLE_SIL, replace=False)

print("=" * 70)
print(f"COMPARAÇÃO ALGORÍTMICA — k={K_FINAL}")
print("K-Means  ·  GMM (full covariance)  ·  Hierarchical (Ward)")
print("=" * 70)
print(f"\n  Dataset  : {X.shape[0]:,} obs × {X.shape[1]} variáveis")
print(f"  k final  : {K_FINAL}")


# =============================================================================
# BLOCO 1 — K-MEANS (referência — já treinado)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 1 — K-Means  (solução de referência)")
print("─" * 70)

# Reutilizar km4 e labels_k4 já existentes em memória
labels_kmeans = labels_k4.copy()

sil_km = silhouette_score(X[idx_sil], labels_kmeans[idx_sil])
db_km  = davies_bouldin_score(X, labels_kmeans)
ch_km  = calinski_harabasz_score(X, labels_kmeans)

print(f"\n  Silhouette       : {sil_km:.4f}")
print(f"  Davies-Bouldin   : {db_km:.4f}")
print(f"  Calinski-Harabasz: {ch_km:,.1f}")
print(f"\n  Distribuição:")
for c in range(K_FINAL):
    n   = (labels_kmeans == c).sum()
    print(f"    {cluster_names[c]:<42}: {n:>7,}  ({n/len(labels_kmeans)*100:.1f}%)")


# =============================================================================
# BLOCO 2 — GAUSSIAN MIXTURE MODEL (full covariance)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Gaussian Mixture Model  (full covariance)")
print("─" * 70)

t0 = time.time()

gmm = GaussianMixture(
    n_components    = K_FINAL,
    covariance_type = 'full',
    n_init          = 5,
    max_iter        = 200,
    random_state    = RANDOM_STATE,
    verbose         = 0,
)
gmm.fit(X)
labels_gmm  = gmm.predict(X)
probs_gmm   = gmm.predict_proba(X)
conf_gmm    = probs_gmm.max(axis=1)   # probabilidade de pertencimento ao cluster atribuído

print(f"\n  Tempo de treino  : {time.time()-t0:.1f}s")
print(f"  BIC              : {gmm.bic(X):,.2f}")
print(f"  AIC              : {gmm.aic(X):,.2f}")
print(f"  Log-likelihood   : {gmm.score(X)*len(X):,.2f}")
print(f"  Convergiu        : {'✔  Sim' if gmm.converged_ else '✗  Não'}")
print(f"  Nº iterações     : {gmm.n_iter_}")

sil_gmm = silhouette_score(X[idx_sil], labels_gmm[idx_sil])
db_gmm  = davies_bouldin_score(X, labels_gmm)
ch_gmm  = calinski_harabasz_score(X, labels_gmm)

print(f"\n  Silhouette       : {sil_gmm:.4f}")
print(f"  Davies-Bouldin   : {db_gmm:.4f}")
print(f"  Calinski-Harabasz: {ch_gmm:,.1f}")

print(f"\n  Confiança de atribuição (probabilidade máxima):")
print(f"    Mediana : {conf_gmm.median() if hasattr(conf_gmm,'median') else np.median(conf_gmm):.4f}")
print(f"    % > 0.90: {(conf_gmm > 0.90).mean()*100:.1f}%")
print(f"    % > 0.70: {(conf_gmm > 0.70).mean()*100:.1f}%")
print(f"    % < 0.50: {(conf_gmm < 0.50).mean()*100:.1f}%  (atribuição ambígua)")

print(f"\n  Distribuição:")
for c in range(K_FINAL):
    n = (labels_gmm == c).sum()
    print(f"    Componente {c}: {n:>7,}  ({n/len(labels_gmm)*100:.1f}%)")


# =============================================================================
# BLOCO 3 — HIERARCHICAL WARD — amostra estratificada + propagação
# =============================================================================

print("\n" + "─" * 70)
print(f"BLOCO 3 — Hierarchical Ward  (amostra estratificada {SAMPLE_HIER:,})")
print("─" * 70)

# --- 3.1 Amostra estratificada por cluster K-Means ---
idx_hier     = []
per_cluster  = SAMPLE_HIER // K_FINAL

for c in range(K_FINAL):
    idx_c = np.where(labels_kmeans == c)[0]
    n_s   = min(per_cluster, len(idx_c))
    idx_hier.extend(rng.choice(idx_c, size=n_s, replace=False).tolist())

idx_hier = np.array(idx_hier)
X_hier   = X[idx_hier]

print(f"\n  Amostra estratificada:")
for c in range(K_FINAL):
    n_c = (labels_kmeans[idx_hier] == c).sum()
    print(f"    C{c}: {n_c:,} obs  ({n_c/len(idx_hier)*100:.1f}%)")

# --- 3.2 Linkage e dendrograma ---
print(f"\n  Calculando linkage matrix (Ward)...")
t0 = time.time()
Z  = linkage(X_hier, method='ward', metric='euclidean')
print(f"  Concluído em {time.time()-t0:.1f}s")

# Cortar dendrograma em k=4
labels_hier_sample = fcluster(Z, K_FINAL, criterion='maxclust') - 1  # 0-indexed

print(f"\n  Distribuição na amostra (corte k={K_FINAL}):")
for c in range(K_FINAL):
    n = (labels_hier_sample == c).sum()
    print(f"    Cluster {c}: {n:>6,}  ({n/len(labels_hier_sample)*100:.1f}%)")

# --- 3.3 Propagação para o dataset completo ---
# Calcular centróides dos clusters hierárquicos na amostra
# Propagar por distância mínima ao centróide (nearest centroid)
print(f"\n  Propagando labels para {len(X):,} devedores via nearest centroid...")
t0 = time.time()

centroides_hier = np.array([
    X_hier[labels_hier_sample == c].mean(axis=0)
    for c in range(K_FINAL)
])

# Distância de cada devedor aos 4 centróides hierárquicos
# Processado em batches para eficiência de memória
BATCH = 50_000
labels_hier_full = np.empty(len(X), dtype=int)

for start in range(0, len(X), BATCH):
    end       = min(start + BATCH, len(X))
    dists     = euclidean_distances(X[start:end], centroides_hier)
    labels_hier_full[start:end] = dists.argmin(axis=1)

print(f"  Concluído em {time.time()-t0:.1f}s")

labels_hier = labels_hier_full

# Métricas no dataset completo
sil_hier = silhouette_score(X[idx_sil], labels_hier[idx_sil])
db_hier  = davies_bouldin_score(X, labels_hier)
ch_hier  = calinski_harabasz_score(X, labels_hier)

print(f"\n  Métricas (dataset completo — labels propagados):")
print(f"    Silhouette        : {sil_hier:.4f}")
print(f"    Davies-Bouldin    : {db_hier:.4f}")
print(f"    Calinski-Harabasz : {ch_hier:,.1f}")

print(f"\n  Distribuição (dataset completo):")
for c in range(K_FINAL):
    n = (labels_hier == c).sum()
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_hier)*100:.1f}%)")


# =============================================================================
# BLOCO 4 — CONCORDÂNCIA ENTRE ALGORITMOS (ARI + NMI)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Concordância entre Algoritmos (ARI / NMI)")
print("─" * 70)

pares = [
    ('K-Means',      'GMM',          labels_kmeans, labels_gmm),
    ('K-Means',      'Hierarchical', labels_kmeans, labels_hier),
    ('GMM',          'Hierarchical', labels_gmm,    labels_hier),
]

print(f"\n  {'Par':<35} {'ARI':>8}  {'NMI':>8}  Interpretação")
print("  " + "─" * 65)

concordancias = {}
for nome_a, nome_b, lab_a, lab_b in pares:
    ari = adjusted_rand_score(lab_a, lab_b)
    nmi = normalized_mutual_info_score(lab_a, lab_b,
                                       average_method='arithmetic')
    concordancias[f"{nome_a} × {nome_b}"] = {'ari': ari, 'nmi': nmi}

    if ari >= 0.90:
        interp = "✔  Concordância forte"
    elif ari >= 0.70:
        interp = "~  Concordância moderada"
    else:
        interp = "⚠  Concordância fraca"

    print(f"  {nome_a+' × '+nome_b:<35} {ari:>8.4f}  {nmi:>8.4f}  {interp}")


# =============================================================================
# BLOCO 5 — TABELA SÍNTESE COMPARATIVA
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 5 — Síntese Comparativa")
print("─" * 70)

print(f"""
  {'Métrica':<28} {'K-Means':>12} {'GMM':>12} {'Hierarchical':>14}
  {'─'*68}
  {'Silhouette (↑)':<28} {sil_km:>12.4f} {sil_gmm:>12.4f} {sil_hier:>14.4f}
  {'Davies-Bouldin (↓)':<28} {db_km:>12.4f} {db_gmm:>12.4f} {db_hier:>14.4f}
  {'Calinski-Harabasz (↑)':<28} {ch_km:>12,.0f} {ch_gmm:>12,.0f} {ch_hier:>14,.0f}
  {'─'*68}
  {'ARI vs K-Means':<28} {'referência':>12} {concordancias['K-Means × GMM']['ari']:>12.4f} {concordancias['K-Means × Hierarchical']['ari']:>14.4f}
  {'NMI vs K-Means':<28} {'referência':>12} {concordancias['K-Means × GMM']['nmi']:>12.4f} {concordancias['K-Means × Hierarchical']['nmi']:>14.4f}
""")


# =============================================================================
# BLOCO 6 — VISUALIZAÇÕES
# =============================================================================

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

algoritmos    = ['K-Means', 'GMM', 'Hierarchical']
labels_list   = [labels_kmeans, labels_gmm, labels_hier]
cores_algo    = ['#2980b9', '#8e44ad', '#27ae60']

# --- 1. Distribuição de tamanhos por algoritmo ---
ax1 = fig.add_subplot(gs[0, :])
x      = np.arange(K_FINAL)
width  = 0.25

for i, (algo, labels, cor) in enumerate(zip(algoritmos, labels_list, cores_algo)):
    ns = [(labels == c).sum() for c in range(K_FINAL)]
    bars = ax1.bar(x + (i-1)*width, ns, width,
                   label=algo, color=cor, alpha=0.85, edgecolor='white')

ax1.set_xticks(x)
ax1.set_xticklabels([f'C{c}\n{cluster_short[c]}' for c in range(K_FINAL)],
                    fontsize=9)
ax1.set_title('Distribuição de Devedores por Cluster e Algoritmo',
              fontweight='bold')
ax1.set_ylabel('N devedores')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)


# --- 2. Heatmap de ARI entre pares ---
ax2 = fig.add_subplot(gs[1, 0])
ari_matrix = np.ones((3, 3))
ari_matrix[0, 1] = ari_matrix[1, 0] = concordancias['K-Means × GMM']['ari']
ari_matrix[0, 2] = ari_matrix[2, 0] = concordancias['K-Means × Hierarchical']['ari']
ari_matrix[1, 2] = ari_matrix[2, 1] = concordancias['GMM × Hierarchical']['ari']

sns.heatmap(ari_matrix, annot=True, fmt='.4f',
            xticklabels=algoritmos, yticklabels=algoritmos,
            cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, ax=ax2,
            annot_kws={'size': 10})
ax2.set_title('ARI — Concordância entre Algoritmos',
              fontweight='bold', fontsize=10)
ax2.tick_params(axis='x', rotation=15)


# --- 3. Heatmap de NMI entre pares ---
ax3 = fig.add_subplot(gs[1, 1])
nmi_matrix = np.ones((3, 3))
nmi_matrix[0, 1] = nmi_matrix[1, 0] = concordancias['K-Means × GMM']['nmi']
nmi_matrix[0, 2] = nmi_matrix[2, 0] = concordancias['K-Means × Hierarchical']['nmi']
nmi_matrix[1, 2] = nmi_matrix[2, 1] = concordancias['GMM × Hierarchical']['nmi']

sns.heatmap(nmi_matrix, annot=True, fmt='.4f',
            xticklabels=algoritmos, yticklabels=algoritmos,
            cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.5, ax=ax3,
            annot_kws={'size': 10})
ax3.set_title('NMI — Concordância entre Algoritmos',
              fontweight='bold', fontsize=10)
ax3.tick_params(axis='x', rotation=15)


# --- 4. Métricas internas — barras comparativas ---
ax4 = fig.add_subplot(gs[1, 2])
metricas_nomes  = ['Silhouette', 'DB (inv.)', 'CH (norm.)']
vals_km  = [sil_km,  1/(1+db_km),  ch_km/(ch_km+ch_gmm+ch_hier)]
vals_gmm = [sil_gmm, 1/(1+db_gmm), ch_gmm/(ch_km+ch_gmm+ch_hier)]
vals_hie = [sil_hier,1/(1+db_hier),ch_hier/(ch_km+ch_gmm+ch_hier)]

xm = np.arange(len(metricas_nomes))
ax4.bar(xm - width, vals_km,  width, label='K-Means',      color=cores_algo[0], alpha=0.85)
ax4.bar(xm,         vals_gmm, width, label='GMM',           color=cores_algo[1], alpha=0.85)
ax4.bar(xm + width, vals_hie, width, label='Hierarchical',  color=cores_algo[2], alpha=0.85)
ax4.set_xticks(xm)
ax4.set_xticklabels(metricas_nomes, fontsize=9)
ax4.set_title('Métricas Internas (normalizadas)',
              fontweight='bold', fontsize=10)
ax4.legend(fontsize=8); ax4.grid(axis='y', alpha=0.3)


# --- 5. PCA 2D — visualização dos três algoritmos ---
print("\n  Calculando PCA 2D para visualização...")

pca      = PCA(n_components=2, random_state=RANDOM_STATE)
idx_pca  = rng.choice(X.shape[0], size=8_000, replace=False)
X_pca    = pca.fit_transform(X[idx_pca])

var_exp  = pca.explained_variance_ratio_ * 100

for i, (algo, labels, cor_base) in enumerate(
        zip(algoritmos, labels_list, cores_algo)):

    ax = fig.add_subplot(gs[2, i])
    cores_pca = plt.cm.tab10(np.linspace(0, 0.4, K_FINAL))

    for c in range(K_FINAL):
        mask = labels[idx_pca] == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   s=2, alpha=0.4,
                   color=cores_pca[c],
                   label=f'C{c}')

    ax.set_title(f'{algo} — PCA 2D', fontweight='bold', fontsize=10)
    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', fontsize=8)
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', fontsize=8)
    ax.legend(fontsize=7, markerscale=4)
    ax.grid(alpha=0.2)


plt.suptitle(
    f"Comparação Algorítmica — K-Means · GMM · Hierarchical  |  k={K_FINAL}\n"
    f"377.419 devedores | RFB 9ª Região Fiscal | 22 variáveis",
    fontsize=12, fontweight='bold', y=1.01
)
plt.savefig('comparacao_algoritmica.png', dpi=150, bbox_inches='tight')
plt.show()


# --- 6. Dendrograma (amostra estratificada) ---
fig2, ax_dend = plt.subplots(figsize=(14, 6))

dendrogram(
    Z,
    truncate_mode   = 'lastp',
    p               = 20,
    leaf_rotation   = 45,
    leaf_font_size  = 8,
    show_leaf_counts= True,
    ax              = ax_dend,
    color_threshold = Z[-(K_FINAL-1), 2],
)
ax_dend.axhline(
    y     = Z[-(K_FINAL-1), 2],
    color = '#e74c3c',
    linestyle = '--',
    lw    = 1.8,
    label = f'Corte k={K_FINAL}'
)
ax_dend.set_title(
    f'Dendrograma — Hierarchical Clustering (Ward)  |  '
    f'Amostra estratificada {len(X_hier):,} obs',
    fontweight='bold'
)
ax_dend.set_xlabel('Observações (nº no parêntesis)')
ax_dend.set_ylabel('Distância de Ward')
ax_dend.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dendrograma_hierarchical.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE — Comparação Algorítmica")
print("=" * 70)

ari_km_gmm  = concordancias['K-Means × GMM']['ari']
ari_km_hier = concordancias['K-Means × Hierarchical']['ari']
ari_gmm_hier = concordancias['GMM × Hierarchical']['ari']
ari_min_todos = min(ari_km_gmm, ari_km_hier, ari_gmm_hier)

if ari_min_todos >= 0.90:
    veredicto = ("✔  CONCORDÂNCIA FORTE entre os três algoritmos.\n"
                 "   Os segmentos reflectem estrutura genuína dos dados,\n"
                 "   independente do método de clustering utilizado.")
elif ari_min_todos >= 0.70:
    veredicto = ("~  CONCORDÂNCIA MODERADA. Os segmentos principais\n"
                 "   são consistentes mas com diferenças nos clusters menores.")
else:
    veredicto = ("⚠  CONCORDÂNCIA FRACA. Reportar limitação na dissertação.")

print(f"""
  ARI K-Means × GMM          : {ari_km_gmm:.4f}
  ARI K-Means × Hierarchical : {ari_km_hier:.4f}
  ARI GMM × Hierarchical     : {ari_gmm_hier:.4f}
  ARI mínimo (pior par)      : {ari_min_todos:.4f}

  {veredicto}

  Algoritmo seleccionado para deployment: K-Means
  Justificativa: interpretabilidade dos centróides, escalabilidade
  para classificação de novos devedores, e desempenho competitivo
  nas métricas internas.

  Ficheiros guardados:
  → comparacao_algoritmica.png
  → dendrograma_hierarchical.png
""")

## Visualização 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
rng          = np.random.default_rng(RANDOM_STATE)

# Paleta e identidade visual dos clusters — consistente com sessões anteriores
CORES = {
    0: '#2980b9',   # AHCD — azul
    1: '#27ae60',   # SLED — verde
    2: '#e74c3c',   # LDD  — vermelho
    3: '#f39c12',   # FSAD — laranja
}
SIGLAS = {c: cluster_short[c]  for c in range(4)}
NOMES  = {c: cluster_names[c]  for c in range(4)}

# Etiquetas para eixos — versão curta para gráficos
LABEL_CURTO = {
    0: 'AHCD\nActive High-Capacity',
    1: 'SLED\nSmall Low-Engagement',
    2: 'LDD\nLegacy Debt',
    3: 'FSAD\nFinancially Stressed',
}

print("=" * 70)
print("VISUALIZAÇÕES FINAIS — Capítulo de Resultados")
print("K-Means k=4 | 377.419 devedores | RFB 9ª Região Fiscal")
print("=" * 70)


# =============================================================================
# FIGURA 1 — PCA 2D  (publicação)
# =============================================================================

print("\n  [1/4] PCA 2D...")

X       = df_clustering[var_clustering_final].values
labels  = labels_k4.copy()

# PCA no dataset completo — projectar amostra para visualização
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca.fit(X)
var_exp = pca.explained_variance_ratio_ * 100

# Amostra estratificada 5.000 por cluster — 20.000 total
idx_plot = []
for c in range(4):
    idx_c = np.where(labels == c)[0]
    n_s   = min(5_000, len(idx_c))
    idx_plot.extend(rng.choice(idx_c, size=n_s, replace=False).tolist())
idx_plot = np.array(idx_plot)

X_2d  = pca.transform(X[idx_plot])
lab_2d = labels[idx_plot]

# Centróides projectados
centroides_2d = pca.transform(km4.cluster_centers_)

fig1, ax = plt.subplots(figsize=(10, 8))

for c in range(4):
    mask = lab_2d == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=3, alpha=0.35, color=CORES[c],
               label=NOMES[c], rasterized=True)

# Centróides
for c in range(4):
    ax.scatter(centroides_2d[c, 0], centroides_2d[c, 1],
               s=280, marker='*', color=CORES[c],
               edgecolors='white', linewidths=0.8, zorder=5)
    ax.annotate(SIGLAS[c],
                xy=(centroides_2d[c, 0], centroides_2d[c, 1]),
                xytext=(8, 8), textcoords='offset points',
                fontsize=9, fontweight='bold', color=CORES[c])

ax.set_xlabel(f'Principal Component 1  ({var_exp[0]:.1f}% variance explained)',
              fontsize=11)
ax.set_ylabel(f'Principal Component 2  ({var_exp[1]:.1f}% variance explained)',
              fontsize=11)
ax.set_title('Two-Dimensional PCA Projection of Debtor Clusters\n'
             f'K-Means k=4 | n = {len(idx_plot):,} (stratified sample) | '
             f'Total variance explained: {sum(var_exp):.1f}%',
             fontsize=12, fontweight='bold')

handles = [mpatches.Patch(color=CORES[c], label=f'C{c} — {NOMES[c]}')
           for c in range(4)]
ax.legend(handles=handles, fontsize=9, loc='best',
          framealpha=0.9, edgecolor='#bdc3c7')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig_pca2d_clusters.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"    Variância explicada: PC1={var_exp[0]:.1f}%  PC2={var_exp[1]:.1f}%  "
      f"Total={sum(var_exp):.1f}%")


# =============================================================================
# FIGURA 2 — HEATMAP DE DESVIOS DA MEDIANA GLOBAL
# =============================================================================

print("\n  [2/4] Heatmap de desvios...")

# Variáveis operacionais em escala original — selecção para o capítulo
vars_heatmap = {
    'da_total_debt_value':                  'Total Debt Value',
    'da_debt_0_6_months_value':             'Recent Debt (0–6 months)',
    'da_debt_over_48_months_value':         'Aged Debt (>48 months)',
    'da_penalty_to_principal_ratio':        'Penalties / Principal Ratio',
    'da_age_weighted_debt_value_months':    'Weighted Debt Age (months)',
    'da_number_of_debts_count':             'Number of Debts',
    'ap_declared_gross_revedue_value':      'Declared Gross Revenue',
    'ap_payment_capacity_value':            'Payment Capacity',
    'ap_debt_to_revenue_ratio':             'Debt-to-Revenue Ratio',
    'pc_total_payment_2025_value':          'Payments Made (2025)',
    'pc_instalment_plan_count':             'Instalment Plans',
    'pc_payment_regularity_ratio':          'Payment Regularity',
    'pc_pending_filing_omissions_ratio':    'Filing Omissions Ratio',
    'tp_enterprise_age_years':              'Enterprise Age (years)',
    'ci_sectoral_delinquency_deviation_score': 'Sectoral Delinquency Deviation',
}

# Filtrar vars presentes
vars_hm_ok = {k: v for k, v in vars_heatmap.items() if k in df_orig_k4.columns}

# Calcular mediana global e por cluster
med_global = df_orig_k4[list(vars_hm_ok.keys())].median()

desvios = pd.DataFrame(index=[f'C{c}\n{SIGLAS[c]}' for c in range(4)],
                       columns=list(vars_hm_ok.values()),
                       dtype=float)

for c in range(4):
    mask    = df_orig_k4['cluster'] == c
    med_c   = df_orig_k4.loc[mask, list(vars_hm_ok.keys())].median()
    desvio  = ((med_c - med_global) / med_global.replace(0, np.nan)) * 100
    desvio.index = list(vars_hm_ok.values())
    desvios.loc[f'C{c}\n{SIGLAS[c]}'] = desvio

fig2, ax2 = plt.subplots(figsize=(14, 7))

sns.heatmap(
    desvios.T.astype(float),
    annot=True, fmt='.0f',
    cmap='RdBu_r', center=0,
    linewidths=0.4, linecolor='#f0f0f0',
    annot_kws={'size': 9},
    cbar_kws={'label': '% deviation from global median', 'shrink': 0.8},
    ax=ax2
)

# Colorir labels do eixo x pela cor do cluster
for i, tick in enumerate(ax2.get_xticklabels()):
    tick.set_color(CORES[i])
    tick.set_fontweight('bold')
    tick.set_fontsize(9)

ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0, fontsize=9)
ax2.set_title('Cluster Characterisation — Percentage Deviation from Global Median\n'
              'K-Means k=4 | 377,419 debtors | RFB 9th Fiscal Region',
              fontsize=12, fontweight='bold', pad=15)
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig('fig_heatmap_desvios.png', dpi=200, bbox_inches='tight')
plt.show()


# =============================================================================
# FIGURA 3 — RADAR DE PERFIS OPERACIONAIS
# =============================================================================

print("\n  [3/4] Radar de perfis...")

vars_radar = {
    'da_total_debt_value':           'Total\nDebt',
    'ap_payment_capacity_value':     'Payment\nCapacity',
    'pc_total_payment_2025_value':   'Payments\n2025',
    'da_age_weighted_debt_value_months': 'Debt\nAge',
    'ap_debt_to_revenue_ratio':      'Debt /\nRevenue',
    'pc_payment_regularity_ratio':   'Payment\nRegularity',
    'da_penalty_to_principal_ratio': 'Penalties /\nPrincipal',
    'tp_enterprise_age_years':       'Enterprise\nAge',
}
vars_radar_ok = {k: v for k, v in vars_radar.items() if k in df_orig_k4.columns}

# Medianas por cluster — normalizar 0-1 por variável
medianas = np.array([
    [df_orig_k4.loc[df_orig_k4['cluster']==c, v].median()
     for v in vars_radar_ok.keys()]
    for c in range(4)
])

v_min = medianas.min(axis=0)
v_max = medianas.max(axis=0)
norm  = (medianas - v_min) / (v_max - v_min + 1e-10)

labels_radar = list(vars_radar_ok.values())
N            = len(labels_radar)
angles       = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles      += angles[:1]

fig3, axes3 = plt.subplots(2, 2, figsize=(14, 12),
                            subplot_kw=dict(polar=True))
axes3 = axes3.flatten()

for c in range(4):
    ax   = axes3[c]
    vals = norm[c].tolist() + [norm[c][0]]

    ax.plot(angles, vals, 'o-', color=CORES[c], linewidth=2.5)
    ax.fill(angles, vals, color=CORES[c], alpha=0.20)

    # Grade de referência
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_radar, fontsize=8.5)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.0'], fontsize=6.5,
                       color='grey')
    ax.grid(color='grey', alpha=0.3)

    # Título com contagem
    n_c = (labels_k4 == c).sum()
    ax.set_title(f'C{c} — {NOMES[c]}\n'
                 f'n = {n_c:,}  ({n_c/len(labels_k4)*100:.1f}%)',
                 fontsize=10, fontweight='bold',
                 color=CORES[c], pad=18)

    # Anotar valores reais nos vértices
    for angle, raw_val, var in zip(angles[:-1], medianas[c], vars_radar_ok.keys()):
        v = raw_val
        if v >= 1_000_000:
            label = f'R${v/1e6:.1f}M'
        elif v >= 1_000:
            label = f'R${v/1e3:.0f}K'
        elif v >= 1:
            label = f'{v:.1f}'
        else:
            label = f'{v:.2f}'
        ax.annotate(label,
                    xy=(angle, norm[c][list(vars_radar_ok.keys()).index(var)]),
                    xytext=(0, 10), textcoords='offset points',
                    fontsize=7, ha='center', color=CORES[c])

fig3.suptitle('Operational Profile Radar — Four Debtor Segments\n'
              'K-Means k=4 | Normalised medians (0 = min, 1 = max across clusters)',
              fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig('fig_radar_perfis.png', dpi=200, bbox_inches='tight')
plt.show()


# =============================================================================
# FIGURA 4 — PAINEL SÍNTESE (publicação — figura única para dissertação)
# =============================================================================

print("\n  [4/4] Painel síntese...")

fig4 = plt.figure(figsize=(20, 22))
gs4  = gridspec.GridSpec(3, 4, figure=fig4, hspace=0.55, wspace=0.40)

# ── A. Distribuição de tamanhos ──────────────────────────────────────────────
ax_a = fig4.add_subplot(gs4[0, :2])
ns   = [(labels_k4 == c).sum() for c in range(4)]
bars = ax_a.barh([LABEL_CURTO[c] for c in range(4)],
                  ns, color=[CORES[c] for c in range(4)],
                  edgecolor='white', height=0.55)
for bar, n in zip(bars, ns):
    ax_a.text(bar.get_width() + 1_500,
              bar.get_y() + bar.get_height()/2,
              f'{n:,}  ({n/len(labels_k4)*100:.1f}%)',
              va='center', fontsize=9)
ax_a.set_title('Portfolio Composition', fontweight='bold', fontsize=11)
ax_a.set_xlabel('Number of debtors')
ax_a.invert_yaxis()
ax_a.grid(axis='x', alpha=0.3)
ax_a.set_xlim(0, max(ns)*1.22)


# ── B. PCA 2D mini ───────────────────────────────────────────────────────────
ax_b = fig4.add_subplot(gs4[0, 2:])
for c in range(4):
    mask = lab_2d == c
    ax_b.scatter(X_2d[mask, 0], X_2d[mask, 1],
                 s=2, alpha=0.30, color=CORES[c], rasterized=True)
for c in range(4):
    ax_b.scatter(centroides_2d[c, 0], centroides_2d[c, 1],
                 s=220, marker='*', color=CORES[c],
                 edgecolors='white', linewidths=0.8, zorder=5)
    ax_b.annotate(SIGLAS[c], xy=(centroides_2d[c, 0], centroides_2d[c, 1]),
                  xytext=(6, 6), textcoords='offset points',
                  fontsize=8.5, fontweight='bold', color=CORES[c])
ax_b.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)', fontsize=10)
ax_b.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)', fontsize=10)
ax_b.set_title('PCA 2D Projection', fontweight='bold', fontsize=11)
ax_b.grid(alpha=0.2)


# ── C. Heatmap de desvios (compacto) ─────────────────────────────────────────
ax_c = fig4.add_subplot(gs4[1, :])

vars_hm_compact = {
    'da_total_debt_value':               'Total Debt',
    'da_debt_0_6_months_value':          'Recent Debt',
    'da_debt_over_48_months_value':      'Aged Debt',
    'da_penalty_to_principal_ratio':     'Penalties Ratio',
    'da_age_weighted_debt_value_months': 'Debt Age',
    'ap_payment_capacity_value':         'Payment Capacity',
    'ap_debt_to_revenue_ratio':          'Debt/Revenue',
    'pc_total_payment_2025_value':       'Payments 2025',
    'pc_payment_regularity_ratio':       'Regularity',
    'tp_enterprise_age_years':           'Enterprise Age',
    'ci_sectoral_delinquency_deviation_score': 'Sectoral Deviation',
}
vars_hm_c_ok = {k: v for k, v in vars_hm_compact.items()
                if k in df_orig_k4.columns}

med_g2 = df_orig_k4[list(vars_hm_c_ok.keys())].median()
desv2  = pd.DataFrame(index=[f'C{c} {SIGLAS[c]}' for c in range(4)],
                      columns=list(vars_hm_c_ok.values()), dtype=float)
for c in range(4):
    mask  = df_orig_k4['cluster'] == c
    med_c = df_orig_k4.loc[mask, list(vars_hm_c_ok.keys())].median()
    d     = ((med_c - med_g2) / med_g2.replace(0, np.nan)) * 100
    d.index = list(vars_hm_c_ok.values())
    desv2.loc[f'C{c} {SIGLAS[c]}'] = d

sns.heatmap(desv2.T.astype(float),
            annot=True, fmt='.0f',
            cmap='RdBu_r', center=0,
            linewidths=0.4, linecolor='#f0f0f0',
            annot_kws={'size': 9},
            cbar_kws={'label': '% deviation from median', 'shrink': 0.7},
            ax=ax_c)

for i, tick in enumerate(ax_c.get_xticklabels()):
    tick.set_color(CORES[i])
    tick.set_fontweight('bold')
    tick.set_fontsize(10)

ax_c.set_yticklabels(ax_c.get_yticklabels(), rotation=0, fontsize=9)
ax_c.set_title('Cluster Characterisation — Deviation from Global Median (%)',
               fontweight='bold', fontsize=11)


# ── D. Boxplots operacionais ──────────────────────────────────────────────────
vars_box4 = [
    ('da_total_debt_value',       'Total Debt — log₁₀ (R$)'),
    ('ap_payment_capacity_value', 'Payment Capacity — log₁₀ (R$)'),
    ('ap_debt_to_revenue_ratio',  'Debt-to-Revenue Ratio'),
    ('pc_payment_regularity_ratio', 'Payment Regularity'),
]

for idx, (var, titulo) in enumerate(vars_box4):
    if var not in df_orig_k4.columns:
        continue
    ax = fig4.add_subplot(gs4[2, idx])

    dados = []
    for c in range(4):
        s = df_orig_k4.loc[df_orig_k4['cluster'] == c, var].dropna()
        if var in ['da_total_debt_value', 'ap_payment_capacity_value']:
            s = np.log10(s.clip(lower=1))
        dados.append(s)

    bp = ax.boxplot(dados, patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2),
                    flierprops=dict(marker='.', markersize=1.5, alpha=0.3),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2))

    for patch, c in zip(bp['boxes'], range(4)):
        patch.set_facecolor(CORES[c]); patch.set_alpha(0.85)

    ax.set_xticklabels([SIGLAS[c] for c in range(4)], fontsize=8)
    ax.set_title(titulo, fontweight='bold', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for i, tick in enumerate(ax.get_xticklabels()):
        tick.set_color(CORES[i])


fig4.suptitle(
    'Final Segmentation Results — K-Means k=4\n'
    'Active High-Capacity  ·  Small Low-Engagement  ·  '
    'Legacy Debt  ·  Financially Stressed Active\n'
    '377,419 debtors | Brazilian Federal Revenue Service — 9th Fiscal Region',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('fig_painel_sintese_final.png', dpi=200, bbox_inches='tight')
plt.show()


# =============================================================================
# SUMÁRIO DOS FICHEIROS GERADOS
# =============================================================================

print("\n" + "=" * 70)
print("FICHEIROS GERADOS — Capítulo de Resultados")
print("=" * 70)
print(f"""
  fig_pca2d_clusters.png         ← Figura principal PCA 2D
  fig_heatmap_desvios.png        ← Heatmap desvios (versão completa)
  fig_radar_perfis.png           ← Radar individual por cluster
  fig_painel_sintese_final.png   ← Painel síntese (figura única dissertação)

  Variância explicada PCA:
    PC1 : {var_exp[0]:.1f}%
    PC2 : {var_exp[1]:.1f}%
    Total: {sum(var_exp):.1f}%

  Nota para a dissertação:
  → PCA foi utilizado exclusivamente para visualização bidimensional.
    O clustering foi realizado no espaço original de 22 dimensões.
    A variância explicada pelos dois primeiros componentes ({sum(var_exp):.1f}%)
    deve ser reportada explicitamente na legenda da figura.
""")

## Modeling 3 (Comparação K = 2)

### Treino e comparação k= 2 , k = 3 e k = 4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
X = df_clustering[var_clustering_final].values

# Amostra para silhouette
rng        = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X.shape[0], size=30_000, replace=False)

# =============================================================================
# DICIONÁRIOS DE PERFILAMENTO
# Variáveis operacionais em escala original (df)
# =============================================================================

vars_perfil_continuas = {
    'da_total_debt_value':                'Dívida Total (R$)',
    'da_enforceable_debt_value':          'Dívida Executável (R$)',
    'da_debt_pgfn_value':                 'Dívida PGFN (R$)',
    'da_debt_0_6_months_value':           'Dívida 0–6 meses (R$)',
    'da_debt_over_48_months_value':       'Dívida >48 meses (R$)',
    'da_penalty_to_principal_ratio':      'Ratio Multas/Principal',
    'da_age_weighted_debt_value_months':  'Idade Pond. Dívida (meses)',
    'da_number_of_debts_count':           'Nº Débitos',
    'da_number_distinct_tax_types_count': 'Nº Tipos Tributo',
    'ap_declared_gross_revedue_value':    'Receita Declarada (R$)',
    'ap_payment_capacity_value':          'Capacidade Pagamento (R$)',
    'ap_debt_to_revenue_ratio':           'Dívida/Receita',
    'pc_total_payment_2025_value':        'Pagamentos 2025 (R$)',
    'pc_instalment_plan_count':           'Nº Parcelamentos',
    'pc_defaulted_instalment_plans_count':'Parcelamentos Rompidos',
    'pc_payment_regularity_ratio':        'Regularidade Parcelamentos',
    'pc_pending_filing_omissions_ratio':  'Omissões de Declaração',
    'tp_enterprise_age_years':            'Idade Empresa (anos)',
    'ci_sectoral_delinquency_deviation_score': 'Desvio Setorial',
}

vars_perfil_binarias = {
    'da_prescription_risk_bin':        'Risco Prescrição (%)',
    'da_new_debt_incidence_bin':       'Incidência Nova Dívida (%)',
    'pc_recent_payment_indicator_bin': 'Pagamento Recente (%)',
}

vars_flags = {
    'is_missing_ap_declared_gross_revenue_value':  'Sem Receita Declarada (%)',
    'is_missing_ap_debt_to_revenue_ratio':         'Sem Ratio Dívida/Receita (%)',
    'is_missing_pc_pending_filing_omissions_ratio':'Sem Dados Omissões (%)',
}

# Filtrar apenas colunas existentes em df
vars_cont_ok  = {k: v for k, v in vars_perfil_continuas.items() if k in df.columns}
vars_bin_ok   = {k: v for k, v in vars_perfil_binarias.items()  if k in df.columns}
vars_flags_ok = {k: v for k, v in vars_flags.items()            if k in df.columns}


# =============================================================================
# FUNÇÃO DE PERFILAMENTO — reutilizada para k=2, k=3 e k=4
# =============================================================================

def formatar_valor(v, p25=None, p75=None):
    """Formata mediana + IQR de acordo com magnitude."""
    iqr = f"  [{p25:,.0f} – {p75:,.0f}]" if p25 is not None and v >= 1_000 else \
          f"  [{p25:.2f} – {p75:.2f}]"   if p25 is not None and v >= 1   else \
          f"  [{p25:.4f} – {p75:.4f}]"   if p25 is not None              else ""
    if v >= 1_000:
        return f"R$ {v:>12,.0f}{iqr}"
    elif v >= 1:
        return f"{v:>10.2f}{iqr}"
    else:
        return f"{v:>10.4f}{iqr}"


def gerar_perfil(df_labeled, col_cluster, n_clusters):
    """
    Retorna DataFrame com mediana + IQR para contínuas
    e percentual para binárias/flags, por cluster.
    """
    clusters  = list(range(n_clusters))
    col_names = [f'C{c}' for c in clusters] + ['Global']
    rows      = []

    # --- Tamanho ---
    row = {'Label': 'Tamanho do cluster', 'Secao': '── Dimensão'}
    n_total = len(df_labeled)
    for c in clusters:
        n   = (df_labeled[col_cluster] == c).sum()
        row[f'C{c}'] = f"{n:,}  ({n/n_total*100:.1f}%)"
    row['Global'] = f"{n_total:,}  (100%)"
    rows.append(row)

    # --- Contínuas ---
    rows.append({'Label': '', 'Secao': '── Variáveis Operacionais (Mediana  [P25 – P75])'})
    for var, label in vars_cont_ok.items():
        row = {'Label': label, 'Secao': ''}
        for c in clusters:
            s   = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            row[f'C{c}'] = formatar_valor(s.median(), s.quantile(0.25), s.quantile(0.75))
        sg  = df_labeled[var].dropna()
        row['Global'] = formatar_valor(sg.median(), sg.quantile(0.25), sg.quantile(0.75))
        rows.append(row)

    # --- Binárias e flags ---
    rows.append({'Label': '', 'Secao': '── Indicadores Descritivos (%)'})
    for var, label in {**vars_bin_ok, **vars_flags_ok}.items():
        row = {'Label': label, 'Secao': ''}
        for c in clusters:
            s   = df_labeled.loc[df_labeled[col_cluster] == c, var].dropna()
            row[f'C{c}'] = f"{s.mean()*100:.1f}%"
        row['Global'] = f"{df_labeled[var].dropna().mean()*100:.1f}%"
        rows.append(row)

    return pd.DataFrame(rows)


def imprimir_perfil(df_perfil, n_clusters, titulo):
    col_data  = [f'C{c}' for c in range(n_clusters)] + ['Global']
    largura_c = 32

    print(f"\n{'='*70}")
    print(titulo)
    print(f"{'='*70}")

    for _, row in df_perfil.iterrows():
        if row['Secao'] and row['Label'] == '':
            print(f"\n  {row['Secao']}")
            print("  " + "─" * (45 + largura_c * len(col_data)))
            continue
        if row['Label'] == '':
            continue
        linha = f"  {row['Label']:<44}"
        for col in col_data:
            linha += f"  {str(row.get(col,'')):<{largura_c}}"
        print(linha)


# =============================================================================
# BLOCO A0 — K-MEANS k=2
# =============================================================================

print("=" * 70)
print("MODELAGEM — K-Means  k=2")
print("=" * 70)

km2 = KMeans(n_clusters=2, init='k-means++', n_init=20,
             max_iter=500, random_state=RANDOM_STATE)
labels_k2 = km2.fit_predict(X)

sil_k2 = silhouette_score(X[idx_sample], labels_k2[idx_sample])
dist_k2 = pd.Series(labels_k2).value_counts().sort_index()

print(f"\n  Inércia            : {km2.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k2:.4f}")
print(f"\n  Distribuição:")
for c, n in dist_k2.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k2)*100:.1f}%)")

df_orig_k2            = df.copy()
df_orig_k2['cluster'] = labels_k2

perfil_k2 = gerar_perfil(df_orig_k2, 'cluster', 2)
imprimir_perfil(perfil_k2, 2, "TABELA DE PERFILAMENTO — k=2")


# =============================================================================
# BLOCO A — K-MEANS k=3
# =============================================================================

print("\n\n" + "=" * 70)
print("MODELAGEM — K-Means  k=3")
print("=" * 70)

km3 = KMeans(n_clusters=3, init='k-means++', n_init=20,
             max_iter=500, random_state=RANDOM_STATE)
labels_k3 = km3.fit_predict(X)

sil_k3 = silhouette_score(X[idx_sample], labels_k3[idx_sample])
dist_k3 = pd.Series(labels_k3).value_counts().sort_index()

print(f"\n  Inércia            : {km3.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k3:.4f}")
print(f"\n  Distribuição:")
for c, n in dist_k3.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k3)*100:.1f}%)")

df_orig_k3            = df.copy()
df_orig_k3['cluster'] = labels_k3

perfil_k3 = gerar_perfil(df_orig_k3, 'cluster', 3)
imprimir_perfil(perfil_k3, 3, "TABELA DE PERFILAMENTO — k=3")


# =============================================================================
# BLOCO B — K-MEANS k=4
# =============================================================================

print("\n\n" + "=" * 70)
print("MODELAGEM — K-Means  k=4")
print("=" * 70)

km4 = KMeans(n_clusters=4, init='k-means++', n_init=20,
             max_iter=500, random_state=RANDOM_STATE)
labels_k4 = km4.fit_predict(X)

sil_k4 = silhouette_score(X[idx_sample], labels_k4[idx_sample])
dist_k4 = pd.Series(labels_k4).value_counts().sort_index()

print(f"\n  Inércia            : {km4.inertia_:,.2f}")
print(f"  Silhouette (30k)   : {sil_k4:.4f}")
print(f"\n  Distribuição:")
for c, n in dist_k4.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k4)*100:.1f}%)")

df_orig_k4            = df.copy()
df_orig_k4['cluster'] = labels_k4

perfil_k4 = gerar_perfil(df_orig_k4, 'cluster', 4)
imprimir_perfil(perfil_k4, 4, "TABELA DE PERFILAMENTO — k=4")


# =============================================================================
# BLOCO C — VISUALIZAÇÕES COMPARATIVAS (k=2, k=3, k=4)
# Grelha: 4 linhas × 6 colunas
#   Linha 0 : Distribuição   k=2 [0:2] | k=3 [2:4] | k=4 [4:6]
#   Linha 1 : Boxplot        k=2 [0:2] | k=3 [2:4] | k=4 [4:6]
#   Linha 2 : Heatmap desvios k=3 [0:3] | k=4 [3:6]   (candidatos)
#   Linha 3 : Silhouette     k=2 [0:2] | k=3 [2:4] | k=4 [4:6]
# =============================================================================

fig = plt.figure(figsize=(22, 22))
gs  = gridspec.GridSpec(4, 6, figure=fig, hspace=0.52, wspace=0.42)

cores_k2 = ['#2980b9', '#27ae60']
cores_k3 = ['#2980b9', '#e74c3c', '#27ae60']
cores_k4 = ['#2980b9', '#27ae60', '#e74c3c', '#f39c12']

modelos = [
    (2, labels_k2, dist_k2, df_orig_k2, cores_k2),
    (3, labels_k3, dist_k3, df_orig_k3, cores_k3),
    (4, labels_k4, dist_k4, df_orig_k4, cores_k4),
]

var_box = 'da_total_debt_value'

# --- Linha 0: Distribuição ---
for idx, (k, labels, dist, _, cores) in enumerate(modelos):
    col_start = idx * 2
    ax = fig.add_subplot(gs[0, col_start:col_start + 2])
    ns  = [dist[c] for c in range(k)]
    bars = ax.bar([f'C{c}' for c in range(k)], ns,
                  color=cores, edgecolor='white')
    for bar, n in zip(bars, ns):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 500,
                f'{n:,}\n({n/len(labels)*100:.1f}%)',
                ha='center', va='bottom', fontsize=8)
    ax.set_title(f'Tamanho — k={k}', fontweight='bold')
    ax.set_ylabel('N devedores')
    ax.grid(axis='y', alpha=0.3)

# --- Linha 1: Boxplot Dívida Total ---
for idx, (k, labels, dist, df_orig, cores) in enumerate(modelos):
    col_start = idx * 2
    ax = fig.add_subplot(gs[1, col_start:col_start + 2])
    dados = [np.log1p(df_orig.loc[df_orig['cluster'] == c, var_box].dropna())
             for c in range(k)]
    bp = ax.boxplot(dados, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, cor in zip(bp['boxes'], cores):
        patch.set_facecolor(cor)
        patch.set_alpha(0.8)
    ax.set_xticklabels([f'C{c}' for c in range(k)])
    ax.set_title(f'Dívida Total log — k={k}', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

# --- Linha 2: Heatmap desvios — apenas k=3 e k=4 (candidatos) ---
vars_hm = [v for v in [
    'da_total_debt_value', 'da_debt_over_48_months_value',
    'da_penalty_to_principal_ratio', 'pc_total_payment_2025_value',
    'pc_instalment_plan_count', 'pc_payment_regularity_ratio',
    'ap_payment_capacity_value', 'ap_debt_to_revenue_ratio',
    'tp_enterprise_age_years', 'ci_sectoral_delinquency_deviation_score'
] if v in df.columns]

def heatmap_desvios(ax, df_labeled, n_clusters, titulo):
    med_g   = df_labeled[vars_hm].median()
    desvios = pd.DataFrame(index=range(n_clusters), columns=vars_hm, dtype=float)
    for c in range(n_clusters):
        med_c = df_labeled.loc[df_labeled['cluster'] == c, vars_hm].median()
        desvios.loc[c] = ((med_c - med_g) / med_g.replace(0, np.nan)) * 100
    desvios.index = [f'C{c}' for c in range(n_clusters)]
    sns.heatmap(desvios.T.astype(float), annot=True, fmt='.0f',
                cmap='RdBu_r', center=0, linewidths=0.4,
                linecolor='#ecf0f1', annot_kws={'size': 8}, ax=ax,
                cbar_kws={'shrink': 0.6})
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=7)

ax_hm3 = fig.add_subplot(gs[2, :3])
heatmap_desvios(ax_hm3, df_orig_k3, 3, 'Desvio da Mediana Global — k=3')

ax_hm4 = fig.add_subplot(gs[2, 3:])
heatmap_desvios(ax_hm4, df_orig_k4, 4, 'Desvio da Mediana Global — k=4')

# --- Linha 3: Silhouette plots ---
def silhouette_subplot(ax, X, labels, k, cores, titulo):
    sil_vals = silhouette_samples(X[idx_sample], labels[idx_sample])
    y_lower  = 10
    for i in range(k):
        sil_i  = np.sort(sil_vals[labels[idx_sample] == i])
        size_i = len(sil_i)
        ax.fill_betweenx(np.arange(y_lower, y_lower + size_i),
                         0, sil_i, facecolor=cores[i], alpha=0.8)
        ax.text(-0.06, y_lower + size_i / 2, f'C{i}',
                fontsize=9, color=cores[i], fontweight='bold')
        y_lower += size_i + 10
    media = np.mean(sil_vals)
    ax.axvline(media, color='black', linestyle='--', lw=1.8,
               label=f'Média = {media:.4f}')
    ax.set_title(titulo, fontweight='bold', fontsize=10)
    ax.set_xlabel('Coeficiente de Silhouette')
    ax.set_yticks([])
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

for idx, (k, labels, dist, df_orig, cores) in enumerate(modelos):
    col_start = idx * 2
    ax = fig.add_subplot(gs[3, col_start:col_start + 2])
    silhouette_subplot(ax, X, labels, k, cores,
                       f'Silhouette — k={k}  (amostra 30k)')

plt.suptitle(
    "Modelagem Comparativa — K-Means k=2 vs k=3 vs k=4\n"
    "377.419 devedores | RFB 9ª Região Fiscal | 22 variáveis",
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('modelagem_k2_k3_k4_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()


# =============================================================================
# BLOCO D — SÍNTESE COMPARATIVA
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE COMPARATIVA — k=2 vs k=3 vs k=4")
print("=" * 70)

print(f"""
  {'Métrica':<30} {'k=2':>12} {'k=3':>12} {'k=4':>12}
  {'─'*68}
  {'Inércia':<30} {km2.inertia_:>12,.0f} {km3.inertia_:>12,.0f} {km4.inertia_:>12,.0f}
  {'Silhouette (30k)':<30} {sil_k2:>12.4f} {sil_k3:>12.4f} {sil_k4:>12.4f}
  {'Nº variáveis':<30} {'22':>12} {'22':>12} {'22':>12}
  {'Nº devedores':<30} {len(labels_k2):>12,} {len(labels_k3):>12,} {len(labels_k4):>12,}

  Distribuição k=2:""")
for c, n in dist_k2.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k2)*100:.1f}%)")

print(f"\n  Distribuição k=3:")
for c, n in dist_k3.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k3)*100:.1f}%)")

print(f"\n  Distribuição k=4:")
for c, n in dist_k4.items():
    print(f"    Cluster {c}: {n:>7,}  ({n/len(labels_k4)*100:.1f}%)")

print(f"""
  Modelos guardados em    : km2, km3, km4
  Labels k=2 guardadas em : df_orig_k2['cluster']
  Labels k=3 guardadas em : df_orig_k3['cluster']
  Labels k=4 guardadas em : df_orig_k4['cluster']

  Próximo passo:
  → Confirmar que k=2 colapsa distinções operacionais relevantes
  → Analisar qual cluster de k=3 é subdividido em k=4
  → Seleccionar k final com base na utilidade operacional dos segmentos
""")

## Deployment


In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import json
from datetime import datetime
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing    import StandardScaler

# =============================================================================
# DEPLOYMENT PIPELINE — Collections Segmentation Cockpit
# RFB 9ª Região Fiscal | K-Means k=4
# =============================================================================

RANDOM_STATE = 42
K_FINAL      = 4
OUTPUT_DIR   = 'deployment'
TIMESTAMP    = datetime.now().strftime('%Y%m%d')

os.makedirs(f'{OUTPUT_DIR}/model',   exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/data',    exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/monitor', exist_ok=True)

print("=" * 70)
print("DEPLOYMENT PIPELINE — Collections Segmentation Cockpit")
print(f"Timestamp : {TIMESTAMP}")
print(f"Output    : {OUTPUT_DIR}/")
print("=" * 70)


# =============================================================================
# BLOCO 0 — ENRIQUECER df_orig_k4 COM VARIÁVEIS CATEGÓRICAS
# As categóricas não entraram no clustering mas são essenciais
# para o Cockpit (filtros, drill-down, validação de perfis)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 0 — Merge de Variáveis Categóricas")
print("─" * 70)

# Variáveis categóricas a incorporar
VARS_CATEGORICAS = [

    'tp_enterprise_size_cat',
    'tp_economic_sector_desc_cat',
    'tp_registration_status_cat', 
    'tp_tax_regime_cat',
    'tp_legal_form_cat',
    'tp_fiscal_region_cat',
    'tp_geographic_location_cat',
    'tp_special_taxpayer_status_cat',
    'tp_special_tax_regime_cat'
    
]

# Verificar quais já estão em df_orig_k4
cats_ja_presentes = [v for v in VARS_CATEGORICAS if v in df_orig_k4.columns]
cats_para_merge   = [v for v in VARS_CATEGORICAS if v not in df_orig_k4.columns]

print(f"\n  Variáveis categóricas já presentes em df_orig_k4 : {len(cats_ja_presentes)}")
print(f"  Variáveis categóricas a incorporar via merge      : {len(cats_para_merge)}")

if cats_para_merge:
    # Identificar fonte — df_orig é o dataframe antes do pipeline de clustering
    fonte_disponivel = [v for v in cats_para_merge if v in df_orig.columns]
    fonte_ausente    = [v for v in cats_para_merge if v not in df_orig.columns]

    if fonte_disponivel:
        cols_merge = ['id_cnpj'] + fonte_disponivel
        df_orig_k4 = df_orig_k4.merge(
            df_orig[cols_merge].drop_duplicates('id_cnpj'),
            on='id_cnpj',
            how='left',
            suffixes=('', '_dup')
        )
        # Remover eventuais colunas duplicadas geradas pelo merge
        df_orig_k4 = df_orig_k4[[c for c in df_orig_k4.columns
                                  if not c.endswith('_dup')]]
        print(f"\n  ✔  Merge concluído: {len(fonte_disponivel)} variáveis adicionadas")
        for v in fonte_disponivel:
            print(f"       {v}")

    if fonte_ausente:
        print(f"\n  ⚠  Variáveis não encontradas em df_orig (verificar fonte):")
        for v in fonte_ausente:
            print(f"       {v}")

# Lista final de categóricas disponíveis
CATS_OK = [v for v in VARS_CATEGORICAS if v in df_orig_k4.columns]
print(f"\n  Total categóricas disponíveis para o Cockpit: {len(CATS_OK)}")
for v in CATS_OK:
    n_vals  = df_orig_k4[v].nunique()
    n_nulos = df_orig_k4[v].isna().sum()
    pct_nulos = n_nulos / len(df_orig_k4) * 100
    flag_nulos = "  ⚠  missings" if pct_nulos > 5 else ""
    print(f"    ✔  {v:<45} {n_vals:>3} cat  |  {pct_nulos:.1f}% nulos{flag_nulos}")


# =============================================================================
# BLOCO 1 — SERIALIZAÇÃO DO ARTEFACTO COMPLETO  (corrigido)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 1 — Serialização do Artefacto")
print("─" * 70)

# ------------------------------------------------------------------
# Diagnóstico explícito do problema e solução adoptada
#
# O scaler em memória (RobustScaler) foi ajustado em 25 variáveis
# contínuas antes da análise de correlação. O espaço de features
# final tem 22 variáveis:
#   → 19 contínuas (foram escaladas pelo RobustScaler original)
#   → 3 flags is_missing_* (não foram escaladas — escala binária 0/1)
#
# Para inverse_transform funcionar, o scaler de deployment deve ser
# ajustado exactamente nas 19 variáveis contínuas presentes no modelo.
# As colunas de flags NÃO são invertidas — são preservadas tal como
# estão nos centróides.
# ------------------------------------------------------------------

# Separar contínuas e flags no espaço de features final
var_continuas_modelo = [
    v for v in var_clustering_final
    if not v.startswith('is_missing_')
]
var_flags_modelo = [
    v for v in var_clustering_final
    if v.startswith('is_missing_')
]

print(f"\n  Espaço de features final        : {len(var_clustering_final)} variáveis")
print(f"    → Contínuas (escala RobustScaler): {len(var_continuas_modelo)}")
print(f"    → Flags is_missing_ (sem escala) : {len(var_flags_modelo)}")

# Índices das colunas contínuas dentro de var_clustering_final
# (necessário para extrair as colunas correctas dos centróides)
idx_continuas = [var_clustering_final.index(v) for v in var_continuas_modelo]
idx_flags     = [var_clustering_final.index(v) for v in var_flags_modelo]

print(f"\n  Índices contínuas nos centróides : {idx_continuas[:5]}...{idx_continuas[-3:]}")
print(f"  Índices flags nos centróides     : {idx_flags}")

# Reajustar RobustScaler nas 19 variáveis contínuas do modelo final
# df_pos_outlier contém estas variáveis na escala pré-normalização
scaler_deploy = RobustScaler()
scaler_deploy.fit(df_pos_outlier[var_continuas_modelo])

assert scaler_deploy.n_features_in_ == len(var_continuas_modelo), (
    f"Scaler ajustado em {scaler_deploy.n_features_in_} vars — "
    f"esperado {len(var_continuas_modelo)}"
)
assert km4.cluster_centers_.shape == (K_FINAL, len(var_clustering_final)), (
    f"Centróides shape {km4.cluster_centers_.shape} — "
    f"esperado ({K_FINAL}, {len(var_clustering_final)})"
)

print(f"\n  ✔  RobustScaler reajustado: {scaler_deploy.n_features_in_} variáveis contínuas")
print(f"  ✔  Centróides verificados : {km4.cluster_centers_.shape}")

# ------------------------------------------------------------------
# Centróides no espaço normalizado (22 dim — usado para classificação)
# ------------------------------------------------------------------
centroides_norm = km4.cluster_centers_.copy()

# ------------------------------------------------------------------
# Centróides no espaço original (interpretação humana)
# Apenas as colunas contínuas são invertidas; flags mantêm valor 0/1
# ------------------------------------------------------------------
centroides_continuas_orig = scaler_deploy.inverse_transform(
    centroides_norm[:, idx_continuas]
)

# Montar DataFrame interpretável: contínuas em escala original + flags
df_centroides_orig = pd.DataFrame(
    centroides_continuas_orig,
    columns=var_continuas_modelo
)

# Adicionar colunas de flags (valor original dos centróides — próximo de 0)
for i, v in zip(idx_flags, var_flags_modelo):
    df_centroides_orig[v] = centroides_norm[:, i]

# Reordenar para seguir a ordem de var_clustering_final
df_centroides_orig = df_centroides_orig[var_clustering_final]

df_centroides_orig.insert(0, 'cluster_id',    range(K_FINAL))
df_centroides_orig.insert(1, 'cluster_name',  [cluster_names[c] for c in range(K_FINAL)])
df_centroides_orig.insert(2, 'cluster_short', [cluster_short[c] for c in range(K_FINAL)])

# Verificação visual rápida — 3 variáveis diagnóstico
print(f"\n  Centróides — escala original (selecção de variáveis):")
print(f"\n  {'Cluster':<8} {'Dívida Total':>18} {'Cap. Pagamento':>18} {'Pagamentos 2025':>18}")
print("  " + "─" * 65)
for c in range(K_FINAL):
    row = df_centroides_orig[df_centroides_orig['cluster_id'] == c].iloc[0]
    # inverse_transform de log1p para escala R$
    def from_log(v): return np.expm1(v) if v > 0 else 0
    d  = from_log(row.get('da_total_debt_value_log', 0))
    cp = from_log(row.get('ap_payment_capacity_value_log', 0))
    pg = from_log(row.get('pc_total_payment_2025_value_log', 0))
    print(f"  {cluster_short[c]:<8} {d:>18,.0f} {cp:>18,.0f} {pg:>18,.0f}")

# ------------------------------------------------------------------
# Artefacto completo — dicionário único
# ------------------------------------------------------------------
artefacto = {
    'model_version'          : f'kmeans_k4_{TIMESTAMP}',
    'training_date'          : TIMESTAMP,
    'n_clusters'             : K_FINAL,
    'n_training_obs'         : len(labels_k4),
    'n_features'             : len(var_clustering_final),
    'feature_names'          : var_clustering_final,
    'var_continuas_modelo'   : var_continuas_modelo,   # 19 vars contínuas
    'var_flags_modelo'       : var_flags_modelo,       # 3 flags is_missing_
    'idx_continuas'          : idx_continuas,          # índices no vector de 22
    'cluster_names'          : cluster_names,
    'cluster_short'          : cluster_short,
    'cluster_colors'         : cluster_colors,
    'kmeans_model'           : km4,
    'scaler'                 : scaler_deploy,          # RobustScaler, 19 vars
    'centroids_normalised'   : centroides_norm,        # (4, 22)
    'inertia'                : km4.inertia_,
    'silhouette'             : 0.2558,
    'ari_min_20seeds'        : 0.9930,
}

# Serializar
with open(f'{OUTPUT_DIR}/model/artefacto_v{TIMESTAMP}.pkl', 'wb') as f:
    pickle.dump(artefacto, f)
with open(f'{OUTPUT_DIR}/model/kmeans_k4.pkl', 'wb') as f:
    pickle.dump(km4, f)
with open(f'{OUTPUT_DIR}/model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler_deploy, f)
with open(f'{OUTPUT_DIR}/model/feature_names.json', 'w') as f:
    json.dump(var_clustering_final, f, indent=2)
with open(f'{OUTPUT_DIR}/model/cluster_metadata.json', 'w',
          encoding='utf-8') as f:
    json.dump({
        'cluster_names'       : cluster_names,
        'cluster_short'       : cluster_short,
        'var_continuas_modelo': var_continuas_modelo,
        'var_flags_modelo'    : var_flags_modelo,
    }, f, indent=2, ensure_ascii=False)

df_centroides_orig.to_csv(
    f'{OUTPUT_DIR}/model/centroides_escala_original.csv', index=False)

print(f"\n  Artefacto serializado:")
print(f"    Versão         : kmeans_k4_{TIMESTAMP}")
print(f"    Scaler         : RobustScaler — {len(var_continuas_modelo)} variáveis contínuas")
print(f"    Centróides     : {centroides_norm.shape}  (normalizado)")
print(f"    N observações  : {len(labels_k4):,}")
print(f"    Inércia        : {km4.inertia_:,.2f}")
print(f"    Silhouette     : 0.2558")


# =============================================================================
# BLOCO 2 — FUNÇÃO DE CLASSIFICAÇÃO (corrigida)
# Escala apenas as 19 variáveis contínuas; flags is_missing_* passam
# directamente sem transformação, tal como no pipeline de treino
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 2 — Função de Classificação (Nearest Centroid)")
print("─" * 70)

def classificar_novos_devedores(
    df_novos            : pd.DataFrame,
    artefacto           : dict,
    id_col              : str  = 'id_cnpj',
    retornar_distancias : bool = True,
) -> pd.DataFrame:
    """
    Classifica novos devedores no segmento mais próximo via distância
    euclidiana ao centróide no espaço normalizado de 22 variáveis.

    Pipeline interno:
      1. Extrair as 19 variáveis contínuas → aplicar RobustScaler
      2. Extrair as 3 flags is_missing_*   → manter em escala original
      3. Concatenar → vector de 22 dimensões → distância aos centróides
    """
    feature_names    = artefacto['feature_names']       # 22 vars (ordem)
    var_continuas    = artefacto['var_continuas_modelo'] # 19 contínuas
    var_flags        = artefacto['var_flags_modelo']     # 3 flags
    scaler           = artefacto['scaler']               # RobustScaler (19)
    centroids        = artefacto['centroids_normalised'] # (4, 22)
    c_names          = artefacto['cluster_names']
    c_short          = artefacto['cluster_short']

    # Verificar colunas necessárias
    cols_faltando = [f for f in feature_names if f not in df_novos.columns]
    if cols_faltando:
        raise ValueError(f"Colunas em falta no input:\n{cols_faltando}")

    # ------------------------------------------------------------------
    # 1. Escalar as 19 variáveis contínuas
    # ------------------------------------------------------------------
    X_cont_norm = scaler.transform(df_novos[var_continuas].values)  # (n, 19)

    # ------------------------------------------------------------------
    # 2. Extrair as 3 flags sem transformação
    # ------------------------------------------------------------------
    X_flags = df_novos[var_flags].values.astype(float)              # (n, 3)

    # ------------------------------------------------------------------
    # 3. Montar vector completo na ordem de feature_names (22 dimensões)
    # ------------------------------------------------------------------
    # Reconstruir a ordem correcta coluna a coluna
    n = len(df_novos)
    X_norm = np.empty((n, len(feature_names)), dtype=float)

    idx_cont  = [feature_names.index(v) for v in var_continuas]
    idx_flags = [feature_names.index(v) for v in var_flags]

    X_norm[:, idx_cont]  = X_cont_norm
    X_norm[:, idx_flags] = X_flags

    # ------------------------------------------------------------------
    # 4. Distância euclidiana aos centróides → atribuição de cluster
    # ------------------------------------------------------------------
    dists       = euclidean_distances(X_norm, centroids)   # (n, 4)
    cluster_ids = dists.argmin(axis=1)

    # ------------------------------------------------------------------
    # 5. Montar resultado
    # ------------------------------------------------------------------
    resultado = pd.DataFrame()
    if id_col in df_novos.columns:
        resultado[id_col] = df_novos[id_col].values

    resultado['cluster_id']    = cluster_ids
    resultado['cluster_name']  = [c_names[c] for c in cluster_ids]
    resultado['cluster_short'] = [c_short[c]  for c in cluster_ids]

    if retornar_distancias:
        resultado['dist_cluster_atribuido'] = \
            dists[np.arange(len(dists)), cluster_ids]
        for c in range(len(c_names)):
            resultado[f'dist_c{c}'] = dists[:, c]
        resultado['confianca_relativa'] = (
            1 - resultado['dist_cluster_atribuido'] / dists.sum(axis=1)
        )

    return resultado

print("  ✔  Função classificar_novos_devedores() redefinida (corrigida)")
print(f"     Pipeline: {len(artefacto['var_continuas_modelo'])} vars → RobustScaler "
      f"+ {len(artefacto['var_flags_modelo'])} flags → 22 dims → nearest centroid")


# =============================================================================
# BLOCO 3 — VALIDAÇÃO (auto-classificação da carteira de treino)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 3 — Validação da Função")
print("─" * 70)

df_validacao_bruta          = df_pos_outlier[var_clustering_final].copy()
df_validacao_bruta['id_cnpj'] = df_orig_k4['id_cnpj'].values

resultado_validacao = classificar_novos_devedores(
    df_validacao_bruta, artefacto,
    id_col='id_cnpj', retornar_distancias=True
)

labels_reconst = resultado_validacao['cluster_id'].values
concordancia   = (labels_reconst == labels_k4).mean() * 100

print(f"\n  Concordância com labels originais: {concordancia:.2f}%")
print(f"  {'✔  Validação aprovada' if concordancia >= 99.0 else '⚠  Verificar pipeline'}")

print(f"\n  Distribuição reclassificada vs referência:")
for c in range(K_FINAL):
    n   = (labels_reconst == c).sum()
    ref = (labels_k4 == c).sum()
    print(f"    C{c} {cluster_short[c]:<8}: {n:>7,}  (ref: {ref:>7,}  Δ = {n-ref:>+5})")

print(f"\n  Confiança relativa — mediana por cluster:")
for c in range(K_FINAL):
    mask = resultado_validacao['cluster_id'] == c
    med  = resultado_validacao.loc[mask, 'confianca_relativa'].median()
    print(f"    C{c} {cluster_short[c]:<8}: {med:.4f}")


# =============================================================================
# BLOCO 4 — CONSTRUÇÃO E EXPORTAÇÃO DO DATASET PARA POWER BI  (completo)
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 4 — Construção e Exportação do Dataset para Power BI")
print("─" * 70)

# ------------------------------------------------------------------
# 4.0 Base: df_orig_k4 + garantir colunas de cluster
# ------------------------------------------------------------------
df_export = df_orig_k4.copy()

# Normalizar coluna de cluster — pode estar como 'cluster' ou 'cluster_id'
if 'cluster_id' not in df_export.columns:
    if 'cluster' in df_export.columns:
        df_export['cluster_id'] = df_export['cluster']
    else:
        df_export['cluster_id'] = labels_k4

if 'cluster_name' not in df_export.columns:
    df_export['cluster_name']  = df_export['cluster_id'].map(cluster_names)

if 'cluster_short' not in df_export.columns:
    df_export['cluster_short'] = df_export['cluster_id'].map(cluster_short)

print(f"\n  Base: {len(df_export):,} devedores")
print(f"  Distribuição:")
for c in range(K_FINAL):
    n = (df_export['cluster_id'] == c).sum()
    print(f"    C{c} {cluster_short[c]:<8}: {n:>7,}  ({n/len(df_export)*100:.1f}%)")

# ------------------------------------------------------------------
# 4.1 Estratégias e prioridade
# ------------------------------------------------------------------
ESTRATEGIAS = {
    0: 'Direct Negotiation — Priority Instalment Proposal',
    1: 'Automated Outreach — Digital Self-Regularisation',
    2: 'Legal Action — Urgent Judicial Enforcement',
    3: 'Accessible Instalment — Financial Stress Monitoring',
}
PRIORIDADE = {0: 1, 2: 2, 3: 3, 1: 4}

df_export['collection_strategy'] = df_export['cluster_id'].map(ESTRATEGIAS)
df_export['priority_rank']       = df_export['cluster_id'].map(PRIORIDADE)


# ------------------------------------------------------------------
# 4.1b — CÁLCULO E JOIN DOS FLAGS DE MISSING
# Calculados a partir de df_orig (dataframe ANTES da imputação,
# onde os NaN ainda estão presentes).
# ATENÇÃO: usar df_orig e não df_export — df_export já tem os
# valores imputados e todos os flags ficariam a zero.
# ------------------------------------------------------------------

VARS_MISSING_FLAGS = {
    'is_missing_ap_declared_gross_revenue_value' : 'ap_declared_gross_revedue_value',
    'is_missing_ap_debt_to_revenue_ratio'        : 'ap_debt_to_revenue_ratio',
    'is_missing_pc_pending_filing_omissions_ratio': 'pc_pending_filing_omissions_ratio',
}

print("\n  4.1b — Flags de Missing:")

df_flags = pd.DataFrame({'id_cnpj': df_orig['id_cnpj']})

for flag_name, source_col in VARS_MISSING_FLAGS.items():
    if source_col in df_orig.columns:
        df_flags[flag_name] = df_orig[source_col].isna().astype(int)
        n   = df_flags[flag_name].sum()
        pct = n / len(df_flags) * 100
        print(f"    ✔  {flag_name}: {n:,} ({pct:.1f}%)")
    else:
        df_flags[flag_name] = 0
        print(f"    ⚠  {source_col} não encontrada em df_orig — flag = 0")

# Validação: se todos forem zero, a fonte está errada
flags_cols = list(VARS_MISSING_FLAGS.keys())
all_zero = all(df_flags[c].sum() == 0 for c in flags_cols)
if all_zero:
    raise ValueError(
        "ERRO: todos os flags têm soma zero.\n"
        "Verifique se df_orig é o dataframe PRÉ-imputação com NaN intactos."
    )

# Join ao df_export pelo id_cnpj
df_export = df_export.merge(df_flags, on='id_cnpj', how='left')

# Garantir que não ficaram NaN residuais (devedores sem par em df_orig)
for c in flags_cols:
    df_export[c] = df_export[c].fillna(0).astype(int)

print(f"\n    Shape df_export após join dos flags: {df_export.shape}")




# ------------------------------------------------------------------
# 4.2 Merge de distâncias e confiança (resultado_validacao do Bloco 3)
# ------------------------------------------------------------------
df_export = df_export.merge(
    resultado_validacao[[
        'id_cnpj',
        'dist_cluster_atribuido',
        'dist_c0', 'dist_c1', 'dist_c2', 'dist_c3',
        'confianca_relativa',
    ]],
    on='id_cnpj', how='left'
)

# ------------------------------------------------------------------
# 4.3 Definição dos blocos de colunas
# ------------------------------------------------------------------
COLS_SEGMENTO = [
    'id_cnpj',
    'cluster_id', 'cluster_name', 'cluster_short',
    'collection_strategy', 'priority_rank',
]

COLS_CADASTRAIS = [
    'tp_enterprise_size_cat',
    'tp_economic_sector_desc_cat',
    'tp_registration_status_cat',
    'tp_tax_regime_cat',
    'tp_legal_form_cat',
    'tp_fiscal_region_cat',
    'tp_geographic_location_cat',
    'tp_special_taxpayer_status_cat',
    'tp_special_tax_regime_cat',
]

COLS_DIVIDA = [
    'da_total_debt_value',
    'da_enforceable_debt_value',
    'da_debt_pgfn_value',
    'da_debt_0_6_months_value',
    'da_debt_over_48_months_value',
    'da_penalty_to_principal_ratio',
    'da_age_weighted_debt_value_months',
    'da_number_of_debts_count',
    'da_number_distinct_tax_types_count',
]

COLS_CAPAG = [
    'ap_declared_gross_revedue_value',
    'ap_payment_capacity_value',
    'ap_debt_to_revenue_ratio',
]

COLS_HISTORICO = [
    'pc_total_payment_2025_value',
    'pc_instalment_plan_count',
    'pc_defaulted_instalment_plans_count',
    'pc_payment_regularity_ratio',
    'pc_pending_filing_omissions_ratio',
]

COLS_PERFIL = [
    'tp_enterprise_age_years',
    'ci_sectoral_delinquency_deviation_score',
]

COLS_DISTANCIA = [
    'dist_cluster_atribuido',
    'confianca_relativa',
    'dist_c0', 'dist_c1', 'dist_c2', 'dist_c3',
]

COLS_FLAGS = [
    'is_missing_ap_declared_gross_revenue_value',
    'is_missing_ap_debt_to_revenue_ratio',
    'is_missing_pc_pending_filing_omissions_ratio',
]

# ------------------------------------------------------------------
# 4.4 Consolidar — apenas colunas presentes
# ------------------------------------------------------------------
def filtrar_presentes(cols, df):
    return [c for c in cols if c in df.columns]

def contar_presentes(cols, df):
    return len(filtrar_presentes(cols, df))

colunas_finais = list(dict.fromkeys(
    filtrar_presentes(COLS_SEGMENTO,   df_export) +
    filtrar_presentes(COLS_CADASTRAIS, df_export) +
    filtrar_presentes(COLS_DIVIDA,     df_export) +
    filtrar_presentes(COLS_CAPAG,      df_export) +
    filtrar_presentes(COLS_HISTORICO,  df_export) +
    filtrar_presentes(COLS_PERFIL,     df_export) +
    filtrar_presentes(COLS_DISTANCIA,  df_export) +
    filtrar_presentes(COLS_FLAGS,      df_export)
))

df_cockpit = df_export[colunas_finais].copy()

# ------------------------------------------------------------------
# 4.5 Exportar
# ------------------------------------------------------------------
df_cockpit.to_csv(
    f'{OUTPUT_DIR}/data/devedores_classificados_{TIMESTAMP}.csv',
    index=False, encoding='utf-8-sig'
)

# ------------------------------------------------------------------
# 4.6 Relatório de composição
# ------------------------------------------------------------------
print(f"\n  Dataset exportado:")
print(f"    Linhas  : {len(df_cockpit):,}")
print(f"    Colunas : {len(colunas_finais)}")
print(f"""
  Composição por bloco:
    A — Identidade / Segmento          : {contar_presentes(COLS_SEGMENTO,   df_cockpit):>2} colunas
    B — Perfil Cadastral (categóricas) : {contar_presentes(COLS_CADASTRAIS, df_cockpit):>2} colunas
    C — Atributos da Dívida            : {contar_presentes(COLS_DIVIDA,     df_cockpit):>2} colunas
    D — Capacidade de Pagamento        : {contar_presentes(COLS_CAPAG,      df_cockpit):>2} colunas
    E — Histórico / Conformidade       : {contar_presentes(COLS_HISTORICO,  df_cockpit):>2} colunas
    F — Perfil / Contexto              : {contar_presentes(COLS_PERFIL,     df_cockpit):>2} colunas
    G — Distância / Confiança          : {contar_presentes(COLS_DISTANCIA,  df_cockpit):>2} colunas
    H — Flags de Missing               : {contar_presentes(COLS_FLAGS,      df_cockpit):>2} colunas
""")

# Categóricas presentes
cats_presentes  = filtrar_presentes(COLS_CADASTRAIS, df_cockpit)
cats_ausentes   = [c for c in COLS_CADASTRAIS if c not in df_cockpit.columns]

print(f"  Categóricas incluídas ({len(cats_presentes)}):")
for v in cats_presentes:
    n_cat  = df_cockpit[v].nunique()
    n_null = df_cockpit[v].isna().sum()
    print(f"    ✔  {v:<45} {n_cat:>3} cat  |  "
          f"{n_null/len(df_cockpit)*100:.1f}% nulos")

if cats_ausentes:
    print(f"\n  Categóricas ausentes ({len(cats_ausentes)}):")
    for v in cats_ausentes:
        print(f"    ✗  {v}")

# ------------------------------------------------------------------
# 4.7 Validação — crosstab por cluster
# ------------------------------------------------------------------
print(f"\n  Validação — distribuição categórica por cluster (%):")
for var in cats_presentes[:3]:   # primeiras 3 para não sobrecarregar output
    ct = pd.crosstab(
        df_cockpit['cluster_short'],
        df_cockpit[var],
        normalize='index'
    ).round(3) * 100
    print(f"\n  {var}:")
    print(ct.to_string())

# =============================================================================
# BLOCO 5 — TABELAS DE APOIO PARA O COCKPIT
# =============================================================================

print("─" * 70)
print("BLOCO 5 — Tabelas de Apoio para o Power BI")
print("─" * 70)

# Tabela de centróides em escala original
df_centroides_orig.to_csv(
    f'{OUTPUT_DIR}/data/centroides_perfil_{TIMESTAMP}.csv',
    index=False, encoding='utf-8-sig'
)

# Tabela de estratégias
df_estrategias = pd.DataFrame([
    {
        'cluster_id'         : c,
        'cluster_name'       : cluster_names[c],
        'cluster_short'      : cluster_short[c],
        'n_devedores'        : int((labels_k4 == c).sum()),
        'pct_carteira'       : round((labels_k4 == c).mean() * 100, 1),
        'collection_strategy': ESTRATEGIAS[c],
        'priority_rank'      : PRIORIDADE[c],
    }
    for c in range(K_FINAL)
]).sort_values('priority_rank')

df_estrategias.to_csv(
    f'{OUTPUT_DIR}/data/estrategias_cluster_{TIMESTAMP}.csv',
    index=False, encoding='utf-8-sig'
)

print(f"\n  Estratégias por cluster (ordem de prioridade):")
print(f"\n  {'Rank':<5} {'Sigla':<8} {'N':>8} {'%':>6}  Estratégia")
print("  " + "─" * 72)
for _, row in df_estrategias.iterrows():
    print(f"  {int(row.priority_rank):<5} {row.cluster_short:<8} "
          f"{int(row.n_devedores):>8,} {row.pct_carteira:>5.1f}%  "
          f"{row.collection_strategy}")


# =============================================================================
# BLOCO 6 — MONITOR DE DRIFT
# =============================================================================

print("\n" + "─" * 70)
print("BLOCO 6 — Estatísticas de Referência para Monitorização de Drift")
print("─" * 70)

dist_referencia = {
    str(c): {
        'n'   : int((labels_k4 == c).sum()),
        'pct' : round(float((labels_k4 == c).mean() * 100), 2),
        'name': cluster_names[c],
    }
    for c in range(K_FINAL)
}

vars_monitor = [v for v in [
    'da_total_debt_value',
    'ap_payment_capacity_value',
    'pc_total_payment_2025_value',
    'ap_debt_to_revenue_ratio',
    'pc_payment_regularity_ratio',
] if v in df_orig_k4.columns]

stats_referencia = {}
for v in vars_monitor:
    stats_referencia[v] = {
        'global': {q: float(df_orig_k4[v].quantile(p))
                   for q, p in [('p10',.1),('p25',.25),
                                 ('p50',.5),('p75',.75),('p90',.9)]}
    }
    for c in range(K_FINAL):
        mask = df_orig_k4['cluster'] == c
        stats_referencia[v][f'c{c}'] = {
            'p50': float(df_orig_k4.loc[mask, v].quantile(0.50)),
            'p75': float(df_orig_k4.loc[mask, v].quantile(0.75)),
        }

monitor_config = {
    'model_version'              : f'kmeans_k4_{TIMESTAMP}',
    'training_date'              : TIMESTAMP,
    'n_training_obs'             : int(len(labels_k4)),
    'inertia_reference'          : float(km4.inertia_),
    'inertia_alert_threshold'    : float(km4.inertia_ * 1.15),
    'silhouette_reference'       : 0.2558,
    'silhouette_alert_threshold' : 0.20,
    'cluster_distribution'       : dist_referencia,
    'distribution_alert_pct'     : 5.0,
    'key_variable_stats'         : stats_referencia,
    'recommended_retrain_months' : 6,
}

with open(f'{OUTPUT_DIR}/monitor/monitor_config_{TIMESTAMP}.json', 'w',
          encoding='utf-8') as f:
    json.dump(monitor_config, f, indent=2, ensure_ascii=False)

print(f"\n  Inércia referência      : {km4.inertia_:,.2f}")
print(f"  Threshold alerta (+15%) : {km4.inertia_*1.15:,.2f}")
print(f"  Silhouette referência   : 0.2558  |  Alerta abaixo de: 0.20")
print(f"  Alerta distribuição     : ±5 p.p. por cluster")
print(f"  Retreino recomendado    : a cada 6 meses")


# =============================================================================
# BLOCO 7 — SCRIPT STANDALONE classify_debtors.py
# =============================================================================

script = f'''#!/usr/bin/env python3
"""
classify_debtors.py — RFB 9ª Região Fiscal
Collections Segmentation Cockpit  |  versão kmeans_k4_{TIMESTAMP}

Uso:
    python classify_debtors.py --input novos_devedores.csv \\
                               --output resultado.csv \\
                               --model deployment/model/artefacto_v{TIMESTAMP}.pkl
"""
import argparse, pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances

def classificar(df, artefacto, id_col='id_cnpj'):
    feat  = artefacto['feature_names']
    sc    = artefacto['scaler']
    cents = artefacto['centroids_normalised']
    nomes = artefacto['cluster_names']
    short = artefacto['cluster_short']

    miss = [f for f in feat if f not in df.columns]
    if miss:
        raise ValueError(f"Colunas em falta: {{miss}}")

    X     = sc.transform(df[feat].values)
    dists = euclidean_distances(X, cents)
    ids   = dists.argmin(axis=1)

    res = pd.DataFrame()
    if id_col in df.columns:
        res[id_col] = df[id_col].values
    res['cluster_id']             = ids
    res['cluster_name']           = [nomes[c] for c in ids]
    res['cluster_short']          = [short[c]  for c in ids]
    res['dist_cluster_atribuido'] = dists[np.arange(len(dists)), ids]
    res['confianca_relativa']     = 1 - res['dist_cluster_atribuido'] / dists.sum(axis=1)
    return res

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--input',  required=True)
    ap.add_argument('--output', required=True)
    ap.add_argument('--model',  required=True)
    ap.add_argument('--id_col', default='id_cnpj')
    args = ap.parse_args()

    with open(args.model, 'rb') as f:
        art = pickle.load(f)
    print(f"Modelo  : {{art['model_version']}}")
    print(f"Treino  : {{art['training_date']}}")

    df  = pd.read_csv(args.input)
    res = classificar(df, art, id_col=args.id_col)
    res.to_csv(args.output, index=False, encoding='utf-8-sig')
    print(f"Output  : {{args.output}}  ({{len(res):,}} registos)")

    for c, name in art['cluster_names'].items():
        n = (res['cluster_id'] == c).sum()
        print(f"  C{{c}} {{art['cluster_short'][c]}}: {{n:,}} ({{n/len(res)*100:.1f}}%)")

if __name__ == '__main__':
    main()
'''

with open(f'{OUTPUT_DIR}/classify_debtors.py', 'w', encoding='utf-8') as f:
    f.write(script)

print(f"  ✔  classify_debtors.py gerado")


# =============================================================================
# SÍNTESE FINAL
# =============================================================================

print("\n" + "=" * 70)
print("SÍNTESE — Deployment Pipeline Completo")
print("=" * 70)
print(f"""
  {OUTPUT_DIR}/
  ├── model/
  │   ├── artefacto_v{TIMESTAMP}.pkl
  │   ├── kmeans_k4.pkl
  │   ├── scaler.pkl
  │   ├── feature_names.json
  │   ├── cluster_metadata.json
  │   └── centroides_escala_original.csv
  │
  ├── data/
  │   ├── devedores_classificados_{TIMESTAMP}.csv   ← input principal Power BI
  │   │     {len(df_cockpit):,} linhas × {len(colunas_finais)} colunas
  │   │     → categóricas: {contar_presentes(COLS_CADASTRAIS, df_cockpit)} vars  (porte, regime, UF, CNAE...)
  │   │     → contínuas  : {contar_presentes(COLS_DIVIDA + COLS_CAPAG + COLS_HISTORICO + COLS_PERFIL, df_cockpit)} vars  (dívida, pagamento, conformidade...)
  │   │     → distâncias : {contar_presentes(COLS_DISTANCIA, df_cockpit)} vars
  │   ├── centroides_perfil_{TIMESTAMP}.csv
  │   └── estrategias_cluster_{TIMESTAMP}.csv
  │
  ├── monitor/
  │   └── monitor_config_{TIMESTAMP}.json
  │
  └── classify_debtors.py
""")

## DBSCAN

In [ ]:
# =============================================================================
# TAX SEGMENTATION — V04 — DBSCAN (versão optimizada)
# RFB 9ª Região Fiscal | Dissertação de Mestrado BI — Nova IMS
#
# Decisões de desempenho:
#   - PCA a 10 componentes ANTES do DBSCAN (redução dimensional crítica)
#   - Amostra fixa de 30k observações para k-distance e pesquisa em grelha
#   - Grelha enxuta: 10 eps × 4 min_samples = 40 combinações
#   - Silhouette calculado APENAS no treino final, não na grelha
#   - Treino completo (377k) executado uma única vez, no melhor par
#
# Prerequisitos (memória do notebook V03):
#   - df_clustering        : DataFrame pós-normalização, pós-correlação
#   - var_clustering_final : lista de 22 variáveis
#   - labels_k4            : labels K-Means k=4 (referência para ARI)
#   - cluster_names, cluster_short, cluster_colors : mapeamentos visuais
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 1 — IMPORTAÇÕES E CONFIGURAÇÕES
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings

from sklearn.cluster       import DBSCAN
from sklearn.neighbors     import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics       import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from kneed import KneeLocator

warnings.filterwarnings('ignore')

RANDOM_STATE   = 42
N_PCA          = 10       # componentes PCA — redução dimensional pré-DBSCAN
N_SAMPLE       = 30_000   # amostra fixa para k-distance e grelha
N_SAMPLE_SIL   = 30_000   # amostra para silhouette no treino final

np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

print("=" * 70)
print("PIPELINE V04 — DBSCAN (optimizado)")
print("RFB 9ª Região Fiscal")
print("=" * 70)
print(f"\n  PCA: {N_PCA} componentes antes do DBSCAN")
print(f"  Amostra para grelha: {N_SAMPLE:,} observações")
print(f"  Grelha: 10 eps × 4 min_samples = 40 testes")
print(f"  Silhouette: calculado apenas no treino final")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 2 — REDUÇÃO DIMENSIONAL: PCA
#
# Justificativa metodológica:
#   O DBSCAN baseia-se em distâncias euclidianas num espaço de 22 dimensões.
#   Em alta dimensão (maldição da dimensionalidade), as distâncias entre
#   pontos tendem a convergir — o que elimina o contraste entre "vizinhos"
#   e "não-vizinhos" que o DBSCAN precisa para definir densidade.
#   A redução a 10 componentes PCA preserva a estrutura de variância
#   dominante e torna o cálculo de vizinhança tratável.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 2 — REDUÇÃO DIMENSIONAL: PCA")
print("=" * 70)

X_orig = df_clustering[var_clustering_final].values

# Ajustar e transformar
pca = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_orig)

var_exp      = pca.explained_variance_ratio_ * 100
var_acum     = var_exp.cumsum()

print(f"\n  Dataset original : {X_orig.shape[0]:,} obs × {X_orig.shape[1]} variáveis")
print(f"  Dataset PCA      : {X_pca.shape[0]:,} obs × {X_pca.shape[1]} componentes")
print(f"\n  Variância explicada por componente:")
print(f"  {'PC':<6} {'Variância (%)':>15} {'Acumulado (%)':>15}")
print("  " + "─" * 38)
for i, (v, a) in enumerate(zip(var_exp, var_acum), 1):
    print(f"  PC{i:<4} {v:>14.2f}% {a:>14.2f}%")

print(f"\n  ✔  {N_PCA} componentes explicam {var_acum[-1]:.1f}% da variância total.")

# Verificação de NaN
n_nan = np.isnan(X_pca).sum()
if n_nan > 0:
    print(f"\n  ⚠  {n_nan} NaN detectados na matriz PCA — verificar pipeline V03.")
else:
    print(f"  ✔  Matriz PCA sem valores em falta.")

# Variância acumulada — gráfico
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, N_PCA + 1), var_exp, color='#2980b9', alpha=0.7, label='Por componente')
ax.plot(range(1, N_PCA + 1), var_acum, color='#e74c3c', marker='o',
        linewidth=2, markersize=5, label='Acumulado')
ax.axhline(y=var_acum[-1], color='#e74c3c', linestyle=':', alpha=0.5)
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Variância Explicada (%)')
ax.set_title(f'PCA — Variância Explicada por Componente\n'
             f'({N_PCA} PC → {var_acum[-1]:.1f}% da variância total)',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('v04_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 3 — GRÁFICO K-DISTANCE (estimativa de eps)
#
# O gráfico k-distance ordena as distâncias ao k-ésimo vizinho mais próximo.
# O "cotovelo" indica o eps a partir do qual os pontos deixam de estar
# densamente agrupados. Trabalhamos no espaço PCA (10 dimensões).
#
# k_dist = min_samples escolhido = 50 (heurística: 2 × n_dimensões para
# dados de densidade moderada; ajustável conforme o cotovelo observado).
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 3 — GRÁFICO K-DISTANCE")
print(f"Espaço PCA ({N_PCA} dimensões) | amostra {N_SAMPLE:,} observações")
print("=" * 70)

# Amostra fixa — reutilizada em toda a experimentação
idx_sample = rng.choice(X_pca.shape[0], size=min(N_SAMPLE, X_pca.shape[0]), replace=False)
X_sample   = X_pca[idx_sample]

# Três valores de min_samples para calibração
MIN_SAMPLES_KDIST = [20, 50, 100]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
eps_estimativas = {}

for ax, ms in zip(axes, MIN_SAMPLES_KDIST):

    t0   = time.time()
    nbrs = NearestNeighbors(n_neighbors=ms, algorithm='ball_tree', n_jobs=-1)
    nbrs.fit(X_sample)
    distances, _ = nbrs.kneighbors(X_sample)

    dist_k = np.sort(distances[:, -1])[::-1]

    kneedle = KneeLocator(
        range(len(dist_k)), dist_k,
        curve='convex', direction='decreasing', interp_method='polynomial'
    )
    eps_est = dist_k[kneedle.knee] if kneedle.knee else np.percentile(dist_k, 10)
    eps_estimativas[ms] = round(float(eps_est), 4)

    ax.plot(dist_k, color='#2c3e50', linewidth=0.7, alpha=0.8)
    if kneedle.knee:
        ax.axvline(x=kneedle.knee, color='#e74c3c', linestyle='--',
                   linewidth=1.2, label=f'Cotovelo → eps ≈ {eps_est:.3f}')
    ax.axhline(y=eps_est, color='#e74c3c', linestyle=':', linewidth=1.0, alpha=0.6)
    ax.set_title(f'min_samples = {ms}', fontweight='bold')
    ax.set_xlabel('Pontos ordenados (decrescente)')
    ax.set_ylabel(f'Distância ao {ms}º vizinho (PCA)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    print(f"  min_samples={ms:>4d} → eps estimado: {eps_est:.4f}  ({time.time()-t0:.1f}s)")

fig.suptitle(
    f'Gráfico K-Distance — Estimativa de eps (espaço PCA, {N_PCA} dim)\n'
    f'Amostra: {N_SAMPLE:,} devedores',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('v04_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n  Resumo:")
for ms, eps in eps_estimativas.items():
    print(f"    min_samples={ms:>4d} → eps ≈ {eps:.4f}")

# Eps de referência para centrar a grelha
eps_ref = float(np.median(list(eps_estimativas.values())))
print(f"\n  eps de referência (mediana): {eps_ref:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 4 — PESQUISA EM GRELHA ENXUTA (40 combinações)
#
# Grelha centrada no eps_ref estimado pelo cotovelo.
# Executa apenas na amostra de 30k — sem métricas internas.
# Regista apenas: n_clusters e % ruído.
# Duração esperada: < 2 minutos.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 4 — PESQUISA EM GRELHA ENXUTA")
print(f"10 eps × 4 min_samples = 40 testes | amostra {N_SAMPLE:,} obs")
print("=" * 70)

# Grelha: 10 valores de eps centrados no cotovelo
EPS_GRID = np.round(
    np.linspace(max(0.05, eps_ref * 0.3), eps_ref * 2.5, 10), 4
)
MIN_SAMPLES_GRID = [20, 50, 100, 200]

print(f"\n  eps testados     : {list(EPS_GRID)}")
print(f"  min_samples      : {MIN_SAMPLES_GRID}")

resultados_grid = []
t0_grid = time.time()

for ms in MIN_SAMPLES_GRID:
    for eps in EPS_GRID:

        db     = DBSCAN(eps=eps, min_samples=ms, algorithm='ball_tree', n_jobs=-1)
        labels = db.fit_predict(X_sample)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise    = (labels == -1).sum()
        pct_noise  = n_noise / len(labels) * 100

        resultados_grid.append({
            'eps':        eps,
            'min_samples': ms,
            'n_clusters': n_clusters,
            'pct_noise':  round(pct_noise, 1),
        })

df_grid = pd.DataFrame(resultados_grid)

print(f"\n  Grelha concluída em {time.time()-t0_grid:.1f}s")
print(f"\n  Distribuição de n_clusters encontrados:")
print(df_grid['n_clusters'].value_counts().sort_index().to_string())

# ─── Heatmaps da grelha ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

eps_labels = [f"{e:.3f}" for e in sorted(df_grid['eps'].unique())]

for ax, col, titulo, cmap in [
    (axes[0], 'n_clusters', 'Número de Clusters',  'RdYlGn'),
    (axes[1], 'pct_noise',  '% Ruído',             'Reds'),
]:
    pivot = df_grid.pivot_table(
        index='min_samples', columns='eps', values=col, aggfunc='first'
    )
    pivot.columns = [f"{c:.3f}" for c in pivot.columns]

    sns.heatmap(
        pivot, ax=ax, cmap=cmap, annot=True,
        fmt='.0f' if col == 'n_clusters' else '.1f',
        linewidths=0.3, cbar_kws={'label': col}
    )
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('eps')
    ax.set_ylabel('min_samples')

fig.suptitle(
    f'DBSCAN — Pesquisa em Grelha (espaço PCA, {N_PCA} dim)\n'
    f'Amostra: {N_SAMPLE:,} devedores',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('v04_grid_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Seleccionar melhor candidato ─────────────────────────────────────────────
# Critério: menor % ruído entre configurações com ≥ 2 clusters.
# Não forçamos k específico — o DBSCAN encontra o que os dados suportam.

candidatos = df_grid[
    (df_grid['n_clusters'] >= 2) &
    (df_grid['pct_noise']  <  50)
].sort_values('pct_noise')

print(f"\n  Candidatos (k ≥ 2, ruído < 50%), ordenados por % ruído:")
print(f"\n  {'eps':>8}  {'min_samples':>12}  {'k':>5}  {'% ruído':>9}")
print("  " + "─" * 40)

for _, row in candidatos.head(10).iterrows():
    print(f"  {row['eps']:>8.4f}  {row['min_samples']:>12.0f}  "
          f"{row['n_clusters']:>5.0f}  {row['pct_noise']:>8.1f}%")

if len(candidatos) == 0:
    print("\n  ⚠  Nenhum candidato com ruído < 50%.")
    print("     Utilizar o par com menor % ruído global:")
    melhor = df_grid.sort_values('pct_noise').iloc[0]
else:
    melhor = candidatos.iloc[0]

EPS_FINAL        = float(melhor['eps'])
MIN_SAMPLES_FINAL = int(melhor['min_samples'])
K_ENCONTRADO     = int(melhor['n_clusters'])

print(f"\n  ✔  Configuração seleccionada:")
print(f"     eps         = {EPS_FINAL}")
print(f"     min_samples = {MIN_SAMPLES_FINAL}")
print(f"     k esperado  = {K_ENCONTRADO}")
print(f"     % ruído     ≈ {melhor['pct_noise']:.1f}%")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 5 — TREINO EM AMOSTRA GRANDE + PROJECÇÃO KNN PARA O DATASET COMPLETO
#
# Razão da mudança de abordagem:
#   O DBSCAN com n_jobs=-1 no dataset completo (377k obs) lança MemoryError.
#   O método radius_neighbors tem complexidade de memória O(n²) — o sklearn
#   documenta explicitamente esta limitação no código-fonte.
#
# Estratégia adoptada:
#   1. Treinar DBSCAN numa amostra estratificada de 80k observações.
#      Esta amostra representa ~21% da carteira e é suficiente para
#      estabilizar a estrutura de clusters identificada na grelha.
#   2. Para os restantes ~297k devedores, usar KNN (k=1) sobre os pontos
#      classificados da amostra para atribuir o cluster do vizinho mais
#      próximo. Pontos sem vizinho classificado dentro de eps ficam como -1.
#
# Nota metodológica para a dissertação:
#   Este procedimento é análogo ao que o K-Means faz com predict().
#   A diferença é que o K-Means tem centróides fixos; aqui usamos
#   os pontos classificados da amostra como referência de projecção.
#   Os labels resultantes são aproximados — suficientes para comparação
#   metodológica, mas não equivalentes a um treino completo.
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.neighbors import KNeighborsClassifier

N_SAMPLE_TRAIN = 80_000   # amostra para treino DBSCAN — ~21% da carteira

print("\n" + "=" * 70)
print("CÉLULA 5 — TREINO EM AMOSTRA + PROJECÇÃO KNN")
print(f"eps={EPS_FINAL} | min_samples={MIN_SAMPLES_FINAL}")
print(f"Amostra de treino: {N_SAMPLE_TRAIN:,} | Dataset completo: {X_pca.shape[0]:,}")
print("=" * 70)

# ─── Passo 1: amostra estratificada para treino ───────────────────────────────

idx_train  = rng.choice(X_pca.shape[0], size=min(N_SAMPLE_TRAIN, X_pca.shape[0]),
                        replace=False)
idx_resto  = np.setdiff1d(np.arange(X_pca.shape[0]), idx_train)
X_train    = X_pca[idx_train]

print(f"\n  [Passo 1] Treino DBSCAN na amostra ({len(X_train):,} obs)...")
t0 = time.time()

db_final     = DBSCAN(eps=EPS_FINAL, min_samples=MIN_SAMPLES_FINAL,
                      algorithm='ball_tree', n_jobs=1)   # n_jobs=1 — evita MemoryError
labels_train = db_final.fit_predict(X_train)

t_treino = time.time() - t0
n_cl_train  = len(set(labels_train)) - (1 if -1 in labels_train else 0)
n_ruid_train = (labels_train == -1).sum()

print(f"  Tempo          : {t_treino:.1f}s")
print(f"  Clusters       : {n_cl_train}")
print(f"  Ruído (amostra): {n_ruid_train:,}  ({n_ruid_train/len(labels_train)*100:.1f}%)")

# ─── Passo 2: projecção KNN para os restantes devedores ───────────────────────
# Treinar KNN apenas nos pontos classificados (label != -1) da amostra.
# Cada devedor fora da amostra recebe o label do seu vizinho mais próximo.
# Se esse vizinho estava a distância > eps, mantém label = -1.

print(f"\n  [Passo 2] Projecção KNN para {len(idx_resto):,} devedores restantes...")
t0 = time.time()

mask_train_valid = labels_train != -1
X_ref            = X_train[mask_train_valid]
labels_ref       = labels_train[mask_train_valid]

labels_db  = np.full(X_pca.shape[0], -1, dtype=int)
labels_db[idx_train] = labels_train   # atribuir directamente os da amostra

if len(X_ref) > 0 and len(idx_resto) > 0:
    # KNN com k=1 — vizinho mais próximo entre os pontos classificados
    knn = KNeighborsClassifier(n_neighbors=1, algorithm='ball_tree', n_jobs=1)
    knn.fit(X_ref, labels_ref)

    # Distância ao vizinho mais próximo — pontos além de eps ficam como ruído
    dists, _ = knn.kneighbors(X_pca[idx_resto])
    pred_labels = knn.predict(X_pca[idx_resto])

    # Aplicar limiar eps: só aceitar projecção se o vizinho estiver dentro do raio
    dentro_eps = dists[:, 0] <= EPS_FINAL
    labels_db[idx_resto[dentro_eps]]  = pred_labels[dentro_eps]
    labels_db[idx_resto[~dentro_eps]] = -1   # fora do raio → ruído

    n_projectados = dentro_eps.sum()
    n_ruido_proj  = (~dentro_eps).sum()
    print(f"  Projectados com sucesso : {n_projectados:,}  ({n_projectados/len(idx_resto)*100:.1f}%)")
    print(f"  Classificados como ruído: {n_ruido_proj:,}  ({n_ruido_proj/len(idx_resto)*100:.1f}%)")

print(f"  Tempo projecção: {time.time()-t0:.1f}s")

# ─── Resultado global ─────────────────────────────────────────────────────────

n_clusters_final = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_final    = (labels_db == -1).sum()
pct_noise_final  = n_noise_final / len(labels_db) * 100
n_valid_final    = len(labels_db) - n_noise_final

print(f"\n  RESULTADO GLOBAL ({X_pca.shape[0]:,} devedores):")
print(f"  Clusters encontrados: {n_clusters_final}")
print(f"  Ruído (label = -1)  : {n_noise_final:,}  ({pct_noise_final:.1f}%)")
print(f"  Pontos classificados: {n_valid_final:,}  ({100-pct_noise_final:.1f}%)")
print(f"\n  Distribuição por cluster:")
for c in sorted(set(labels_db)):
    n   = (labels_db == c).sum()
    lbl = "  Ruído" if c == -1 else f"  Cluster {c}"
    print(f"  {lbl:<15}: {n:>8,}  ({n/len(labels_db)*100:.1f}%)")

# ─── Métricas internas (amostra de treino — espaço comparável) ────────────────
# Calculadas na amostra, não no dataset completo, para manter consistência
# com o espaço em que o DBSCAN foi efectivamente treinado.

mask_valid   = labels_train != -1
X_valid      = X_train[mask_valid]
labels_valid = labels_train[mask_valid]

metricas_db = {
    'n_clusters':        n_clusters_final,
    'n_noise':           n_noise_final,
    'pct_noise':         round(pct_noise_final, 2),
    'silhouette':        np.nan,
    'davies_bouldin':    np.nan,
    'calinski_harabasz': np.nan,
}

if n_clusters_final >= 2 and mask_valid.sum() > 1000:

    print(f"\n  Métricas internas (amostra treino, n={mask_valid.sum():,}, sem ruído):")

    idx_sil = rng.choice(mask_valid.sum(), size=min(N_SAMPLE_SIL, mask_valid.sum()),
                         replace=False)
    try:
        sil = silhouette_score(X_valid[idx_sil], labels_valid[idx_sil])
        metricas_db['silhouette'] = round(sil, 4)
        print(f"    Silhouette       : {sil:.4f}")
    except Exception as e:
        print(f"    Silhouette       : erro — {e}")

    try:
        db_score = davies_bouldin_score(X_valid, labels_valid)
        metricas_db['davies_bouldin'] = round(db_score, 4)
        print(f"    Davies-Bouldin   : {db_score:.4f}")
    except Exception as e:
        print(f"    Davies-Bouldin   : erro — {e}")

    try:
        ch_score = calinski_harabasz_score(X_valid, labels_valid)
        metricas_db['calinski_harabasz'] = round(ch_score, 1)
        print(f"    Calinski-Harabasz: {ch_score:,.1f}")
    except Exception as e:
        print(f"    Calinski-Harabasz: erro — {e}")

else:
    print(f"\n  ⚠  Métricas não calculadas (k={n_clusters_final}, "
          f"n_valid={mask_valid.sum():,}).")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 6 — COMPARAÇÃO COM K-MEANS k=4 (ARI / NMI)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 6 — COMPARAÇÃO COM K-MEANS k=4  (ARI / NMI)")
print("=" * 70)

if 'labels_k4' not in dir():
    print("  ⚠  labels_k4 não encontrado em memória.")
    print("     Executar o notebook V03 até à secção Modelling antes de continuar.")
else:
    # (a) Incluindo ruído como categoria separada
    ari_a = adjusted_rand_score(labels_k4, labels_db)
    nmi_a = normalized_mutual_info_score(labels_k4, labels_db, average_method='arithmetic')
    print(f"\n  (a) Incluindo ruído (label=-1) como categoria:")
    print(f"      ARI = {ari_a:.4f}  |  NMI = {nmi_a:.4f}")

    # (b) Apenas pontos classificados pelo DBSCAN
    mask_valid_cmp = labels_db != -1
    ari_b = adjusted_rand_score(labels_k4[mask_valid_cmp], labels_db[mask_valid_cmp])
    nmi_b = normalized_mutual_info_score(
        labels_k4[mask_valid_cmp], labels_db[mask_valid_cmp],
        average_method='arithmetic'
    )
    pct_cmp = mask_valid_cmp.mean() * 100
    print(f"\n  (b) Apenas pontos classificados ({pct_cmp:.1f}% do total):")
    print(f"      ARI = {ari_b:.4f}  |  NMI = {nmi_b:.4f}")

    print(f"\n  Interpretação:")
    for ari_val, cenario in [(ari_a, '(a)'), (ari_b, '(b)')]:
        if   ari_val >= 0.80: interp = "concordância muito elevada"
        elif ari_val >= 0.50: interp = "concordância moderada"
        elif ari_val >= 0.20: interp = "concordância baixa"
        else:                 interp = "próxima do aleatório"
        print(f"    Cenário {cenario}: {interp} (ARI = {ari_val:.4f})")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 7 — VISUALIZAÇÃO PCA 2D
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 7 — VISUALIZAÇÃO PCA 2D")
print("=" * 70)

# PC1 e PC2 já estão em X_pca — usar directamente (sem segundo PCA)
var_exp_plot = pca.explained_variance_ratio_[:2] * 100

# Paleta: cada cluster uma cor distinta; ruído a cinzento claro
clusters_unicos = sorted(set(labels_db))
palette = ['#8e44ad', '#16a085', '#d35400', '#2c3e50',
           '#c0392b', '#f39c12', '#27ae60', '#2980b9']
cor_map = {}
idx_cor = 0
for c in clusters_unicos:
    if c == -1:
        cor_map[c] = '#cccccc'
    else:
        cor_map[c] = palette[idx_cor % len(palette)]
        idx_cor += 1

# Amostra estratificada para visualização
idx_plot = []
for c in clusters_unicos:
    idx_c = np.where(labels_db == c)[0]
    n_s   = 1_500 if c == -1 else min(4_000, len(idx_c))
    idx_plot.extend(rng.choice(idx_c, size=min(n_s, len(idx_c)), replace=False).tolist())
idx_plot = np.array(idx_plot)

lab_plot = labels_db[idx_plot]

fig, ax = plt.subplots(figsize=(10, 8))

for c in clusters_unicos:
    mask   = lab_plot == c
    lbl    = "Ruído" if c == -1 else f"Cluster {c}  (n={( labels_db==c).sum():,})"
    alpha  = 0.15 if c == -1 else 0.50
    size   = 2    if c == -1 else 5
    ax.scatter(
        X_pca[idx_plot[mask], 0],
        X_pca[idx_plot[mask], 1],
        s=size, alpha=alpha, color=cor_map[c],
        label=lbl, rasterized=True
    )

sil_str = f"Silhouette={metricas_db['silhouette']:.4f}" \
          if not np.isnan(metricas_db['silhouette']) else "Silhouette=n/a"

ax.set_xlabel(f'PC1  ({var_exp_plot[0]:.1f}% da variância)', fontsize=11)
ax.set_ylabel(f'PC2  ({var_exp_plot[1]:.1f}% da variância)', fontsize=11)
ax.set_title(
    f'DBSCAN — Projecção PCA 2D\n'
    f'eps={EPS_FINAL} | min_samples={MIN_SAMPLES_FINAL} | '
    f'k={n_clusters_final} | ruído={pct_noise_final:.1f}% | {sil_str}',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9, markerscale=3, loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('v04_dbscan_pca.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 8 — PAINEL DE COMPARAÇÃO: K-MEANS k=4 vs DBSCAN
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 8 — COMPARAÇÃO K-MEANS k=4 vs DBSCAN")
print("=" * 70)

# Calcular métricas K-Means para comparação directa
if 'labels_k4' in dir():
    idx_sil_ref = rng.choice(X_orig.shape[0], size=30_000, replace=False)
    sil_km   = silhouette_score(X_orig[idx_sil_ref], labels_k4[idx_sil_ref])
    db_km    = davies_bouldin_score(X_orig, labels_k4)
    ch_km    = calinski_harabasz_score(X_orig, labels_k4)

    linhas = [
        {
            'Algoritmo':         'K-Means  k=4',
            'k':                  4,
            'Silhouette ↑':       round(sil_km, 4),
            'Davies-Bouldin ↓':   round(db_km,  4),
            'Calinski-H ↑':       round(ch_km,  1),
            '% Ruído ↓':          0.0,
            'Cobertura (%)':      100.0,
        },
        {
            'Algoritmo':         f'DBSCAN  k={n_clusters_final}',
            'k':                  n_clusters_final,
            'Silhouette ↑':       metricas_db['silhouette'],
            'Davies-Bouldin ↓':   metricas_db['davies_bouldin'],
            'Calinski-H ↑':       metricas_db['calinski_harabasz'],
            '% Ruído ↓':          round(pct_noise_final, 1),
            'Cobertura (%)':      round(100 - pct_noise_final, 1),
        },
    ]

    df_comp = pd.DataFrame(linhas)
    print(f"\n  Nota: métricas do DBSCAN calculadas nos {n_valid_final:,} pontos classificados.")
    print(f"  Métricas do K-Means calculadas no espaço original (22 dim).\n")
    print(df_comp.to_string(index=False))

    # Gráfico de barras comparativo
    metricas_plot = [
        ('Silhouette ↑',     True,  '#2ecc71'),
        ('Davies-Bouldin ↓', False, '#e74c3c'),
        ('Calinski-H ↑',     True,  '#3498db'),
        ('% Ruído ↓',        False, '#e67e22'),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))

    for ax, (col, maior_melhor, cor) in zip(axes, metricas_plot):
        vals = df_comp[col].values.astype(float)
        rots = df_comp['Algoritmo'].values

        cores_bar = ['#2980b9', '#8e44ad']
        bars = ax.bar(rots, vals, color=cores_bar, edgecolor='white',
                      alpha=0.85, width=0.5)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max([x for x in vals if not np.isnan(x)]) * 0.02,
                    f'{v:,.3f}' if v < 100 else f'{v:,.0f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold'
                )

        seta = '↑ maior=melhor' if maior_melhor else '↓ menor=melhor'
        ax.set_title(f'{col}\n({seta})', fontweight='bold', fontsize=9)
        ax.set_xticks(range(len(rots)))
        ax.set_xticklabels(rots, rotation=20, ha='right', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    patch_km = mpatches.Patch(color='#2980b9', label='K-Means k=4')
    patch_db = mpatches.Patch(color='#8e44ad', label=f'DBSCAN k={n_clusters_final}')
    fig.legend(handles=[patch_km, patch_db], loc='lower center',
               ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(
        'Comparação K-Means k=4 vs DBSCAN\n'
        'Métricas de Validação Interna | RFB 9ª Região Fiscal',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('v04_comparacao.png', dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("  ⚠  labels_k4 não disponível — comparação omitida.")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 9 — RELATÓRIO FINAL
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 9 — RELATÓRIO FINAL")
print("=" * 70)

print(f"""
DBSCAN — SÍNTESE DOS RESULTADOS
─────────────────────────────────────────────────────────────────────

  Configuração final:
    eps             = {EPS_FINAL}
    min_samples     = {MIN_SAMPLES_FINAL}
    Clusters        = {n_clusters_final}
    Ruído           = {n_noise_final:,} devedores  ({pct_noise_final:.1f}%)
    Cobertura total = {100 - pct_noise_final:.1f}%

  Métricas internas (pontos classificados):
    Silhouette       = {metricas_db['silhouette']}
    Davies-Bouldin   = {metricas_db['davies_bouldin']}
    Calinski-H.      = {metricas_db['calinski_harabasz']}

─────────────────────────────────────────────────────────────────────

  LIMITAÇÕES METODOLÓGICAS DO DBSCAN NESTE CONTEXTO:

  1. Maldição da dimensionalidade:
     Em 22 dimensões, as distâncias euclidianas convergem — o contraste
     entre "vizinhos densos" e "pontos isolados" que o DBSCAN precisa
     esbate-se. A redução PCA a {N_PCA} componentes atenua este problema,
     mas não o elimina.

  2. Heterogeneidade de densidade:
     A carteira apresenta clusters com densidades muito distintas
     (AHCD concentrado vs SLED disperso). O DBSCAN usa um único eps
     global — inadequado para este perfil distribucional.

  3. Cobertura parcial:
     {pct_noise_final:.1f}% dos devedores classificados como ruído representam
     informação descartada. Para um artefacto operacional de cobrança,
     segmentar a totalidade da carteira é um requisito não-negociável.

─────────────────────────────────────────────────────────────────────

  CONCLUSÃO:
  O DBSCAN não produziu uma partição operacionalmente utilizável.
  Este resultado é esperado e coerente com a literatura sobre
  clustering em dados financeiros de alta dimensão.
  A experiência reforça a adequação do K-Means k=4, que oferece:
    ✔  Cobertura total da carteira (0% ruído)
    ✔  Estabilidade de labels (ARI ≥ 0.9930 em 20 sementes)
    ✔  Quatro perfis nomeados com interpretabilidade operacional

─────────────────────────────────────────────────────────────────────
""")

print("  ✔  Pipeline V04 concluído.")
print("     Ficheiros gerados:")
for f in ['v04_pca_variance.png', 'v04_kdistance.png',
          'v04_grid_heatmap.png',  'v04_dbscan_pca.png',
          'v04_comparacao.png']:
    print(f"       {f}")

## HDBSCAN

In [ ]:
# =============================================================================
# TAX SEGMENTATION — V05 — HDBSCAN
# RFB 9ª Região Fiscal | Dissertação de Mestrado BI — Nova IMS
#
# HDBSCAN vs DBSCAN — diferença fundamental:
#   O DBSCAN usa um limiar de densidade global (eps único).
#   O HDBSCAN constrói uma hierarquia de densidades e extrai os clusters
#   mais estáveis ao longo dessa hierarquia — cada região do espaço
#   pode ter a sua própria densidade local. Isso resolve directamente
#   o problema identificado no V04: heterogeneidade de densidade entre
#   os segmentos da carteira (AHCD concentrado vs SLED disperso).
#
# Hiperparâmetros principais:
#   min_cluster_size : tamanho mínimo de um cluster (análogo ao k do K-Means)
#   min_samples      : controla conservadorismo — quanto maior, mais ruído
#
# Prerequisitos (memória do notebook V03 + Célula 2 do V04):
#   - df_clustering        : DataFrame pós-normalização (22 variáveis)
#   - var_clustering_final : lista das 22 variáveis
#   - X_pca                : matriz PCA 10 componentes (da Célula 2 do V04)
#   - pca                  : objecto PCA ajustado (da Célula 2 do V04)
#   - labels_k4            : labels K-Means k=4 (referência para ARI)
#   - cluster_names, cluster_short, cluster_colors : mapeamentos visuais
#
# Se X_pca não estiver em memória, a Célula 1 reconstrói-o automaticamente.
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 1 — IMPORTAÇÕES, CONFIGURAÇÕES E VERIFICAÇÃO DE PRÉ-REQUISITOS
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings

from sklearn.cluster        import HDBSCAN
from sklearn.decomposition  import PCA
from sklearn.metrics        import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_PCA        = 10       # componentes PCA — herdado do V04
N_SAMPLE_SIL = 30_000   # amostra para silhouette (custo O(n²))

np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

print("=" * 70)
print("PIPELINE V05 — HDBSCAN")
print("RFB 9ª Região Fiscal | Dissertação de Mestrado BI")
print("=" * 70)
print(f"\n  sklearn: {__import__('sklearn').__version__}  (HDBSCAN nativo ≥ 1.3)")


# ─── Verificar / reconstruir X_pca ───────────────────────────────────────────
# Se o V04 foi executado na mesma sessão, X_pca já está em memória.
# Caso contrário, é reconstruído aqui a partir de df_clustering.

if 'X_pca' not in dir() or X_pca is None:
    print("\n  X_pca não encontrado em memória — reconstruindo PCA...")

    X_orig = df_clustering[var_clustering_final].values
    pca    = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
    X_pca  = pca.fit_transform(X_orig)

    var_acum = pca.explained_variance_ratio_.cumsum() * 100
    print(f"  ✔  PCA reconstruído: {N_PCA} componentes → {var_acum[-1]:.1f}% da variância")

else:
    print(f"\n  ✔  X_pca encontrado em memória: {X_pca.shape[0]:,} obs × {X_pca.shape[1]} dim")

print(f"\n  Dataset para clustering: {X_pca.shape[0]:,} observações × {X_pca.shape[1]} componentes")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 2 — PESQUISA EM GRELHA: min_cluster_size × min_samples
#
# Estratégia de calibração:
#   min_cluster_size: define o menor cluster aceitável operacionalmente.
#     Com 377k devedores, valores entre 500 e 5.000 são razoáveis
#     (0.1% a 1.3% da carteira). Valores muito pequenos produzem
#     muitos micro-clusters; valores muito grandes colapsam tudo num só.
#
#   min_samples: controla a robustez à densidade local.
#     None (padrão) = igual a min_cluster_size.
#     Aumentar min_samples torna o algoritmo mais conservador
#     (mais ruído, clusters mais compactos).
#
# Execução na amostra de 30k — menos de 3 minutos para a grelha completa.
# Silhouette NÃO está no loop da grelha (evitar repetição do travamento V04).
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 2 — PESQUISA EM GRELHA: min_cluster_size × min_samples")
print("Amostra fixa: 30.000 observações | sem Silhouette no loop")
print("=" * 70)

# Amostra fixa
idx_sample = rng.choice(X_pca.shape[0], size=min(30_000, X_pca.shape[0]), replace=False)
X_sample   = X_pca[idx_sample]

# Grelha
# min_cluster_size em unidades absolutas sobre a amostra de 30k
# (proporcionalmente: 100=0.3%, 300=1%, 1000=3.3%, 3000=10%)
MIN_CLUSTER_SIZES = [50, 100, 300, 500, 1000, 2000, 3000]

# min_samples: None = igual a min_cluster_size (padrão conservador)
# Testamos None e duas fracções de min_cluster_size
MIN_SAMPLES_RATIOS = {
    'None (padrão)': None,
    '10%  de mcs':   0.10,
    '25%  de mcs':   0.25,
}

resultados_grid = []
t0_grid = time.time()

total_testes = len(MIN_CLUSTER_SIZES) * len(MIN_SAMPLES_RATIOS)
print(f"\n  {len(MIN_CLUSTER_SIZES)} min_cluster_size × {len(MIN_SAMPLES_RATIOS)} min_samples = {total_testes} testes\n")
print(f"  {'mcs':>6}  {'ms_cfg':<18}  {'ms_val':>8}  {'k':>5}  {'% ruído':>9}  {'tempo':>7}")
print("  " + "─" * 60)

for mcs in MIN_CLUSTER_SIZES:
    for ms_label, ms_ratio in MIN_SAMPLES_RATIOS.items():

        ms_val = None if ms_ratio is None else max(1, int(mcs * ms_ratio))

        t0 = time.time()

        hdb = HDBSCAN(
            min_cluster_size = mcs,
            min_samples      = ms_val,
            cluster_selection_method = 'eom',   # Excess of Mass — padrão
            n_jobs           = -1,
        )
        labels = hdb.fit_predict(X_sample)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise    = (labels == -1).sum()
        pct_noise  = n_noise / len(labels) * 100
        tempo      = time.time() - t0

        resultados_grid.append({
            'min_cluster_size': mcs,
            'min_samples_cfg':  ms_label,
            'min_samples_val':  ms_val if ms_val is not None else mcs,
            'n_clusters':       n_clusters,
            'pct_noise':        round(pct_noise, 1),
            'tempo_s':          round(tempo, 1),
        })

        print(f"  {mcs:>6}  {ms_label:<18}  "
              f"{'None' if ms_val is None else ms_val:>8}  "
              f"{n_clusters:>5}  {pct_noise:>8.1f}%  {tempo:>5.1f}s")

df_grid = pd.DataFrame(resultados_grid)
print(f"\n  Grelha concluída em {time.time()-t0_grid:.1f}s")


# ─── Heatmap da grelha ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, titulo, cmap, fmt in [
    (axes[0], 'n_clusters', 'Número de Clusters',  'RdYlGn', '.0f'),
    (axes[1], 'pct_noise',  '% Ruído',             'Reds',   '.1f'),
]:
    pivot = df_grid.pivot_table(
        index='min_samples_cfg',
        columns='min_cluster_size',
        values=col,
        aggfunc='first'
    )
    sns.heatmap(
        pivot, ax=ax, cmap=cmap, annot=True, fmt=fmt,
        linewidths=0.4, cbar_kws={'label': col}
    )
    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_xlabel('min_cluster_size', fontsize=10)
    ax.set_ylabel('min_samples', fontsize=10)

fig.suptitle(
    f'HDBSCAN — Pesquisa em Grelha (espaço PCA, {N_PCA} dim)\n'
    f'Amostra: 30.000 devedores | cluster_selection_method=eom',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('v05_hdbscan_grid.png', dpi=150, bbox_inches='tight')
plt.show()


# ─── Seleccionar candidatos ───────────────────────────────────────────────────

print("\n" + "─" * 70)
print("CANDIDATOS — k ∈ {2, 3, 4, 5} e % ruído < 40%")
print("─" * 70)

candidatos = df_grid[
    df_grid['n_clusters'].between(2, 5) &
    (df_grid['pct_noise'] < 40)
].sort_values(['n_clusters', 'pct_noise'])

if len(candidatos) == 0:
    print("\n  ⚠  Nenhum candidato com ruído < 40%. Relaxando para 60%:")
    candidatos = df_grid[
        df_grid['n_clusters'].between(2, 6) &
        (df_grid['pct_noise'] < 60)
    ].sort_values(['n_clusters', 'pct_noise'])

if len(candidatos) == 0:
    print("  ⚠  Nenhum candidato com k ∈ {2..6}. Mostrar todos:")
    candidatos = df_grid.sort_values(['n_clusters', 'pct_noise'])

print(f"\n  {'mcs':>6}  {'ms_cfg':<18}  {'k':>5}  {'% ruído':>9}")
print("  " + "─" * 44)
for _, row in candidatos.iterrows():
    print(f"  {row['min_cluster_size']:>6.0f}  {row['min_samples_cfg']:<18}  "
          f"{row['n_clusters']:>5.0f}  {row['pct_noise']:>8.1f}%")

# Seleccionar o candidato com menor ruído entre os que têm k ≥ 2
melhor = candidatos.sort_values('pct_noise').iloc[0]
MCS_FINAL = int(melhor['min_cluster_size'])
MS_FINAL  = None if melhor['min_samples_cfg'] == 'None (padrão)' \
            else max(1, int(MCS_FINAL * float(melhor['min_samples_cfg'].split('%')[0]) / 100))

print(f"\n  ✔  Configuração seleccionada para treino final:")
print(f"     min_cluster_size = {MCS_FINAL}")
print(f"     min_samples      = {MS_FINAL}")
print(f"     k esperado       = {int(melhor['n_clusters'])}")
print(f"     % ruído (amostra)= {melhor['pct_noise']:.1f}%")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 3 — TREINO FINAL NO DATASET COMPLETO (377k devedores)
#
# Nota sobre cluster_selection_method:
#   'eom' (Excess of Mass): padrão — tende a produzir clusters de tamanhos
#         mais variados, melhor para dados heterogéneos.
#   'leaf': selecciona os clusters foliares — mais clusters, mais pequenos.
#   Testamos ambos para o par seleccionado.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 3 — TREINO FINAL: DATASET COMPLETO")
print(f"min_cluster_size={MCS_FINAL} | min_samples={MS_FINAL}")
print(f"{X_pca.shape[0]:,} devedores × {X_pca.shape[1]} componentes PCA")
print("=" * 70)

resultados_treino = {}

for metodo in ['eom', 'leaf']:

    print(f"\n  cluster_selection_method = '{metodo}'")
    print(f"  {'─' * 50}")

    t0 = time.time()

    hdb = HDBSCAN(
        min_cluster_size         = MCS_FINAL,
        min_samples              = MS_FINAL,
        cluster_selection_method = metodo,
        n_jobs                   = -1,
    )
    labels_hdb = hdb.fit_predict(X_pca)

    tempo = time.time() - t0

    n_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
    n_noise    = (labels_hdb == -1).sum()
    pct_noise  = n_noise / len(labels_hdb) * 100
    n_valid    = len(labels_hdb) - n_noise

    print(f"  Tempo de treino     : {tempo:.1f}s")
    print(f"  Clusters encontrados: {n_clusters}")
    print(f"  Ruído (label = -1)  : {n_noise:,}  ({pct_noise:.1f}%)")
    print(f"  Pontos classificados: {n_valid:,}  ({100 - pct_noise:.1f}%)")
    print(f"\n  Distribuição por cluster:")
    for c in sorted(set(labels_hdb)):
        n   = (labels_hdb == c).sum()
        lbl = "  Ruído" if c == -1 else f"  Cluster {c}"
        print(f"  {lbl:<15}: {n:>8,}  ({n / len(labels_hdb) * 100:.1f}%)")

    # ─── Métricas internas (pontos classificados) ────────────────────────────

    mask_valid   = labels_hdb != -1
    X_valid      = X_pca[mask_valid]
    labels_valid = labels_hdb[mask_valid]

    metricas = {
        'n_clusters':        n_clusters,
        'n_noise':           n_noise,
        'pct_noise':         round(pct_noise, 2),
        'silhouette':        np.nan,
        'davies_bouldin':    np.nan,
        'calinski_harabasz': np.nan,
        'labels':            labels_hdb,
    }

    if n_clusters >= 2 and n_valid > 1000:
        print(f"\n  Métricas internas (n={n_valid:,}, excluindo ruído):")

        idx_sil = rng.choice(n_valid, size=min(N_SAMPLE_SIL, n_valid), replace=False)
        try:
            sil = silhouette_score(X_valid[idx_sil], labels_valid[idx_sil])
            metricas['silhouette'] = round(sil, 4)
            print(f"    Silhouette       : {sil:.4f}")
        except Exception as e:
            print(f"    Silhouette       : erro — {e}")

        try:
            db_s = davies_bouldin_score(X_valid, labels_valid)
            metricas['davies_bouldin'] = round(db_s, 4)
            print(f"    Davies-Bouldin   : {db_s:.4f}")
        except Exception as e:
            print(f"    Davies-Bouldin   : erro — {e}")

        try:
            ch_s = calinski_harabasz_score(X_valid, labels_valid)
            metricas['calinski_harabasz'] = round(ch_s, 1)
            print(f"    Calinski-Harabasz: {ch_s:,.1f}")
        except Exception as e:
            print(f"    Calinski-Harabasz: erro — {e}")

    resultados_treino[metodo] = metricas

# Seleccionar o método com menos ruído para análise subsequente
metodo_final = min(resultados_treino, key=lambda m: resultados_treino[m]['pct_noise'])
labels_hdb_final  = resultados_treino[metodo_final]['labels']
metricas_hdb_final = resultados_treino[metodo_final]

print(f"\n  ✔  Método seleccionado para análise: '{metodo_final}' "
      f"({metricas_hdb_final['pct_noise']:.1f}% ruído)")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 4 — COMPARAÇÃO COM K-MEANS k=4  (ARI / NMI)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 4 — COMPARAÇÃO COM K-MEANS k=4  (ARI / NMI)")
print("=" * 70)

if 'labels_k4' not in dir():
    print("  ⚠  labels_k4 não encontrado. Executar V03 antes de continuar.")

else:
    labels_db_f = labels_hdb_final

    # (a) Comparação directa — ruído como categoria separada
    ari_a = adjusted_rand_score(labels_k4, labels_db_f)
    nmi_a = normalized_mutual_info_score(
        labels_k4, labels_db_f, average_method='arithmetic'
    )

    # (b) Apenas pontos classificados pelo HDBSCAN
    mask_v = labels_db_f != -1
    ari_b  = adjusted_rand_score(labels_k4[mask_v], labels_db_f[mask_v])
    nmi_b  = normalized_mutual_info_score(
        labels_k4[mask_v], labels_db_f[mask_v], average_method='arithmetic'
    )

    print(f"\n  n_clusters HDBSCAN : {metricas_hdb_final['n_clusters']}")
    print(f"  % ruído            : {metricas_hdb_final['pct_noise']:.1f}%")
    print(f"  Pontos comparados  : {mask_v.sum():,} / {len(labels_db_f):,}\n")

    print(f"  {'Cenário':<45} {'ARI':>8}  {'NMI':>8}")
    print("  " + "─" * 64)

    def interpretar_ari(v):
        if   v >= 0.80: return "concordância muito elevada"
        elif v >= 0.50: return "concordância moderada"
        elif v >= 0.20: return "concordância baixa"
        else:           return "próxima do aleatório"

    for descricao, ari_v, nmi_v in [
        ('(a) Incluindo ruído como categoria', ari_a, nmi_a),
        ('(b) Apenas pontos classificados',    ari_b, nmi_b),
    ]:
        print(f"  {descricao:<45} {ari_v:>8.4f}  {nmi_v:>8.4f}")
        print(f"  {'    → ' + interpretar_ari(ari_v):<45}")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 5 — VISUALIZAÇÃO PCA 2D
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 5 — VISUALIZAÇÃO PCA 2D")
print("=" * 70)

var_exp_plot     = pca.explained_variance_ratio_[:2] * 100
clusters_unicos  = sorted(set(labels_hdb_final))
n_clusters_final = metricas_hdb_final['n_clusters']
pct_noise_final  = metricas_hdb_final['pct_noise']

palette = [
    '#8e44ad', '#16a085', '#d35400', '#2c3e50',
    '#c0392b', '#f39c12', '#27ae60', '#2980b9',
    '#7f8c8d', '#1abc9c',
]
cor_map = {}
idx_cor = 0
for c in clusters_unicos:
    cor_map[c] = '#cccccc' if c == -1 else palette[idx_cor % len(palette)]
    if c != -1:
        idx_cor += 1

# Amostra estratificada para visualização
idx_plot = []
for c in clusters_unicos:
    idx_c = np.where(labels_hdb_final == c)[0]
    n_s   = 1_500 if c == -1 else min(4_000, len(idx_c))
    idx_plot.extend(
        rng.choice(idx_c, size=min(n_s, len(idx_c)), replace=False).tolist()
    )
idx_plot = np.array(idx_plot)
lab_plot = labels_hdb_final[idx_plot]

sil_str = (f"Silhouette={metricas_hdb_final['silhouette']:.4f}"
           if not np.isnan(metricas_hdb_final['silhouette'])
           else "Silhouette=n/a")

fig, ax = plt.subplots(figsize=(10, 8))

for c in clusters_unicos:
    mask  = lab_plot == c
    n_tot = (labels_hdb_final == c).sum()
    lbl   = (f"Ruído  (n={n_tot:,})" if c == -1
             else f"Cluster {c}  (n={n_tot:,})")
    alpha = 0.15 if c == -1 else 0.55
    size  = 2    if c == -1 else 5

    ax.scatter(
        X_pca[idx_plot[mask], 0],
        X_pca[idx_plot[mask], 1],
        s=size, alpha=alpha, color=cor_map[c],
        label=lbl, rasterized=True
    )

ax.set_xlabel(f'PC1  ({var_exp_plot[0]:.1f}% da variância)', fontsize=11)
ax.set_ylabel(f'PC2  ({var_exp_plot[1]:.1f}% da variância)', fontsize=11)
ax.set_title(
    f"HDBSCAN — Projecção PCA 2D  |  method='{metodo_final}'\n"
    f"min_cluster_size={MCS_FINAL}  |  k={n_clusters_final}  |  "
    f"ruído={pct_noise_final:.1f}%  |  {sil_str}",
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9, markerscale=3, loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('v05_hdbscan_pca.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 6 — PAINEL COMPARATIVO FINAL: K-Means k=4 vs DBSCAN vs HDBSCAN
#
# Consolida os resultados dos três notebooks num único painel.
# Métricas do DBSCAN recuperadas das variáveis do V04 (se disponíveis).
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 6 — PAINEL COMPARATIVO: K-Means k=4 / DBSCAN / HDBSCAN")
print("=" * 70)

# Calcular métricas K-Means no espaço original (22 dim) para consistência
X_orig_ref = df_clustering[var_clustering_final].values

if 'labels_k4' in dir():
    idx_sil_ref = rng.choice(X_orig_ref.shape[0], size=30_000, replace=False)
    sil_km = silhouette_score(X_orig_ref[idx_sil_ref], labels_k4[idx_sil_ref])
    db_km  = davies_bouldin_score(X_orig_ref, labels_k4)
    ch_km  = calinski_harabasz_score(X_orig_ref, labels_k4)

    linhas = [
        {
            'Algoritmo':          'K-Means  k=4',
            'k':                   4,
            'Silhouette ↑':        round(sil_km, 4),
            'Davies-Bouldin ↓':    round(db_km, 4),
            'Calinski-H ↑':        round(ch_km, 1),
            '% Ruído ↓':           0.0,
            'Cobertura (%)':       100.0,
            'Nota':                'Espaço original (22 dim)',
        },
    ]

    # DBSCAN — se disponível em memória do V04
    if 'metricas_db' in dir():
        linhas.append({
            'Algoritmo':          f"DBSCAN",
            'k':                   metricas_db.get('n_clusters', '—'),
            'Silhouette ↑':        metricas_db.get('silhouette', np.nan),
            'Davies-Bouldin ↓':    metricas_db.get('davies_bouldin', np.nan),
            'Calinski-H ↑':        metricas_db.get('calinski_harabasz', np.nan),
            '% Ruído ↓':           metricas_db.get('pct_noise', np.nan),
            'Cobertura (%)':       round(100 - metricas_db.get('pct_noise', 0), 1),
            'Nota':                f"PCA {N_PCA} dim | pontos classificados",
        })
    else:
        print("  ℹ  metricas_db não encontrado — DBSCAN excluído da comparação.")
        print("     Execute o V04 na mesma sessão para incluir o DBSCAN.")

    # HDBSCAN
    linhas.append({
        'Algoritmo':          f"HDBSCAN  method={metodo_final}",
        'k':                   metricas_hdb_final['n_clusters'],
        'Silhouette ↑':        metricas_hdb_final['silhouette'],
        'Davies-Bouldin ↓':    metricas_hdb_final['davies_bouldin'],
        'Calinski-H ↑':        metricas_hdb_final['calinski_harabasz'],
        '% Ruído ↓':           metricas_hdb_final['pct_noise'],
        'Cobertura (%)':       round(100 - metricas_hdb_final['pct_noise'], 1),
        'Nota':                f"PCA {N_PCA} dim | pontos classificados",
    })

    df_comp = pd.DataFrame(linhas)

    print(f"\n  Legenda:")
    print(f"    Silhouette     ↑  (maior é melhor; máximo = 1)")
    print(f"    Davies-Bouldin ↓  (menor é melhor; mínimo = 0)")
    print(f"    Calinski-H     ↑  (maior é melhor)")
    print(f"    % Ruído        ↓  (menor é melhor; 0% = cobertura total)\n")
    print(df_comp.drop(columns='Nota').to_string(index=False))
    print(f"\n  Notas de comparabilidade:")
    for _, row in df_comp.iterrows():
        print(f"    {row['Algoritmo']}: {row['Nota']}")

    # ─── Gráfico de barras comparativo ───────────────────────────────────────

    metricas_plot = [
        ('Silhouette ↑',     True),
        ('Davies-Bouldin ↓', False),
        ('Calinski-H ↑',     True),
        ('% Ruído ↓',        False),
    ]

    n_alg     = len(df_comp)
    cores_alg = ['#2980b9', '#8e44ad', '#e67e22'][:n_alg]

    fig, axes = plt.subplots(1, 4, figsize=(18, 6))

    for ax, (col, maior_melhor) in zip(axes, metricas_plot):
        vals = df_comp[col].values.astype(float)
        rots = df_comp['Algoritmo'].values

        bars = ax.bar(
            range(n_alg), vals,
            color=cores_alg, edgecolor='white', alpha=0.88, width=0.55
        )

        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                v_max = max([x for x in vals if not np.isnan(x)])
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + v_max * 0.025,
                    f'{v:.3f}' if v < 100 else f'{v:,.0f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold'
                )

        seta = '↑ maior=melhor' if maior_melhor else '↓ menor=melhor'
        ax.set_title(f'{col}\n({seta})', fontweight='bold', fontsize=9)
        ax.set_xticks(range(n_alg))
        ax.set_xticklabels(rots, rotation=25, ha='right', fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)

    patches = [mpatches.Patch(color=c, label=r)
               for c, r in zip(cores_alg, df_comp['Algoritmo'].values)]
    fig.legend(handles=patches, loc='lower center', ncol=n_alg,
               fontsize=9, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle(
        'Comparação Algorítmica — K-Means / DBSCAN / HDBSCAN\n'
        'Métricas de Validação Interna | RFB 9ª Região Fiscal',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('v05_comparacao_final.png', dpi=150, bbox_inches='tight')
    plt.show()

else:
    print("  ⚠  labels_k4 não disponível — painel comparativo omitido.")


# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA 7 — RELATÓRIO FINAL E BASE PARA A DISSERTAÇÃO
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("CÉLULA 7 — RELATÓRIO FINAL")
print("=" * 70)

print(f"""
HDBSCAN — SÍNTESE DOS RESULTADOS
─────────────────────────────────────────────────────────────────────────

  Configuração final:
    min_cluster_size         = {MCS_FINAL}
    min_samples              = {MS_FINAL}
    cluster_selection_method = '{metodo_final}'

  Resultado no dataset completo ({X_pca.shape[0]:,} devedores):
    Clusters encontrados     = {metricas_hdb_final['n_clusters']}
    Ruído (não classificado) = {metricas_hdb_final['n_noise']:,}  ({metricas_hdb_final['pct_noise']:.1f}%)
    Cobertura total          = {100 - metricas_hdb_final['pct_noise']:.1f}%
    Silhouette               = {metricas_hdb_final['silhouette']}
    Davies-Bouldin           = {metricas_hdb_final['davies_bouldin']}
    Calinski-Harabasz        = {metricas_hdb_final['calinski_harabasz']}

─────────────────────────────────────────────────────────────────────────

  VANTAGENS DO HDBSCAN FACE AO DBSCAN:
    ✔  Sem parâmetro eps — não requer calibração de raio global
    ✔  Adapta-se a densidades heterogéneas entre clusters
    ✔  Produz uma hierarquia de clusters explorável
    ✔  Mais estável: min_cluster_size tem interpretação operacional directa

  LIMITAÇÕES PERSISTENTES PARA ESTE CONTEXTO:
    ✗  Cobertura parcial: devedores classificados como ruído
       ficam fora da segmentação operacional
    ✗  Sem garantia de reproduzir exactamente k=3 ou k=4
    ✗  Interpretabilidade dos clusters inferior ao K-Means:
       HDBSCAN não gera centróides com semântica directa
    ✗  Métricas calculadas apenas nos pontos classificados —
       incomparabilidade directa com K-Means (cobertura total)

─────────────────────────────────────────────────────────────────────────

  CONCLUSÃO PARA A DISSERTAÇÃO:
  Dos três paradigmas testados — K-Means, DBSCAN e HDBSCAN — o K-Means
  k=4 mantém a superioridade operacional. Os algoritmos baseados em
  densidade (DBSCAN e HDBSCAN) confirmam empiricamente que a carteira
  de devedores da RFB 9ª Região Fiscal não exibe estrutura de clusters
  por densidade, mas sim por centróide. Este resultado é coerente com
  a natureza dos dados: variáveis financeiras com assimetria positiva
  acentuada e heterogeneidade de escala tendem a produzir partições
  mais estáveis e interpretáveis sob distâncias euclidianas ao centróide
  do que sob estimativas de densidade local.

  A comparação algorítmica constitui, portanto, evidência empírica em
  favor da escolha de K-Means k=4 — não um argumento contra ela.

─────────────────────────────────────────────────────────────────────────
""")

print("  ✔  Pipeline V05 concluído.")
print("     Ficheiros gerados:")
for f in ['v05_hdbscan_grid.png', 'v05_hdbscan_pca.png', 'v05_comparacao_final.png']:
    print(f"       {f}")